In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abinavprabhuv/dasonet-phase5c-eod-20260913/DASO-Net-Phase5C-EOD-20260913-FULL.sha256.txt
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5c-eod-20260913/DASO-Net-Phase5C-EOD-20260913-RESUME.sha256.txt
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5c-eod-20260913/DASO-Net-Phase5C-EOD-20260913-FULL/workspace/phase5c_pcccenter1000_v2.json
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5c-eod-20260913/DASO-Net-Phase5C-EOD-20260913-FULL/workspace/launch_phase5b_amp32.py
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5c-eod-20260913/DASO-Net-Phase5C-EOD-20260913-FULL/workspace/phase5c_scheduleprobe1000_v1.json
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5c-eod-20260913/DASO-Net-Phase5C-EOD-20260913-FULL/workspace/phase5c_pcccenter_prodshape_sanity_v1.json
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5c-eod-20260913/DASO-Net-Phase5C-EOD-20260913-FULL/workspace/phase5c_baseline_curve1000updates_v1.json
/kaggle/input/datasets/abinavprabhuv/dasonet-p

## Integrity Check


In [3]:
# ============================================================
# DASO-Net — NEW NOTEBOOK RESTORE + FULL INTEGRITY AUDIT v2
#
# EOD package: 2026-09-13
#
# Handles Kaggle-expanded handoff datasets safely:
#
# Kaggle may omit/transform nested historical *.zip archives
# when the outer handoff ZIP is ingested as a dataset.
#
# We therefore:
#   - verify EVERY operational manifest file
#   - allow ONLY missing root-level workspace/*.zip archives
#   - verify ALL six baseline checkpoints by SHA256
#   - verify 1500 checkpoint independently in FULL + RESUME
#   - verify decoder SHA
#   - verify training infrastructure markers
#   - verify frozen project hashes
#   - restore to /kaggle/working
#
# NO TRAINING occurs in this cell.
# ============================================================

from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import torch


# ============================================================
# 1. INPUT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

HANDOFF_DATASET = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-phase5c-eod-20260913"
)

FULL_ROOT = (
    HANDOFF_DATASET
    / "DASO-Net-Phase5C-EOD-20260913-FULL"
)

RESUME_ROOT = (
    HANDOFF_DATASET
    / "DASO-Net-Phase5C-EOD-20260913-RESUME"
)

FULL_HANDOFF = (
    FULL_ROOT
    / "handoff"
)

FULL_WORKSPACE = (
    FULL_ROOT
    / "workspace"
)

RESUME_HANDOFF = (
    RESUME_ROOT
    / "handoff"
)

RESUME_WORKSPACE = (
    RESUME_ROOT
    / "workspace"
)

STATE_PATH = (
    FULL_HANDOFF
    / "PHASE5C_EOD_STATE.json"
)

MANIFEST_PATH = (
    FULL_HANDOFF
    / "FULL_FILE_MANIFEST.json"
)

MANIFEST_SHA_PATH = (
    FULL_HANDOFF
    / "FULL_FILE_MANIFEST.sha256.txt"
)

RESTORE_SCRIPT = (
    FULL_HANDOFF
    / "restore_tomorrow.py"
)

COMPLETE_MARKER = (
    FULL_HANDOFF
    / "HANDOFF_COMPLETE.marker"
)


# ============================================================
# 2. EXPECTED CRITICAL HASHES FROM YESTERDAY
# ============================================================

EXPECTED_CHECKPOINT_1500_SHA = (
    "258b9d4457c367373c07f7e346844559"
    "1fe824dc5ceef27e8cd338fc4b76c6f5"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)

EXPECTED_MORPH_SHA = (
    "01722d67038b9ca44f9b0b11f5bac7a"
    "3e588862bc2a86152cf817edaa3de16ee"
)

EXPECTED_FRUITBG_SHA = (
    "d0108ac720cb03dd565cc6c1fb792762e"
    "7afea808ee596f4b7b2be82e2f223fe"
)

EXPECTED_RENDERER_JSON_SHA = (
    "80d1572e58f84750a975dbed5811b2ad5"
    "6c1c4cc51bdf615a5e33df675aeec32"
)

EXPECTED_RENDERER_PY_SHA = (
    "e1d74610a144e19791f23abaa2679a070"
    "0189c5651f73bc33743aa4c24985a46"
)


# ============================================================
# 3. HELPERS
# ============================================================

def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def human_bytes(
    value: int,
) -> str:

    size = float(
        value
    )

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:

        if (
            size < 1024.0
            or unit == "TiB"
        ):
            return (
                f"{size:.2f} {unit}"
            )

        size /= 1024.0

    return str(
        value
    )


def safe_missing_nested_archive(
    relative: Path,
) -> bool:
    """
    Kaggle can omit/transform ZIP files nested directly inside
    workspace/ when the outer EOD ZIP becomes a Dataset.

    ONLY root-level workspace/*.zip historical archives are
    permitted to be absent.

    Operational files, source code, checkpoints, configs,
    metrics, and nested files are NEVER exempted.
    """

    return (
        relative.parent
        == Path(
            "workspace"
        )
        and relative.suffix.lower()
        == ".zip"
    )


# ============================================================
# 4. BASIC INPUT AUDIT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — NEW NOTEBOOK RESTORE/AUDIT v2"
)

print(
    "============================================================"
)


basic_inputs = [
    (
        PROJECT_ROOT,
        "Persistent DASO-Net project dataset",
    ),
    (
        HANDOFF_DATASET,
        "Phase-5C EOD handoff dataset",
    ),
    (
        FULL_ROOT,
        "Expanded FULL handoff",
    ),
    (
        RESUME_ROOT,
        "Expanded emergency RESUME handoff",
    ),
    (
        FULL_WORKSPACE,
        "FULL packaged workspace",
    ),
    (
        RESUME_WORKSPACE,
        "RESUME packaged workspace",
    ),
    (
        STATE_PATH,
        "Phase-5C EOD state",
    ),
    (
        MANIFEST_PATH,
        "FULL file manifest",
    ),
    (
        MANIFEST_SHA_PATH,
        "FULL manifest SHA record",
    ),
    (
        RESTORE_SCRIPT,
        "Packaged restore script",
    ),
    (
        COMPLETE_MARKER,
        "Handoff completion marker",
    ),
]


for path, label in basic_inputs:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 5. LOAD EOD STATE
# ============================================================

state = json.loads(
    STATE_PATH.read_text(
        encoding="utf-8"
    )
)


print()
print(
    "EOD state:"
)

print(
    "  handoff version:",
    state.get(
        "handoff_version"
    ),
)

print(
    "  phase:",
    state.get(
        "current_phase"
    ),
)

print(
    "  latest optimizer step:",
    state[
        "latest_resume_checkpoint"
    ][
        "optimizer_step"
    ],
)

print(
    "  latest epoch:",
    state[
        "latest_resume_checkpoint"
    ][
        "epoch"
    ],
)


state_checkpoint_sha = (
    state[
        "latest_resume_checkpoint"
    ][
        "sha256"
    ]
)


if (
    state_checkpoint_sha
    != EXPECTED_CHECKPOINT_1500_SHA
):

    raise RuntimeError(
        "EOD state contains unexpected 1500 checkpoint SHA.\n"
        f"Expected: {EXPECTED_CHECKPOINT_1500_SHA}\n"
        f"State:    {state_checkpoint_sha}"
    )


print(
    "EOD checkpoint identity: PASS"
)


# ============================================================
# 6. VERIFY MANIFEST FILE ITSELF
# ============================================================

expected_manifest_sha = (
    MANIFEST_SHA_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
    .split()[0]
)

actual_manifest_sha = sha256_file(
    MANIFEST_PATH
)


if (
    actual_manifest_sha
    != expected_manifest_sha
):

    raise RuntimeError(
        "FULL_FILE_MANIFEST.json SHA mismatch.\n"
        f"Expected: {expected_manifest_sha}\n"
        f"Actual:   {actual_manifest_sha}"
    )


print()
print(
    "FULL manifest SHA: PASS"
)


# ============================================================
# 7. VERIFY EXTRACTED FULL PACKAGE
#
# Important:
# Missing workspace/*.zip archives are tolerated ONLY because
# Kaggle may transform nested ZIPs during dataset ingestion.
#
# All non-ZIP operational files must still match the manifest.
# ============================================================

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

manifest_files = manifest[
    "files"
]

expected_file_count = int(
    manifest[
        "file_count"
    ]
)


if (
    len(
        manifest_files
    )
    != expected_file_count
):

    raise RuntimeError(
        "Manifest internal file-count mismatch."
    )


print()
print(
    "Verifying expanded FULL package..."
)

print(
    f"Manifest entries: {expected_file_count}"
)


verified_count = 0
verified_bytes = 0

missing_safe_archives = []
missing_critical = []

size_failures = []
sha_failures = []


for index, entry in enumerate(
    manifest_files,
    start=1,
):

    relative = Path(
        entry[
            "path"
        ]
    )

    path = (
        FULL_ROOT
        / relative
    )


    if not path.exists():

        if safe_missing_nested_archive(
            relative
        ):

            missing_safe_archives.append(
                str(
                    relative
                )
            )

        else:

            missing_critical.append(
                str(
                    relative
                )
            )

        continue


    expected_size = int(
        entry[
            "bytes"
        ]
    )

    actual_size = int(
        path.stat().st_size
    )


    if actual_size != expected_size:

        size_failures.append(
            {
                "path":
                    str(
                        relative
                    ),

                "expected":
                    expected_size,

                "actual":
                    actual_size,
            }
        )

        continue


    expected_sha = entry[
        "sha256"
    ]

    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        sha_failures.append(
            {
                "path":
                    str(
                        relative
                    ),

                "expected":
                    expected_sha,

                "actual":
                    actual_sha,
            }
        )

        continue


    verified_count += 1
    verified_bytes += (
        actual_size
    )


    if (
        verified_count == 1
        or verified_count % 50 == 0
    ):

        print(
            f"  verified operational files: "
            f"{verified_count}"
        )


# ------------------------------------------------------------
# Any missing NON-archive file is fatal.
# ------------------------------------------------------------

if missing_critical:

    print()
    print(
        "CRITICAL missing manifest files:"
    )

    for item in missing_critical:
        print(
            "  ",
            item
        )

    raise RuntimeError(
        "Expanded FULL package is missing operational files."
    )


if size_failures:

    print()
    print(
        "Manifest size failures:"
    )

    for item in size_failures:
        print(
            item
        )

    raise RuntimeError(
        "Expanded FULL package contains size mismatches."
    )


if sha_failures:

    print()
    print(
        "Manifest SHA failures:"
    )

    for item in sha_failures:
        print(
            item
        )

    raise RuntimeError(
        "Expanded FULL package contains SHA mismatches."
    )


print()
print(
    "Operational manifest verification: PASS"
)

print(
    "Verified files:",
    verified_count,
)

print(
    "Verified bytes:",
    human_bytes(
        verified_bytes
    ),
)


if missing_safe_archives:

    print()
    print(
        "Kaggle-expanded historical archive omissions:"
    )

    for item in missing_safe_archives:

        print(
            "  ALLOWED:",
            item
        )

    print()
    print(
        "These are root-level historical ZIP backups only."
    )

    print(
        "They are NOT used to resume the Phase-5C run."
    )

else:

    print(
        "No nested archive omissions detected."
    )


# ============================================================
# 8. VERIFY FULL-PACKAGE BASELINE SNAPSHOTS BEFORE RESTORE
# ============================================================

FULL_RUN_ROOT = (
    FULL_WORKSPACE
    / "dasonet_phase5b"
    / "dasonet_phase5c_baseline_curve1000updates_v1"
)

FULL_SNAPSHOT_DIR = (
    FULL_RUN_ROOT
    / "snapshots"
)


snapshot_names = {
    250:
        "epoch_000_opt_000250.pt",

    500:
        "epoch_001_opt_000500.pt",

    750:
        "epoch_002_opt_000750.pt",

    1000:
        "epoch_003_opt_001000.pt",

    1250:
        "epoch_004_opt_001250.pt",

    1500:
        "epoch_005_opt_001500.pt",
}


print()
print(
    "============================================================"
)

print(
    "VERIFYING ALL SIX BASELINE SNAPSHOTS IN FULL HANDOFF"
)

print(
    "============================================================"
)


snapshot_hashes = (
    state[
        "all_baseline_snapshots"
    ]
)


for step, filename in snapshot_names.items():

    path = (
        FULL_SNAPSHOT_DIR
        / filename
    )

    require(
        path,
        f"FULL {step}-update snapshot",
    )


    expected_sha = (
        snapshot_hashes[
            str(
                step
            )
        ][
            "sha256"
        ]
    )

    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            f"{step}-update snapshot SHA mismatch.\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}"
        )


    print(
        f"PASS  optimizer step {step:4d}  "
        f"{actual_sha[:16]}..."
    )


# ============================================================
# 9. VERIFY 1500 CHECKPOINT CONTENT IN FULL PACKAGE
# ============================================================

FULL_CHECKPOINT_1500 = (
    FULL_SNAPSHOT_DIR
    / snapshot_names[
        1500
    ]
)


full_checkpoint_sha = sha256_file(
    FULL_CHECKPOINT_1500
)


if (
    full_checkpoint_sha
    != EXPECTED_CHECKPOINT_1500_SHA
):

    raise RuntimeError(
        "FULL 1500 checkpoint SHA does not match yesterday."
    )


full_payload = torch.load(
    FULL_CHECKPOINT_1500,
    map_location="cpu",
    weights_only=False,
)


if int(
    full_payload.get(
        "optimizer_step",
        -1,
    )
) != 1500:

    raise RuntimeError(
        "FULL checkpoint optimizer_step != 1500."
    )


if int(
    full_payload.get(
        "epoch",
        -1,
    )
) != 5:

    raise RuntimeError(
        "FULL checkpoint epoch != 5."
    )


print()
print(
    "FULL 1500 checkpoint:"
)

print(
    "  SHA256:        PASS"
)

print(
    "  optimizer_step: 1500"
)

print(
    "  epoch:          5"
)


# ============================================================
# 10. VERIFY INDEPENDENT EMERGENCY 1500 CHECKPOINT
# ============================================================

RESUME_CHECKPOINT_1500 = (
    RESUME_WORKSPACE
    / "dasonet_phase5b"
    / "dasonet_phase5c_baseline_curve1000updates_v1"
    / "snapshots"
    / "epoch_005_opt_001500.pt"
)


require(
    RESUME_CHECKPOINT_1500,
    "Emergency 1500 checkpoint",
)


resume_checkpoint_sha = sha256_file(
    RESUME_CHECKPOINT_1500
)


if (
    resume_checkpoint_sha
    != EXPECTED_CHECKPOINT_1500_SHA
):

    raise RuntimeError(
        "Emergency RESUME checkpoint SHA mismatch."
    )


resume_payload = torch.load(
    RESUME_CHECKPOINT_1500,
    map_location="cpu",
    weights_only=False,
)


if int(
    resume_payload.get(
        "optimizer_step",
        -1,
    )
) != 1500:

    raise RuntimeError(
        "Emergency checkpoint optimizer_step != 1500."
    )


if int(
    resume_payload.get(
        "epoch",
        -1,
    )
) != 5:

    raise RuntimeError(
        "Emergency checkpoint epoch != 5."
    )


print()
print(
    "Independent emergency checkpoint: PASS"
)

print(
    "FULL checkpoint SHA == RESUME checkpoint SHA: PASS"
)


# ============================================================
# 11. VERIFY DECODER BEFORE RESTORE IN BOTH BACKUPS
# ============================================================

FULL_DECODER = (
    FULL_WORKSPACE
    / "DASO-Net-Phase5B-KaggleTraining"
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

RESUME_DECODER = (
    RESUME_WORKSPACE
    / "DASO-Net-Phase5B-KaggleTraining"
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)


for decoder_path, label in [
    (
        FULL_DECODER,
        "FULL decoder",
    ),
    (
        RESUME_DECODER,
        "RESUME decoder",
    ),
]:

    require(
        decoder_path,
        label,
    )

    actual_sha = sha256_file(
        decoder_path
    )

    if actual_sha != EXPECTED_DECODER_SHA:

        raise RuntimeError(
            f"{label} SHA mismatch.\n"
            f"Expected: {EXPECTED_DECODER_SHA}\n"
            f"Actual:   {actual_sha}"
        )


print(
    "Decoder independently preserved in FULL + RESUME: PASS"
)


# ============================================================
# 12. RUN PACKAGED RESTORE SCRIPT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "RESTORING VERIFIED WORKSPACE TO /kaggle/working"
)

print(
    "============================================================"
)


subprocess.run(
    [
        sys.executable,
        str(
            RESTORE_SCRIPT
        ),
    ],
    check=True,
)


# ============================================================
# 13. RESTORED WORKING PATHS
# ============================================================

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

INSTANCE_LOSS = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "losses"
    / "instance.py"
)

BASE_TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)

CHECKPOINTED_LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE_SCRIPT = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL_SCRIPT = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_baseline_curve1000updates_v1.json"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

CHECKPOINT_1500 = (
    SNAPSHOT_DIR
    / "epoch_005_opt_001500.pt"
)

RUN_STATUS = (
    RUN_ROOT
    / "run_status.json"
)

HISTORY = (
    RUN_ROOT
    / "history.csv"
)

LAST_CHECKPOINT = (
    RUN_ROOT
    / "last.pt"
)

BEST_CHECKPOINT = (
    RUN_ROOT
    / "best.pt"
)

EVAL_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "baseline_curve1000updates_v1"
)

SUMMARY_1500 = (
    EVAL_ROOT
    / "baseline_learning_curve_1500_summary.json"
)

METRIC_1500 = (
    EVAL_ROOT
    / "opt_1500"
    / "metrics"
    / "phase5c_metric_summary.json"
)

ORACLE_1500 = (
    EVAL_ROOT
    / "opt_1500"
    / "oracle"
    / "oracle_component_diagnostic.json"
)

COUNTERFACTUAL_1500 = (
    EVAL_ROOT
    / "extended_best_counterfactual"
    / "counterfactual_center_diagnostic.json"
)


critical_restored = [
    (
        CODE_ROOT,
        "Restored code tree",
    ),
    (
        DECODER,
        "Restored decoder",
    ),
    (
        INSTANCE_LOSS,
        "Restored instance loss",
    ),
    (
        BASE_TRAINER,
        "Restored base trainer",
    ),
    (
        CHECKPOINTED_LAUNCHER,
        "Restored checkpoint launcher",
    ),
    (
        EVALUATOR,
        "Restored evaluator",
    ),
    (
        ORACLE_SCRIPT,
        "Restored oracle diagnostic",
    ),
    (
        COUNTERFACTUAL_SCRIPT,
        "Restored counterfactual diagnostic",
    ),
    (
        CONFIG,
        "Restored training config",
    ),
    (
        CHECKPOINT_1500,
        "Restored 1500 checkpoint",
    ),
    (
        RUN_STATUS,
        "Restored run status",
    ),
    (
        HISTORY,
        "Restored history",
    ),
    (
        SUMMARY_1500,
        "Restored 1500 learning-curve summary",
    ),
    (
        METRIC_1500,
        "Restored 1500 metric summary",
    ),
    (
        ORACLE_1500,
        "Restored 1500 oracle summary",
    ),
    (
        COUNTERFACTUAL_1500,
        "Restored 1500 counterfactual summary",
    ),
]


for path, label in critical_restored:

    require(
        path,
        label,
    )


print()
print(
    "Critical restored paths: PASS"
)


# ============================================================
# 14. VERIFY ALL SIX RESTORED SNAPSHOTS AGAIN
# ============================================================

print()
print(
    "Verifying restored snapshot hashes..."
)


for step, filename in snapshot_names.items():

    path = (
        SNAPSHOT_DIR
        / filename
    )

    require(
        path,
        f"Restored {step}-update snapshot",
    )


    expected_sha = (
        snapshot_hashes[
            str(
                step
            )
        ][
            "sha256"
        ]
    )

    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            f"Restored {step}-update snapshot SHA mismatch."
        )


    print(
        f"PASS  restored optimizer step {step}"
    )


# ============================================================
# 15. VERIFY RESTORED 1500 CHECKPOINT
# ============================================================

restored_checkpoint_sha = sha256_file(
    CHECKPOINT_1500
)


if (
    restored_checkpoint_sha
    != EXPECTED_CHECKPOINT_1500_SHA
):

    raise RuntimeError(
        "Restored 1500 checkpoint SHA mismatch."
    )


payload = torch.load(
    CHECKPOINT_1500,
    map_location="cpu",
    weights_only=False,
)


optimizer_step = int(
    payload.get(
        "optimizer_step",
        -1,
    )
)

epoch = int(
    payload.get(
        "epoch",
        -1,
    )
)


if optimizer_step != 1500:

    raise RuntimeError(
        f"Restored optimizer_step is {optimizer_step}, not 1500."
    )


if epoch != 5:

    raise RuntimeError(
        f"Restored epoch is {epoch}, not 5."
    )


print()
print(
    "Restored 1500 checkpoint state: PASS"
)

print(
    "  optimizer_step:",
    optimizer_step,
)

print(
    "  epoch:",
    epoch,
)

print(
    "  SHA256:",
    restored_checkpoint_sha,
)


# ============================================================
# 16. DECODER AUDIT
# ============================================================

restored_decoder_sha = sha256_file(
    DECODER
)


if (
    restored_decoder_sha
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Restored decoder SHA mismatch."
    )


decoder_text = DECODER.read_text(
    encoding="utf-8"
)


for marker in [
    "local_peak_suppression",
    "F.max_pool2d",
    "local_peak_kernel",
]:

    if marker not in decoder_text:

        raise RuntimeError(
            f"Decoder marker missing: {marker}"
        )


print(
    "Validated 3x3 local-peak decoder: PASS"
)


# ============================================================
# 17. DDP ZERO-PARTICLE FIX AUDIT
# ============================================================

instance_text = INSTANCE_LOSS.read_text(
    encoding="utf-8"
)


if (
    "DDP-safe zero-particle handling"
    not in instance_text
):

    raise RuntimeError(
        "DDP-safe zero-particle marker missing."
    )


print(
    "DDP-safe zero-particle handling: PASS"
)


# ============================================================
# 18. PYTORCH 2.6 RESUME FIX AUDIT
# ============================================================

launcher_text = (
    CHECKPOINTED_LAUNCHER
    .read_text(
        encoding="utf-8"
    )
)


required_launcher_markers = [
    "CHECKPOINTED ORIGINAL-PCC BASELINE LAUNCHER v2",
    "trusted_load_checkpoint",
    "weights_only=False",
    "snapshotting_save_checkpoint",
]


for marker in required_launcher_markers:

    if marker not in launcher_text:

        raise RuntimeError(
            "Checkpoint launcher marker missing:\n"
            f"{marker}"
        )


print(
    "PyTorch 2.6 trusted resume loader: PASS"
)


# ============================================================
# 19. PYTHON SYNTAX AUDIT
# ============================================================

for script in [
    BASE_TRAINER,
    CHECKPOINTED_LAUNCHER,
    EVALUATOR,
    ORACLE_SCRIPT,
    COUNTERFACTUAL_SCRIPT,
]:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "py_compile",
            str(
                script
            ),
        ],
        check=True,
    )


print(
    "Core script syntax audit: PASS"
)


# ============================================================
# 20. PCC-CENTER MUST REMAIN DISABLED
# ============================================================

cfg = json.loads(
    CONFIG.read_text(
        encoding="utf-8"
    )
)


pcc_center_cfg = (
    cfg
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if pcc_center_cfg is not None:

    if bool(
        pcc_center_cfg.get(
            "enabled",
            False,
        )
    ):

        raise RuntimeError(
            "PCC-center is unexpectedly ENABLED."
        )


print(
    "PCC-center: DISABLED — PASS"
)


# ============================================================
# 21. RUN STATUS / AMP AUDIT
# ============================================================

run_status = json.loads(
    RUN_STATUS.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    run_status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if amp_skips != 0:

    raise RuntimeError(
        f"Expected zero AMP skips, found {amp_skips}."
    )


print(
    "AMP skipped optimizer updates: 0 — PASS"
)


# ============================================================
# 22. FROZEN PROJECT HASH AUDIT
# ============================================================

frozen_checks = {
    (
        PROJECT_ROOT
        / "data"
        / "mp_morph"
        / "v1"
        / "morphology_manifest_v1.csv"
    ):
        EXPECTED_MORPH_SHA,

    (
        PROJECT_ROOT
        / "data"
        / "fruit_bg"
        / "v1_final"
        / "fruitbg_rgb_v1_manifest.csv"
    ):
        EXPECTED_FRUITBG_SHA,

    (
        PROJECT_ROOT
        / "configs"
        / "renderer_v3_1.json"
    ):
        EXPECTED_RENDERER_JSON_SHA,

    (
        PROJECT_ROOT
        / "src"
        / "dasonet"
        / "synth"
        / "renderer.py"
    ):
        EXPECTED_RENDERER_PY_SHA,
}


print()
print(
    "============================================================"
)

print(
    "FROZEN PROJECT HASH AUDIT"
)

print(
    "============================================================"
)


for path, expected_sha in frozen_checks.items():

    require(
        path,
        "Frozen project file",
    )


    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            "Frozen project hash mismatch:\n"
            f"{path}\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}"
        )


    print(
        f"PASS  {path.name}"
    )


# ============================================================
# 23. LOAD RESTORED 1500 DEVELOPMENT RESULTS
# ============================================================

metric = json.loads(
    METRIC_1500.read_text(
        encoding="utf-8"
    )
)

oracle = json.loads(
    ORACLE_1500.read_text(
        encoding="utf-8"
    )
)

counterfactual = json.loads(
    COUNTERFACTUAL_1500.read_text(
        encoding="utf-8"
    )
)


ap50 = float(
    metric[
        "mask_ap"
    ][
        "mask_ap50"
    ]
)

ap75 = float(
    metric[
        "mask_ap"
    ][
        "mask_ap75"
    ]
)

ap5095 = float(
    metric[
        "mask_ap"
    ][
        "mask_ap50_95"
    ]
)

best_f1 = float(
    metric[
        "validation_threshold_sweep"
    ][
        "best_f1"
    ]
)

best_threshold = float(
    metric[
        "validation_threshold_sweep"
    ][
        "best_threshold_by_f1"
    ]
)

peak_r8 = float(
    oracle[
        "center_head"
    ][
        "top100_peak_recall_within_8px"
    ]
)

peak_r16 = float(
    oracle[
        "center_head"
    ][
        "top100_peak_recall_within_16px"
    ]
)

box_iou = float(
    oracle[
        "oracle_gt_cell_geometry"
    ][
        "box_iou_stats"
    ][
        "mean"
    ]
)

mask_iou = float(
    oracle[
        "oracle_gt_cell_mask"
    ][
        "gt_box_crop_mask_iou_stats"
    ][
        "mean"
    ]
)

exact_lift = float(
    counterfactual[
        "exact_gt_cell"
    ][
        "composite_minus_clean_stats"
    ][
        "mean"
    ]
)

comp_gt_clean = float(
    counterfactual[
        "exact_gt_cell"
    ][
        "fraction_composite_gt_clean"
    ]
)


# ============================================================
# 24. GPU RUNTIME AUDIT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "GPU RUNTIME AUDIT"
)

print(
    "============================================================"
)


cuda_available = (
    torch.cuda.is_available()
)

gpu_count = (
    torch.cuda.device_count()
)


print(
    "CUDA available:",
    cuda_available,
)

print(
    "GPU count:",
    gpu_count,
)


gpu_names = []


for index in range(
    gpu_count
):

    name = (
        torch.cuda.get_device_name(
            index
        )
    )

    gpu_names.append(
        name
    )

    print(
        f"GPU {index}: {name}"
    )


frozen_runtime_ready = (
    cuda_available
    and gpu_count == 2
    and all(
        "T4" in name
        for name in gpu_names
    )
)


if frozen_runtime_ready:

    print(
        "Frozen 2x T4 runtime: PASS"
    )

else:

    print()
    print(
        "WARNING:"
    )

    print(
        "The restore is valid, but this notebook is not currently "
        "using the frozen 2x T4 runtime."
    )

    print(
        "Do NOT start the next training continuation until "
        "the accelerator is 2x T4."
    )


# ============================================================
# 25. FINAL RESTORE REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "RESTORED PHASE 5C STOPPING POINT"
)

print(
    "============================================================"
)

print(
    f"Mask AP50:              {ap50:.4f}"
)

print(
    f"Mask AP75:              {ap75:.4f}"
)

print(
    f"Mask AP50:95:           {ap5095:.4f}"
)

print(
    f"Best validation F1:     {best_f1:.4f}"
)

print(
    f"Best val threshold:     {best_threshold:.2f}"
)

print(
    f"Peak recall <= 8 px:    {peak_r8:.4f}"
)

print(
    f"Peak recall <=16 px:    {peak_r16:.4f}"
)

print(
    f"Oracle box IoU mean:    {box_iou:.4f}"
)

print(
    f"GT-box mask IoU mean:   {mask_iou:.4f}"
)

print(
    f"Counterfactual lift:    {exact_lift:+.4f}"
)

print(
    f"Composite > clean:      {comp_gt_clean:.4f}"
)


print()
print(
    "============================================================"
)

print(
    "DASO-NET NEW NOTEBOOK RESTORE/AUDIT v2: PASS"
)

print(
    "============================================================"
)

print(
    "Persistent project dataset:       PASS"
)

print(
    "Operational FULL manifest:        PASS"
)

print(
    "Historical nested ZIP omission:   tolerated only if listed above"
)

print(
    "FULL 1500 checkpoint:             PASS"
)

print(
    "Emergency 1500 checkpoint:        PASS"
)

print(
    "All six learning snapshots:       PASS"
)

print(
    "Workspace restoration:            PASS"
)

print(
    "Validated decoder:                PASS"
)

print(
    "DDP zero-particle fix:            PASS"
)

print(
    "PyTorch 2.6 resume fix:           PASS"
)

print(
    "Original feature/embedding PCC:   RETAINED"
)

print(
    "CRPA:                             RETAINED"
)

print(
    "PCC-center:                       REJECTED / DISABLED"
)

print(
    "External-test tuning:             NOT PERFORMED"
)

print()

print(
    "CURRENT RESUME POINT:"
)

print(
    CHECKPOINT_1500
)

print(
    "optimizer_step = 1500"
)

print(
    "epoch          = 5"
)

print()

if frozen_runtime_ready:

    print(
        "Runtime is ready for the next controlled continuation."
    )

else:

    print(
        "Restore is complete, but fix the GPU runtime before training."
    )

print()

print(
    "STOP HERE."
)

print(
    "Paste the output from the final sections before we continue training."
)

DASO-Net — NEW NOTEBOOK RESTORE/AUDIT v2
PASS  Persistent DASO-Net project dataset
PASS  Phase-5C EOD handoff dataset
PASS  Expanded FULL handoff
PASS  Expanded emergency RESUME handoff
PASS  FULL packaged workspace
PASS  RESUME packaged workspace
PASS  Phase-5C EOD state
PASS  FULL file manifest
PASS  FULL manifest SHA record
PASS  Packaged restore script
PASS  Handoff completion marker

EOD state:
  handoff version: DASO-Net-Phase5C-EOD-20260913-v1
  phase: Phase 5C
  latest optimizer step: 1500
  latest epoch: 5
EOD checkpoint identity: PASS

FULL manifest SHA: PASS

Verifying expanded FULL package...
Manifest entries: 173
  verified operational files: 1
  verified operational files: 50
  verified operational files: 100
  verified operational files: 150

Operational manifest verification: PASS
Verified files: 172
Verified bytes: 374.14 MiB

Kaggle-expanded historical archive omissions:
  ALLOWED: workspace/DASO-Net-Phase5B-FROZEN-v1.zip

These are root-level historical ZIP backups o

## Continue baseline from 1500 → 2500

In [4]:
# ============================================================
# PHASE 5C — ORIGINAL-PCC BASELINE CONTINUATION
#
# Resume:
#   optimizer step 1500
#   epoch 5
#
# Continue through:
#   1750
#   2000
#   2250
#   2500 optimizer updates
#
# Architecture/loss/data remain unchanged.
# PCC-center remains DISABLED.
#
# Expected runtime: ~80–90 minutes on 2x T4.
# ============================================================

from __future__ import annotations

import hashlib
import json
import subprocess
import sys
import time
from pathlib import Path

import torch


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_baseline_curve1000updates_v1.json"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

RESUME = (
    SNAPSHOT_DIR
    / "epoch_005_opt_001500.pt"
)


EXPECTED_RESUME_SHA = (
    "258b9d4457c367373c07f7e346844559"
    "1fe824dc5ceef27e8cd338fc4b76c6f5"
)


expected_new_snapshots = {
    1750:
        SNAPSHOT_DIR
        / "epoch_006_opt_001750.pt",

    2000:
        SNAPSHOT_DIR
        / "epoch_007_opt_002000.pt",

    2250:
        SNAPSHOT_DIR
        / "epoch_008_opt_002250.pt",

    2500:
        SNAPSHOT_DIR
        / "epoch_009_opt_002500.pt",
}


# ============================================================
# HELPERS
# ============================================================

def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def require(
    path: Path,
    label: str,
):

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


# ============================================================
# PRE-FLIGHT AUDIT
# ============================================================

for path, label in [
    (
        PROJECT_ROOT,
        "Project dataset",
    ),
    (
        CODE_ROOT,
        "Restored code root",
    ),
    (
        LAUNCHER,
        "Checkpointed launcher",
    ),
    (
        CONFIG,
        "Training config",
    ),
    (
        RESUME,
        "1500 resume checkpoint",
    ),
]:

    require(
        path,
        label,
    )


# Exact checkpoint identity.
resume_sha = sha256_file(
    RESUME
)

if resume_sha != EXPECTED_RESUME_SHA:

    raise RuntimeError(
        "1500 resume checkpoint SHA mismatch.\n"
        f"Expected: {EXPECTED_RESUME_SHA}\n"
        f"Actual:   {resume_sha}"
    )


resume_payload = torch.load(
    RESUME,
    map_location="cpu",
    weights_only=False,
)


if int(
    resume_payload.get(
        "optimizer_step",
        -1,
    )
) != 1500:

    raise RuntimeError(
        "Resume checkpoint optimizer_step != 1500."
    )


if int(
    resume_payload.get(
        "epoch",
        -1,
    )
) != 5:

    raise RuntimeError(
        "Resume checkpoint epoch != 5."
    )


# Confirm correct launcher.
launcher_text = LAUNCHER.read_text(
    encoding="utf-8"
)


for marker in [
    "CHECKPOINTED ORIGINAL-PCC BASELINE LAUNCHER v2",
    "trusted_load_checkpoint",
    "weights_only=False",
    "snapshotting_save_checkpoint",
]:

    if marker not in launcher_text:

        raise RuntimeError(
            f"Launcher marker missing: {marker}"
        )


# Confirm PCC-center still disabled.
cfg = json.loads(
    CONFIG.read_text(
        encoding="utf-8"
    )
)


pcc_center_cfg = (
    cfg
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if (
    pcc_center_cfg is not None
    and bool(
        pcc_center_cfg.get(
            "enabled",
            False,
        )
    )
):

    raise RuntimeError(
        "PCC-center unexpectedly enabled."
    )


# Confirm GPU runtime.
if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA unavailable."
    )


if torch.cuda.device_count() != 2:

    raise RuntimeError(
        "Expected exactly 2 GPUs."
    )


gpu_names = [
    torch.cuda.get_device_name(
        i
    )
    for i in range(
        2
    )
]


if not all(
    "T4" in name
    for name in gpu_names
):

    raise RuntimeError(
        f"Expected 2x T4, got {gpu_names}"
    )


print(
    "============================================================"
)

print(
    "PHASE 5C — CONTINUE ORIGINAL-PCC BASELINE"
)

print(
    "1500 -> 2500 OPTIMIZER UPDATES"
)

print(
    "============================================================"
)

print(
    "Resume checkpoint SHA: PASS"
)

print(
    "Resume optimizer step: 1500"
)

print(
    "Resume epoch:          5"
)

print(
    "Next epochs:           6, 7, 8, 9"
)

print(
    "Updates per epoch:     250"
)

print(
    "Final target:          2500"
)

print(
    "PCC-center:            DISABLED"
)

print(
    "Original PCC:          RETAINED"
)

print(
    "CRPA:                  RETAINED"
)

print(
    "GPU runtime:           2x T4"
)

print(
    "LR:                    3e-4 constant"
)

print()


# ============================================================
# TRAINING COMMAND
#
# start_epoch loaded from checkpoint = 6
# --epochs 10 therefore executes epochs 6,7,8,9 only.
# ============================================================

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(
        LAUNCHER
    ),

    "--project-root",
    str(
        PROJECT_ROOT
    ),

    "--config",
    str(
        CONFIG
    ),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "10",

    "--max-train-steps",
    "1000",

    "--max-val-steps",
    "100",

    "--accum",
    "4",

    "--num-workers",
    "2",

    "--log-every",
    "100",

    "--resume",
    str(
        RESUME
    ),
]


print(
    " ".join(
        cmd
    )
)

print()


start = time.time()


subprocess.run(
    cmd,
    check=True,
)


elapsed = (
    time.time()
    - start
)


# ============================================================
# POST-RUN AUDIT
# ============================================================

print()
print(
    "Verifying new chronological snapshots..."
)


for step, path in expected_new_snapshots.items():

    require(
        path,
        f"{step}-update snapshot",
    )

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    actual_step = int(
        payload.get(
            "optimizer_step",
            -1,
        )
    )

    expected_epoch = {
        1750: 6,
        2000: 7,
        2250: 8,
        2500: 9,
    }[
        step
    ]

    actual_epoch = int(
        payload.get(
            "epoch",
            -1,
        )
    )


    if actual_step != step:

        raise RuntimeError(
            f"{step} checkpoint contains "
            f"optimizer_step={actual_step}."
        )


    if actual_epoch != expected_epoch:

        raise RuntimeError(
            f"{step} checkpoint contains "
            f"epoch={actual_epoch}; "
            f"expected {expected_epoch}."
        )


    print(
        f"PASS  "
        f"step={step}  "
        f"epoch={actual_epoch}  "
        f"sha={sha256_file(path)[:16]}..."
    )


status_path = (
    RUN_ROOT
    / "run_status.json"
)


require(
    status_path,
    "run_status.json",
)


status = json.loads(
    status_path.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if amp_skips != 0:

    raise RuntimeError(
        f"Unexpected AMP skips: {amp_skips}"
    )


final_payload = torch.load(
    expected_new_snapshots[
        2500
    ],
    map_location="cpu",
    weights_only=False,
)


print()
print(
    "============================================================"
)

print(
    "BASELINE CONTINUATION 1500 -> 2500 COMPLETE"
)

print(
    "============================================================"
)

print(
    f"Wall time:              "
    f"{elapsed / 60.0:.2f} min"
)

print(
    "Final optimizer step:   ",
    final_payload[
        "optimizer_step"
    ],
)

print(
    "Final epoch:            ",
    final_payload[
        "epoch"
    ],
)

print(
    "AMP skipped updates:    ",
    amp_skips,
)

print(
    "Final scaler scale:     ",
    final_payload.get(
        "scaler_state_dict",
        {},
    ).get(
        "scale"
    ),
)

print()

print(
    "========================================"
)

print(
    "PHASE 5C BASELINE 2500-UPDATES: PASS"
)

print(
    "========================================"
)

PHASE 5C — CONTINUE ORIGINAL-PCC BASELINE
1500 -> 2500 OPTIMIZER UPDATES
Resume checkpoint SHA: PASS
Resume optimizer step: 1500
Resume epoch:          5
Next epochs:           6, 7, 8, 9
Updates per epoch:     250
Final target:          2500
PCC-center:            DISABLED
Original PCC:          RETAINED
CRPA:                  RETAINED
GPU runtime:           2x T4
LR:                    3e-4 constant

torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/launch_phase5c_checkpointed_baseline.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5c_baseline_curve1000updates_v1.json --output-root /kaggle/working --epochs 10 --max-train-steps 1000 --max-val-steps 100 --accum 4 --num-workers 2 --log-every 100 --resume /kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1/snapshots/epoch_005_opt_001500.pt



W0914 05:04:22.507000 129 torch/distributed/run.py:852] 
W0914 05:04:22.507000 129 torch/distributed/run.py:852] *****************************************
W0914 05:04:22.507000 129 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0914 05:04:22.507000 129 torch/distributed/run.py:852] *****************************************
[W914 05:04:22.836507269 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files
DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files


[W914 05:04:25.778165434 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W914 05:04:25.778801924 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


Resumed from: /kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1/snapshots/epoch_005_opt_001500.pt
=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     1000
Optimizer steps total:      2500
Smoke mode:                 False
epoch=06 step=00000/01000 loss=4.7225 inst=4.4347 pcc=0.0000 crpa=0.2878 lr=3.00e-04 vram_alloc=8.85GB vram_res=9.12GB
epoch=06 step=00100/01000 loss=1.7424 inst=1.7112 pcc=0

## Evaluate 1750 / 2000 / 2250 / 2500 and build the complete 250→2500 curve

In [5]:
# ============================================================
# PHASE 5C — ORIGINAL-PCC LEARNING CURVE TO 2500 UPDATES
#
# Existing evaluated:
#   250
#   500
#   750
#   1000
#   1250
#   1500
#
# New:
#   1750
#   2000
#   2250
#   2500
#
# SAME deterministic 200-scene synthetic validation subset.
#
# No external real test data is touched.
# ============================================================

from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

EVAL_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "baseline_curve1000updates_v1"
)


# ============================================================
# ALL LEARNING-CURVE CHECKPOINTS
# ============================================================

snapshot_names = {
    250:
        "epoch_000_opt_000250.pt",

    500:
        "epoch_001_opt_000500.pt",

    750:
        "epoch_002_opt_000750.pt",

    1000:
        "epoch_003_opt_001000.pt",

    1250:
        "epoch_004_opt_001250.pt",

    1500:
        "epoch_005_opt_001500.pt",

    1750:
        "epoch_006_opt_001750.pt",

    2000:
        "epoch_007_opt_002000.pt",

    2250:
        "epoch_008_opt_002250.pt",

    2500:
        "epoch_009_opt_002500.pt",
}


snapshots = {
    step:
        SNAPSHOT_DIR
        / filename

    for step, filename
    in snapshot_names.items()
}


new_steps = [
    1750,
    2000,
    2250,
    2500,
]


all_steps = [
    250,
    500,
    750,
    1000,
    1250,
    1500,
    1750,
    2000,
    2250,
    2500,
]


for required in [
    EVALUATOR,
    ORACLE,
    COUNTERFACTUAL,
    *snapshots.values(),
]:

    if not required.exists():

        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


# ============================================================
# EVALUATE NEW CHECKPOINTS ONLY
# ============================================================

for optimizer_step in new_steps:

    checkpoint = (
        snapshots[
            optimizer_step
        ]
    )

    print()
    print(
        "============================================================"
    )

    print(
        f"EVALUATING BASELINE @ "
        f"{optimizer_step} OPTIMIZER UPDATES"
    )

    print(
        "============================================================"
    )


    step_root = (
        EVAL_ROOT
        / f"opt_{optimizer_step:04d}"
    )

    metric_out = (
        step_root
        / "metrics"
    )

    oracle_out = (
        step_root
        / "oracle"
    )


    # --------------------------------------------------------
    # Instance-segmentation metrics
    # --------------------------------------------------------

    subprocess.run(
        [
            sys.executable,
            str(
                EVALUATOR
            ),

            "--project-root",
            str(
                PROJECT_ROOT
            ),

            "--checkpoint",
            str(
                checkpoint
            ),

            "--output-dir",
            str(
                metric_out
            ),

            "--split",
            "val",

            "--max-scenes",
            "200",

            "--seed",
            "20260917",

            "--num-workers",
            "2",

            "--decode-min-score",
            "0.001",

            "--operating-score",
            "0.05",

            "--topk",
            "100",

            "--nms-iou",
            "0.5",

            "--mask-threshold",
            "0.5",

            "--match-iou",
            "0.5",
        ],
        check=True,
    )


    # --------------------------------------------------------
    # Oracle component diagnostic
    # --------------------------------------------------------

    subprocess.run(
        [
            sys.executable,
            str(
                ORACLE
            ),

            "--project-root",
            str(
                PROJECT_ROOT
            ),

            "--checkpoint",
            str(
                checkpoint
            ),

            "--evaluator",
            str(
                EVALUATOR
            ),

            "--output-dir",
            str(
                oracle_out
            ),

            "--split",
            "val",

            "--max-scenes",
            "200",

            "--seed",
            "20260917",

            "--num-workers",
            "2",

            "--stride",
            "4",

            "--peak-kernel",
            "3",

            "--topk",
            "100",

            "--mask-threshold",
            "0.5",
        ],
        check=True,
    )


# ============================================================
# LOAD COMPLETE 250 -> 2500 CURVE
# ============================================================

records = {}

sample_particle_distribution = None
sample_fruit_distribution = None


for step in all_steps:

    step_root = (
        EVAL_ROOT
        / f"opt_{step:04d}"
    )

    metric_path = (
        step_root
        / "metrics"
        / "phase5c_metric_summary.json"
    )

    oracle_path = (
        step_root
        / "oracle"
        / "oracle_component_diagnostic.json"
    )


    if not metric_path.exists():

        raise RuntimeError(
            f"Metric file missing for {step}:\n"
            f"{metric_path}"
        )


    if not oracle_path.exists():

        raise RuntimeError(
            f"Oracle file missing for {step}:\n"
            f"{oracle_path}"
        )


    metric = json.loads(
        metric_path.read_text(
            encoding="utf-8"
        )
    )

    oracle = json.loads(
        oracle_path.read_text(
            encoding="utf-8"
        )
    )


    # --------------------------------------------------------
    # Confirm SAME deterministic validation sample.
    # --------------------------------------------------------

    pc_dist = metric[
        "sample_particle_count_distribution"
    ]

    fruit_dist = metric[
        "sample_fruit_distribution"
    ]


    if sample_particle_distribution is None:

        sample_particle_distribution = (
            pc_dist
        )

        sample_fruit_distribution = (
            fruit_dist
        )


    if (
        pc_dist
        != sample_particle_distribution
    ):

        raise RuntimeError(
            f"Particle-count sample mismatch at {step} updates."
        )


    if (
        fruit_dist
        != sample_fruit_distribution
    ):

        raise RuntimeError(
            f"Fruit sample mismatch at {step} updates."
        )


    ap = metric[
        "mask_ap"
    ]

    op = metric[
        "operating_point"
    ]

    sweep = metric[
        "validation_threshold_sweep"
    ]

    center = oracle[
        "center_head"
    ]

    geom = oracle[
        "oracle_gt_cell_geometry"
    ]

    mask = oracle[
        "oracle_gt_cell_mask"
    ]

    empty = oracle[
        "empty_scene_center_behavior"
    ]


    records[
        step
    ] = {
        "AP50":
            float(
                ap[
                    "mask_ap50"
                ]
            ),

        "AP75":
            float(
                ap[
                    "mask_ap75"
                ]
            ),

        "AP50_95":
            float(
                ap[
                    "mask_ap50_95"
                ]
            ),

        "Precision005":
            float(
                op[
                    "precision"
                ]
            ),

        "Recall005":
            float(
                op[
                    "recall"
                ]
            ),

        "F1005":
            float(
                op[
                    "f1"
                ]
            ),

        "MatchedDice":
            float(
                op[
                    "matched_mask_dice"
                ]
            ),

        "CountMAE":
            float(
                op[
                    "count_mae"
                ]
            ),

        "EmptyFP005":
            float(
                op[
                    "empty_scene_fp_per_image"
                ]
            ),

        "BestThreshold":
            float(
                sweep[
                    "best_threshold_by_f1"
                ]
            ),

        "BestF1":
            float(
                sweep[
                    "best_f1"
                ]
            ),

        "GTCenterScore":
            float(
                center[
                    "gt_cell_score_stats"
                ][
                    "mean"
                ]
            ),

        "PeakRecall4":
            float(
                center[
                    "top100_peak_recall_within_4px"
                ]
            ),

        "PeakRecall8":
            float(
                center[
                    "top100_peak_recall_within_8px"
                ]
            ),

        "PeakRecall16":
            float(
                center[
                    "top100_peak_recall_within_16px"
                ]
            ),

        "PeakRecall32":
            float(
                center[
                    "top100_peak_recall_within_32px"
                ]
            ),

        "CenterError":
            float(
                geom[
                    "center_error_px_stats"
                ][
                    "mean"
                ]
            ),

        "BoxIoU":
            float(
                geom[
                    "box_iou_stats"
                ][
                    "mean"
                ]
            ),

        "BoxIoU50":
            float(
                geom[
                    "box_iou_ge_0_50"
                ]
            ),

        "BoxIoU75":
            float(
                geom[
                    "box_iou_ge_0_75"
                ]
            ),

        "WidthRatio":
            float(
                geom[
                    "pred_to_gt_width_ratio_stats"
                ][
                    "median"
                ]
            ),

        "HeightRatio":
            float(
                geom[
                    "pred_to_gt_height_ratio_stats"
                ][
                    "median"
                ]
            ),

        "GTBoxMaskIoU":
            float(
                mask[
                    "gt_box_crop_mask_iou_stats"
                ][
                    "mean"
                ]
            ),

        "GTBoxMaskIoU50":
            float(
                mask[
                    "gt_box_crop_mask_iou_ge_0_50"
                ]
            ),

        "PredBoxMaskIoU":
            float(
                mask[
                    "pred_box_crop_mask_iou_stats"
                ][
                    "mean"
                ]
            ),

        "EmptyMaxPeak":
            float(
                empty[
                    "max_peak_score_stats"
                ][
                    "mean"
                ]
            ),

        "EmptyPeaks010":
            float(
                empty[
                    "mean_local_peak_count_by_threshold"
                ][
                    "0.10"
                ]
            ),
    }


# ============================================================
# TABLE 1 — PRIMARY CURVE
# ============================================================

print()
print(
    "================================================================================================"
)

print(
    "PHASE 5C — ORIGINAL-PCC BASELINE LEARNING CURVE TO 2500"
)

print(
    "================================================================================================"
)

print(
    f"{'Updates':>8}"
    f"{'AP50':>10}"
    f"{'AP75':>10}"
    f"{'AP50:95':>11}"
    f"{'BestF1':>10}"
    f"{'PeakR@8':>10}"
    f"{'BoxIoU':>10}"
    f"{'MaskIoU':>10}"
)

print(
    "-" * 79
)


for step in all_steps:

    r = records[
        step
    ]

    print(
        f"{step:>8d}"
        f"{r['AP50']:>10.4f}"
        f"{r['AP75']:>10.4f}"
        f"{r['AP50_95']:>11.4f}"
        f"{r['BestF1']:>10.4f}"
        f"{r['PeakRecall8']:>10.4f}"
        f"{r['BoxIoU']:>10.4f}"
        f"{r['GTBoxMaskIoU']:>10.4f}"
    )


# ============================================================
# TABLE 2 — DIAGNOSTIC CURVE
# ============================================================

print()
print(
    "------------------------------------------------------------------------------------------------"
)

print(
    f"{'Updates':>8}"
    f"{'GTScore':>10}"
    f"{'PeakR@4':>10}"
    f"{'PeakR@16':>11}"
    f"{'W ratio':>10}"
    f"{'H ratio':>10}"
    f"{'Mask>=.5':>11}"
    f"{'BestThr':>10}"
    f"{'FP@.05':>10}"
)

print(
    "-" * 90
)


for step in all_steps:

    r = records[
        step
    ]

    print(
        f"{step:>8d}"
        f"{r['GTCenterScore']:>10.4f}"
        f"{r['PeakRecall4']:>10.4f}"
        f"{r['PeakRecall16']:>11.4f}"
        f"{r['WidthRatio']:>10.4f}"
        f"{r['HeightRatio']:>10.4f}"
        f"{r['GTBoxMaskIoU50']:>11.4f}"
        f"{r['BestThreshold']:>10.2f}"
        f"{r['EmptyFP005']:>10.2f}"
    )


# ============================================================
# PROVISIONAL BEST DEVELOPMENT CHECKPOINT
#
# Primary criterion:
#   AP50:95
#
# Secondary:
#   Best F1
#
# Still DEVELOPMENT ONLY.
# ============================================================

best_step = max(
    all_steps,
    key=lambda step: (
        records[
            step
        ][
            "AP50_95"
        ],
        records[
            step
        ][
            "BestF1"
        ],
    ),
)


best_checkpoint = (
    snapshots[
        best_step
    ]
)


print()
print(
    "============================================================"
)

print(
    "PROVISIONAL BEST DEVELOPMENT CHECKPOINT"
)

print(
    "============================================================"
)

print(
    "Optimizer updates:",
    best_step,
)

print(
    "AP50:95:",
    f"{records[best_step]['AP50_95']:.4f}",
)

print(
    "AP50:",
    f"{records[best_step]['AP50']:.4f}",
)

print(
    "BestF1:",
    f"{records[best_step]['BestF1']:.4f}",
)

print(
    "PeakR@8:",
    f"{records[best_step]['PeakRecall8']:.4f}",
)

print(
    "BoxIoU:",
    f"{records[best_step]['BoxIoU']:.4f}",
)

print(
    "MaskIoU:",
    f"{records[best_step]['GTBoxMaskIoU']:.4f}",
)

print(
    "Checkpoint:"
)

print(
    best_checkpoint
)


# ============================================================
# COUNTERFACTUAL DIAGNOSTIC
#
# Run only on provisional best checkpoint.
# ============================================================

cf_out = (
    EVAL_ROOT
    / "curve2500_best_counterfactual"
)


subprocess.run(
    [
        sys.executable,
        str(
            COUNTERFACTUAL
        ),

        "--project-root",
        str(
            PROJECT_ROOT
        ),

        "--checkpoint",
        str(
            best_checkpoint
        ),

        "--evaluator",
        str(
            EVALUATOR
        ),

        "--output-dir",
        str(
            cf_out
        ),

        "--split",
        "val",

        "--max-scenes",
        "200",

        "--seed",
        "20260917",

        "--num-workers",
        "2",

        "--stride",
        "4",

        "--peak-kernel",
        "3",

        "--window-radius-feature",
        "2",
    ],
    check=True,
)


cf = json.loads(
    (
        cf_out
        / "counterfactual_center_diagnostic.json"
    ).read_text(
        encoding="utf-8"
    )
)


print()
print(
    "============================================================"
)

print(
    "BEST-CHECKPOINT COUNTERFACTUAL SUMMARY"
)

print(
    "============================================================"
)

print(
    "Exact GT-cell lift mean:",
    f"{cf['exact_gt_cell']['composite_minus_clean_stats']['mean']:+.4f}",
)

print(
    "Composite > clean:",
    f"{cf['exact_gt_cell']['fraction_composite_gt_clean']:.4f}",
)

print(
    "Exact lift >= 0.01:",
    f"{cf['exact_gt_cell']['fraction_lift_ge_0_01']:.4f}",
)

print(
    "Local peak lift mean:",
    f"{cf['local_window_peak']['composite_minus_clean_stats']['mean']:+.4f}",
)

print(
    "Local composite > clean:",
    f"{cf['local_window_peak']['fraction_composite_gt_clean']:.4f}",
)

print(
    "Clean max peak mean:",
    f"{cf['scene_level']['clean_max_peak_stats']['mean']:.4f}",
)

print(
    "Outside max peak mean:",
    f"{cf['scene_level']['composite_outside_effect_max_peak_stats']['mean']:.4f}",
)

print(
    "Effect max peak mean:",
    f"{cf['scene_level']['composite_effect_max_peak_stats']['mean']:.4f}",
)


# ============================================================
# SAVE COMPLETE DEVELOPMENT CURVE
# ============================================================

summary = {
    "phase":
        "5C",

    "paper_result":
        False,

    "experiment":
        "original_pcc_baseline_learning_curve_to_2500_updates",

    "pcc_center":
        "rejected_and_disabled",

    "checkpoint_records":
        {
            str(
                step
            ):
                records[
                    step
                ]

            for step in all_steps
        },

    "provisional_best_optimizer_step":
        int(
            best_step
        ),

    "provisional_best_checkpoint":
        str(
            best_checkpoint
        ),

    "selection_rule":
        (
            "Synthetic validation development only: "
            "maximize mask AP50:95; BestF1 secondary."
        ),

    "particle_count_distribution":
        sample_particle_distribution,

    "fruit_distribution":
        sample_fruit_distribution,

    "best_checkpoint_counterfactual":
        cf,
}


summary_path = (
    EVAL_ROOT
    / "baseline_learning_curve_2500_summary.json"
)


summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print(
    "Summary JSON:"
)

print(
    summary_path
)

print()
print(
    "============================================================"
)

print(
    "PHASE 5C LEARNING CURVE TO 2500: PASS"
)

print(
    "============================================================"
)

print(
    "STOP HERE."
)

print(
    "Do not continue beyond 2500 until this curve is reviewed."
)


EVALUATING BASELINE @ 1750 OPTIMIZER UPDATES
eval 0001/0200 scene=369b96d7adde8c0026a7 gt=1 decoded=100 op_pred=13 elapsed=0.1m
eval 0011/0200 scene=8aab26301d02d13c97c2 gt=3 decoded=99 op_pred=15 elapsed=0.2m
eval 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 decoded=97 op_pred=25 elapsed=0.3m
eval 0031/0200 scene=fe667dc0752460a37390 gt=3 decoded=98 op_pred=17 elapsed=0.4m
eval 0041/0200 scene=11e8bd15bed065be3400 gt=2 decoded=99 op_pred=21 elapsed=0.5m
eval 0051/0200 scene=87b7a8761b585200d3ff gt=1 decoded=99 op_pred=41 elapsed=0.6m
eval 0061/0200 scene=a3cb9b50dca442394cbe gt=2 decoded=99 op_pred=28 elapsed=0.7m
eval 0071/0200 scene=25a9d6b2837276a9dbb7 gt=1 decoded=99 op_pred=52 elapsed=0.8m
eval 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 decoded=99 op_pred=58 elapsed=0.9m
eval 0091/0200 scene=22aeb0f719d57f84b0f7 gt=0 decoded=99 op_pred=37 elapsed=1.0m
eval 0101/0200 scene=d3df4c3febde0d7548bc gt=0 decoded=98 op_pred=73 elapsed=1.1m
eval 0111/0200 scene=858baec99120971a3e47 gt=3 deco

KeyError: 'matched_mask_dice'

In [6]:
# ============================================================
# PHASE 5C — REBUILD COMPLETE ORIGINAL-PCC LEARNING CURVE
#
# IMPORTANT:
# All evaluations 1750/2000/2250/2500 already completed.
#
# This cell DOES NOT rerun those evaluations.
#
# It:
#   1. loads existing 250 -> 2500 JSON outputs
#   2. builds the complete learning curve
#   3. selects development checkpoint by:
#          primary   = AP50:95
#          secondary = Best F1
#   4. runs counterfactual diagnostic ONLY on selected best
#   5. writes corrected summary JSON
#
# No external real dataset is touched.
# ============================================================

from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

COUNTERFACTUAL = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

EVAL_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "baseline_curve1000updates_v1"
)


# ============================================================
# ALL CHECKPOINTS
# ============================================================

snapshot_names = {
    250:
        "epoch_000_opt_000250.pt",

    500:
        "epoch_001_opt_000500.pt",

    750:
        "epoch_002_opt_000750.pt",

    1000:
        "epoch_003_opt_001000.pt",

    1250:
        "epoch_004_opt_001250.pt",

    1500:
        "epoch_005_opt_001500.pt",

    1750:
        "epoch_006_opt_001750.pt",

    2000:
        "epoch_007_opt_002000.pt",

    2250:
        "epoch_008_opt_002250.pt",

    2500:
        "epoch_009_opt_002500.pt",
}


steps = list(
    snapshot_names.keys()
)


snapshots = {
    step:
        SNAPSHOT_DIR
        / filename

    for step, filename
    in snapshot_names.items()
}


# ============================================================
# REQUIRED FILE AUDIT
# ============================================================

for step in steps:

    checkpoint = (
        snapshots[
            step
        ]
    )

    metric_path = (
        EVAL_ROOT
        / f"opt_{step:04d}"
        / "metrics"
        / "phase5c_metric_summary.json"
    )

    oracle_path = (
        EVAL_ROOT
        / f"opt_{step:04d}"
        / "oracle"
        / "oracle_component_diagnostic.json"
    )


    if not checkpoint.exists():

        raise RuntimeError(
            f"Checkpoint missing for {step}:\n"
            f"{checkpoint}"
        )


    if not metric_path.exists():

        raise RuntimeError(
            f"Metric summary missing for {step}:\n"
            f"{metric_path}"
        )


    if not oracle_path.exists():

        raise RuntimeError(
            f"Oracle diagnostic missing for {step}:\n"
            f"{oracle_path}"
        )


print(
    "All 10 checkpoints + metric/oracle outputs: PASS"
)


# ============================================================
# LOAD COMPLETE CURVE
#
# Uses only fields already verified in previous working cells.
# ============================================================

records = {}


for step in steps:

    root = (
        EVAL_ROOT
        / f"opt_{step:04d}"
    )

    metric = json.loads(
        (
            root
            / "metrics"
            / "phase5c_metric_summary.json"
        ).read_text(
            encoding="utf-8"
        )
    )

    oracle = json.loads(
        (
            root
            / "oracle"
            / "oracle_component_diagnostic.json"
        ).read_text(
            encoding="utf-8"
        )
    )


    ap = (
        metric[
            "mask_ap"
        ]
    )

    operating = (
        metric[
            "operating_point"
        ]
    )

    sweep = (
        metric[
            "validation_threshold_sweep"
        ]
    )

    center = (
        oracle[
            "center_head"
        ]
    )

    geometry = (
        oracle[
            "oracle_gt_cell_geometry"
        ]
    )

    masks = (
        oracle[
            "oracle_gt_cell_mask"
        ]
    )


    records[
        step
    ] = {
        "AP50":
            float(
                ap[
                    "mask_ap50"
                ]
            ),

        "AP75":
            float(
                ap[
                    "mask_ap75"
                ]
            ),

        "AP50_95":
            float(
                ap[
                    "mask_ap50_95"
                ]
            ),

        "OperatingPrecision":
            float(
                operating[
                    "precision"
                ]
            ),

        "OperatingRecall":
            float(
                operating[
                    "recall"
                ]
            ),

        "OperatingF1":
            float(
                operating[
                    "f1"
                ]
            ),

        "CountMAE":
            float(
                operating[
                    "count_mae"
                ]
            ),

        "EmptyFP005":
            float(
                operating[
                    "empty_scene_fp_per_image"
                ]
            ),

        "BestThreshold":
            float(
                sweep[
                    "best_threshold_by_f1"
                ]
            ),

        "BestF1":
            float(
                sweep[
                    "best_f1"
                ]
            ),

        "GTCenterScore":
            float(
                center[
                    "gt_cell_score_stats"
                ][
                    "mean"
                ]
            ),

        "PeakRecall4":
            float(
                center[
                    "top100_peak_recall_within_4px"
                ]
            ),

        "PeakRecall8":
            float(
                center[
                    "top100_peak_recall_within_8px"
                ]
            ),

        "PeakRecall16":
            float(
                center[
                    "top100_peak_recall_within_16px"
                ]
            ),

        "PeakRecall32":
            float(
                center[
                    "top100_peak_recall_within_32px"
                ]
            ),

        "BoxIoU":
            float(
                geometry[
                    "box_iou_stats"
                ][
                    "mean"
                ]
            ),

        "BoxIoU50":
            float(
                geometry[
                    "box_iou_ge_0_50"
                ]
            ),

        "BoxIoU75":
            float(
                geometry[
                    "box_iou_ge_0_75"
                ]
            ),

        "WidthRatio":
            float(
                geometry[
                    "pred_to_gt_width_ratio_stats"
                ][
                    "median"
                ]
            ),

        "HeightRatio":
            float(
                geometry[
                    "pred_to_gt_height_ratio_stats"
                ][
                    "median"
                ]
            ),

        "GTBoxMaskIoU":
            float(
                masks[
                    "gt_box_crop_mask_iou_stats"
                ][
                    "mean"
                ]
            ),

        "GTBoxMaskIoU50":
            float(
                masks[
                    "gt_box_crop_mask_iou_ge_0_50"
                ]
            ),

        "PredBoxMaskIoU":
            float(
                masks[
                    "pred_box_crop_mask_iou_stats"
                ][
                    "mean"
                ]
            ),
    }


# ============================================================
# PRIMARY TABLE
# ============================================================

print()
print(
    "=============================================================================================="
)

print(
    "PHASE 5C — ORIGINAL-PCC BASELINE LEARNING CURVE: 250 -> 2500"
)

print(
    "=============================================================================================="
)

print(
    f"{'Updates':>8}"
    f"{'AP50':>10}"
    f"{'AP75':>10}"
    f"{'AP50:95':>11}"
    f"{'BestF1':>10}"
    f"{'PeakR@8':>10}"
    f"{'BoxIoU':>10}"
    f"{'MaskIoU':>10}"
)

print(
    "-" * 79
)


for step in steps:

    r = (
        records[
            step
        ]
    )

    print(
        f"{step:>8d}"
        f"{r['AP50']:>10.4f}"
        f"{r['AP75']:>10.4f}"
        f"{r['AP50_95']:>11.4f}"
        f"{r['BestF1']:>10.4f}"
        f"{r['PeakRecall8']:>10.4f}"
        f"{r['BoxIoU']:>10.4f}"
        f"{r['GTBoxMaskIoU']:>10.4f}"
    )


# ============================================================
# COMPONENT TABLE
# ============================================================

print()
print(
    "----------------------------------------------------------------------------------------------"
)

print(
    f"{'Updates':>8}"
    f"{'GTScore':>10}"
    f"{'PeakR@4':>10}"
    f"{'PeakR@16':>11}"
    f"{'W ratio':>10}"
    f"{'H ratio':>10}"
    f"{'Mask>=.5':>11}"
    f"{'BestThr':>10}"
    f"{'FP@.05':>10}"
)

print(
    "-" * 90
)


for step in steps:

    r = (
        records[
            step
        ]
    )

    print(
        f"{step:>8d}"
        f"{r['GTCenterScore']:>10.4f}"
        f"{r['PeakRecall4']:>10.4f}"
        f"{r['PeakRecall16']:>11.4f}"
        f"{r['WidthRatio']:>10.4f}"
        f"{r['HeightRatio']:>10.4f}"
        f"{r['GTBoxMaskIoU50']:>11.4f}"
        f"{r['BestThreshold']:>10.2f}"
        f"{r['EmptyFP005']:>10.2f}"
    )


# ============================================================
# RECENT TRAJECTORY
# ============================================================

print()
print(
    "============================================================"
)

print(
    "RECENT TRAJECTORY — 2000 / 2250 / 2500"
)

print(
    "============================================================"
)


for step in [
    2000,
    2250,
    2500,
]:

    r = (
        records[
            step
        ]
    )

    print(
        f"{step}: "
        f"AP50:95={r['AP50_95']:.4f}, "
        f"BestF1={r['BestF1']:.4f}, "
        f"PeakR@8={r['PeakRecall8']:.4f}, "
        f"BoxIoU={r['BoxIoU']:.4f}, "
        f"MaskIoU={r['GTBoxMaskIoU']:.4f}, "
        f"FP@.05={r['EmptyFP005']:.2f}"
    )


# ============================================================
# EXACT RECENT DELTAS
# ============================================================

def delta(
    newer: int,
    older: int,
    key: str,
) -> float:

    return (
        records[
            newer
        ][
            key
        ]
        -
        records[
            older
        ][
            key
        ]
    )


print()
print(
    "2250 -> 2500 delta:"
)

for key in [
    "AP50",
    "AP75",
    "AP50_95",
    "BestF1",
    "PeakRecall8",
    "BoxIoU",
    "GTBoxMaskIoU",
]:

    print(
        f"  {key:14s}: "
        f"{delta(2500, 2250, key):+.4f}"
    )


# ============================================================
# PREDEFINED DEVELOPMENT-SELECTION RULE
#
# PRIMARY:   AP50:95
# SECONDARY: BestF1
#
# No external data involved.
# ============================================================

best_step = max(
    steps,
    key=lambda step: (
        records[
            step
        ][
            "AP50_95"
        ],
        records[
            step
        ][
            "BestF1"
        ],
    ),
)


best_checkpoint = (
    snapshots[
        best_step
    ]
)


print()
print(
    "============================================================"
)

print(
    "PROVISIONAL BEST DEVELOPMENT CHECKPOINT"
)

print(
    "============================================================"
)

print(
    "Selection rule:"
)

print(
    "  primary   = synthetic-val mask AP50:95"
)

print(
    "  secondary = synthetic-val BestF1"
)

print()

print(
    "Optimizer updates:",
    best_step,
)

print(
    "AP50:",
    f"{records[best_step]['AP50']:.4f}",
)

print(
    "AP75:",
    f"{records[best_step]['AP75']:.4f}",
)

print(
    "AP50:95:",
    f"{records[best_step]['AP50_95']:.4f}",
)

print(
    "BestF1:",
    f"{records[best_step]['BestF1']:.4f}",
)

print(
    "Checkpoint:"
)

print(
    best_checkpoint
)


# ============================================================
# COUNTERFACTUAL DIAGNOSTIC ON SELECTED BEST ONLY
# ============================================================

cf_out = (
    EVAL_ROOT
    / "curve2500_best_counterfactual"
)


subprocess.run(
    [
        sys.executable,
        str(
            COUNTERFACTUAL
        ),

        "--project-root",
        str(
            PROJECT_ROOT
        ),

        "--checkpoint",
        str(
            best_checkpoint
        ),

        "--evaluator",
        str(
            EVALUATOR
        ),

        "--output-dir",
        str(
            cf_out
        ),

        "--split",
        "val",

        "--max-scenes",
        "200",

        "--seed",
        "20260917",

        "--num-workers",
        "2",

        "--stride",
        "4",

        "--peak-kernel",
        "3",

        "--window-radius-feature",
        "2",
    ],
    check=True,
)


cf_path = (
    cf_out
    / "counterfactual_center_diagnostic.json"
)


if not cf_path.exists():

    raise RuntimeError(
        "Counterfactual output JSON missing."
    )


cf = json.loads(
    cf_path.read_text(
        encoding="utf-8"
    )
)


exact_lift = float(
    cf[
        "exact_gt_cell"
    ][
        "composite_minus_clean_stats"
    ][
        "mean"
    ]
)

exact_fraction = float(
    cf[
        "exact_gt_cell"
    ][
        "fraction_composite_gt_clean"
    ]
)

local_lift = float(
    cf[
        "local_window_peak"
    ][
        "composite_minus_clean_stats"
    ][
        "mean"
    ]
)

local_fraction = float(
    cf[
        "local_window_peak"
    ][
        "fraction_composite_gt_clean"
    ]
)

clean_max = float(
    cf[
        "scene_level"
    ][
        "clean_max_peak_stats"
    ][
        "mean"
    ]
)

outside_max = float(
    cf[
        "scene_level"
    ][
        "composite_outside_effect_max_peak_stats"
    ][
        "mean"
    ]
)

effect_max = float(
    cf[
        "scene_level"
    ][
        "composite_effect_max_peak_stats"
    ][
        "mean"
    ]
)


print()
print(
    "============================================================"
)

print(
    "BEST-CHECKPOINT COUNTERFACTUAL SUMMARY"
)

print(
    "============================================================"
)

print(
    "Exact GT-cell lift mean:",
    f"{exact_lift:+.4f}",
)

print(
    "Composite > clean:",
    f"{exact_fraction:.4f}",
)

print(
    "Local peak lift mean:",
    f"{local_lift:+.4f}",
)

print(
    "Local composite > clean:",
    f"{local_fraction:.4f}",
)

print(
    "Clean max peak mean:",
    f"{clean_max:.4f}",
)

print(
    "Outside max peak mean:",
    f"{outside_max:.4f}",
)

print(
    "Effect max peak mean:",
    f"{effect_max:.4f}",
)


# ============================================================
# SAVE CORRECTED 2500 SUMMARY
# ============================================================

summary = {
    "phase":
        "5C",

    "paper_result":
        False,

    "experiment":
        "original_pcc_baseline_learning_curve_to_2500_updates",

    "architecture_changed":
        False,

    "original_pcc":
        "retained",

    "crpa":
        "retained",

    "pcc_center":
        "rejected_and_disabled",

    "selection_data":
        "synthetic_validation_only",

    "external_test_tuning":
        False,

    "selection_rule":
        (
            "Primary mask AP50:95; "
            "BestF1 secondary."
        ),

    "records":
        {
            str(
                step
            ):
                records[
                    step
                ]

            for step in steps
        },

    "provisional_best_optimizer_step":
        int(
            best_step
        ),

    "provisional_best_checkpoint":
        str(
            best_checkpoint
        ),

    "best_counterfactual":
        {
            "exact_lift_mean":
                exact_lift,

            "exact_composite_gt_clean_fraction":
                exact_fraction,

            "local_lift_mean":
                local_lift,

            "local_composite_gt_clean_fraction":
                local_fraction,

            "clean_max_peak_mean":
                clean_max,

            "outside_max_peak_mean":
                outside_max,

            "effect_max_peak_mean":
                effect_max,
        },

    "development_note":
        (
            "2500-update curve is used only to determine "
            "the final training budget. It is not a final "
            "paper test result."
        ),
}


summary_path = (
    EVAL_ROOT
    / "baseline_learning_curve_2500_summary.json"
)


summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print(
    "Corrected summary:"
)

print(
    summary_path
)

print()
print(
    "========================================"
)

print(
    "PHASE 5C LEARNING CURVE TO 2500: PASS"
)

print(
    "========================================"
)

print(
    "STOP HERE."
)

print(
    "Do not start additional training until this curve is reviewed."
)

All 10 checkpoints + metric/oracle outputs: PASS

PHASE 5C — ORIGINAL-PCC BASELINE LEARNING CURVE: 250 -> 2500
 Updates      AP50      AP75    AP50:95    BestF1   PeakR@8    BoxIoU   MaskIoU
-------------------------------------------------------------------------------
     250    0.0231    0.0001     0.0073    0.0552    0.2575    0.2980    0.3116
     500    0.0273    0.0026     0.0083    0.0958    0.4225    0.3966    0.3446
     750    0.1179    0.0152     0.0400    0.1873    0.5525    0.4583    0.3847
    1000    0.1244    0.0293     0.0538    0.2094    0.6075    0.5002    0.4096
    1250    0.1647    0.0266     0.0513    0.2649    0.6675    0.5037    0.4225
    1500    0.1794    0.0334     0.0610    0.2815    0.7125    0.5275    0.4345
    1750    0.1845    0.0576     0.0712    0.2775    0.7400    0.5377    0.4057
    2000    0.2156    0.0581     0.0893    0.3039    0.7500    0.5585    0.4480
    2250    0.2171    0.0435     0.0739    0.3550    0.7775    0.5529    0.4713
    2500 

## Final schedule probe, 2500 → 3000

In [7]:
# ============================================================
# PHASE 5C — FINAL ORIGINAL-PCC SCHEDULE PROBE
#
# Resume:
#   optimizer step 2500
#   epoch 9
#
# Continue:
#   epoch 10 -> 2750
#   epoch 11 -> 3000
#
# IMPORTANT:
#   This is the FINAL bounded extension used to determine
#   training duration.
#
#   After 3000 we STOP schedule-search and freeze the budget
#   using a higher-resolution synthetic-validation audit.
#
# No architecture/loss/data changes.
# PCC-center remains rejected/disabled.
# ============================================================

from __future__ import annotations

import hashlib
import json
import subprocess
import time
from pathlib import Path

import torch


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_baseline_curve1000updates_v1.json"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

RESUME = (
    SNAPSHOT_DIR
    / "epoch_009_opt_002500.pt"
)

SNAP_2750 = (
    SNAPSHOT_DIR
    / "epoch_010_opt_002750.pt"
)

SNAP_3000 = (
    SNAPSHOT_DIR
    / "epoch_011_opt_003000.pt"
)


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path: Path) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def require(path: Path, label: str):

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


# ============================================================
# PRE-FLIGHT
# ============================================================

for path, label in [
    (PROJECT_ROOT, "Project dataset"),
    (CODE_ROOT, "Code root"),
    (LAUNCHER, "Checkpoint launcher"),
    (CONFIG, "Training config"),
    (RESUME, "2500 checkpoint"),
]:
    require(path, label)


payload = torch.load(
    RESUME,
    map_location="cpu",
    weights_only=False,
)


resume_step = int(
    payload.get(
        "optimizer_step",
        -1,
    )
)

resume_epoch = int(
    payload.get(
        "epoch",
        -1,
    )
)


if resume_step != 2500:

    raise RuntimeError(
        f"Expected optimizer step 2500; got {resume_step}."
    )


if resume_epoch != 9:

    raise RuntimeError(
        f"Expected epoch 9; got {resume_epoch}."
    )


resume_sha = sha256_file(
    RESUME
)


# ------------------------------------------------------------
# Launcher integrity
# ------------------------------------------------------------

launcher_text = LAUNCHER.read_text(
    encoding="utf-8"
)


for marker in [
    "CHECKPOINTED ORIGINAL-PCC BASELINE LAUNCHER v2",
    "trusted_load_checkpoint",
    "weights_only=False",
    "snapshotting_save_checkpoint",
]:

    if marker not in launcher_text:

        raise RuntimeError(
            f"Launcher marker missing: {marker}"
        )


# ------------------------------------------------------------
# PCC-center audit
# ------------------------------------------------------------

cfg = json.loads(
    CONFIG.read_text(
        encoding="utf-8"
    )
)


pcc_center_cfg = (
    cfg
    .get("loss", {})
    .get("pcc_center")
)


if (
    pcc_center_cfg is not None
    and bool(
        pcc_center_cfg.get(
            "enabled",
            False,
        )
    )
):
    raise RuntimeError(
        "PCC-center unexpectedly enabled."
    )


# ------------------------------------------------------------
# Runtime audit
# ------------------------------------------------------------

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA unavailable."
    )


if torch.cuda.device_count() != 2:

    raise RuntimeError(
        "Expected exactly 2 GPUs."
    )


gpu_names = [
    torch.cuda.get_device_name(i)
    for i in range(2)
]


if not all(
    "T4" in name
    for name in gpu_names
):

    raise RuntimeError(
        f"Expected 2x T4; got {gpu_names}"
    )


# ============================================================
# REPORT
# ============================================================

print(
    "============================================================"
)

print(
    "PHASE 5C — FINAL SCHEDULE PROBE"
)

print(
    "2500 -> 3000 OPTIMIZER UPDATES"
)

print(
    "============================================================"
)

print(
    "Resume checkpoint:      PASS"
)

print(
    "Resume SHA256:"
)

print(
    resume_sha
)

print()

print(
    "Resume optimizer step:  2500"
)

print(
    "Resume epoch:           9"
)

print(
    "Next epochs:            10, 11"
)

print(
    "New checkpoints:        2750, 3000"
)

print(
    "Final target:           3000"
)

print(
    "PCC-center:             DISABLED"
)

print(
    "Original PCC:           RETAINED"
)

print(
    "CRPA:                   RETAINED"
)

print(
    "LR:                     3e-4 constant"
)

print(
    "Runtime:                2x T4"
)

print()


# ============================================================
# TRAIN
#
# Resume start_epoch = 10.
# epochs=12 executes epoch 10 and epoch 11.
# ============================================================

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(LAUNCHER),

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "12",

    "--max-train-steps",
    "1000",

    "--max-val-steps",
    "100",

    "--accum",
    "4",

    "--num-workers",
    "2",

    "--log-every",
    "100",

    "--resume",
    str(RESUME),
]


print(
    " ".join(cmd)
)

print()


start = time.time()


subprocess.run(
    cmd,
    check=True,
)


elapsed = (
    time.time()
    - start
)


# ============================================================
# VERIFY NEW CHECKPOINTS
# ============================================================

expected = {
    2750: (SNAP_2750, 10),
    3000: (SNAP_3000, 11),
}


print()
print(
    "Verifying final schedule-probe checkpoints..."
)


for step, (
    path,
    expected_epoch,
) in expected.items():

    require(
        path,
        f"{step}-update checkpoint",
    )

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    actual_step = int(
        checkpoint.get(
            "optimizer_step",
            -1,
        )
    )

    actual_epoch = int(
        checkpoint.get(
            "epoch",
            -1,
        )
    )


    if actual_step != step:

        raise RuntimeError(
            f"{step} snapshot contains "
            f"optimizer_step={actual_step}."
        )


    if actual_epoch != expected_epoch:

        raise RuntimeError(
            f"{step} snapshot contains epoch={actual_epoch}; "
            f"expected {expected_epoch}."
        )


    print(
        f"PASS  "
        f"step={step}  "
        f"epoch={actual_epoch}  "
        f"sha={sha256_file(path)[:20]}..."
    )


# ============================================================
# STATUS
# ============================================================

status_path = (
    RUN_ROOT
    / "run_status.json"
)


require(
    status_path,
    "run_status.json",
)


status = json.loads(
    status_path.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if amp_skips != 0:

    raise RuntimeError(
        f"Unexpected AMP skips: {amp_skips}"
    )


final_payload = torch.load(
    SNAP_3000,
    map_location="cpu",
    weights_only=False,
)


print()
print(
    "============================================================"
)

print(
    "FINAL SCHEDULE PROBE TRAINING COMPLETE"
)

print(
    "============================================================"
)

print(
    f"Wall time:             {elapsed / 60.0:.2f} min"
)

print(
    "Final optimizer step: ",
    final_payload["optimizer_step"],
)

print(
    "Final epoch:          ",
    final_payload["epoch"],
)

print(
    "AMP skips:            ",
    amp_skips,
)

print(
    "GradScaler scale:     ",
    final_payload
    .get(
        "scaler_state_dict",
        {},
    )
    .get("scale"),
)

print()
print(
    "========================================"
)

print(
    "PHASE 5C BASELINE 3000-UPDATES: PASS"
)

print(
    "========================================"
)

PHASE 5C — FINAL SCHEDULE PROBE
2500 -> 3000 OPTIMIZER UPDATES
Resume checkpoint:      PASS
Resume SHA256:
a5d555e94f5f69b654f28cb2276711dd7a0475b4fd5a88f2b998295baffa9412

Resume optimizer step:  2500
Resume epoch:           9
Next epochs:            10, 11
New checkpoints:        2750, 3000
Final target:           3000
PCC-center:             DISABLED
Original PCC:           RETAINED
CRPA:                   RETAINED
LR:                     3e-4 constant
Runtime:                2x T4

torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/launch_phase5c_checkpointed_baseline.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5c_baseline_curve1000updates_v1.json --output-root /kaggle/working --epochs 12 --max-train-steps 1000 --max-val-steps 100 --accum 4 --num-workers 2 --log-every 100 --resume /kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1/snapsho

W0914 06:46:36.186000 564 torch/distributed/run.py:852] 
W0914 06:46:36.186000 564 torch/distributed/run.py:852] *****************************************
W0914 06:46:36.186000 564 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0914 06:46:36.186000 564 torch/distributed/run.py:852] *****************************************
[W914 06:46:36.506919871 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files
DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files


[W914 06:46:38.053831551 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W914 06:46:39.124893214 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


Resumed from: /kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1/snapshots/epoch_009_opt_002500.pt
=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     1000
Optimizer steps total:      3000
Smoke mode:                 False
epoch=10 step=00000/01000 loss=1.1005 inst=1.0875 pcc=0.0000 crpa=0.0130 lr=3.00e-04 vram_alloc=8.85GB vram_res=9.12GB
epoch=10 step=00100/01000 loss=0.1600 inst=0.1476 pcc=0

## Evaluate 2750 and 3000 + complete 250→3000 curve

In [8]:
# ============================================================
# PHASE 5C — FINAL 200-SCENE SCHEDULE-PROBE EVALUATION
#
# New evaluations:
#   2750
#   3000
#
# Existing results 250..2500 are reused.
#
# SAME deterministic:
#   split = val
#   scenes = 200
#   seed = 20260917
#
# No external data.
# ============================================================

from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

EVAL_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "baseline_curve1000updates_v1"
)


# ============================================================
# CHECKPOINT MAP
# ============================================================

snapshot_names = {
    250:  "epoch_000_opt_000250.pt",
    500:  "epoch_001_opt_000500.pt",
    750:  "epoch_002_opt_000750.pt",
    1000: "epoch_003_opt_001000.pt",
    1250: "epoch_004_opt_001250.pt",
    1500: "epoch_005_opt_001500.pt",
    1750: "epoch_006_opt_001750.pt",
    2000: "epoch_007_opt_002000.pt",
    2250: "epoch_008_opt_002250.pt",
    2500: "epoch_009_opt_002500.pt",
    2750: "epoch_010_opt_002750.pt",
    3000: "epoch_011_opt_003000.pt",
}


steps = list(
    snapshot_names.keys()
)


snapshots = {
    step:
        SNAPSHOT_DIR
        / filename

    for step, filename
    in snapshot_names.items()
}


for path in [
    EVALUATOR,
    ORACLE,
    COUNTERFACTUAL,
    *snapshots.values(),
]:

    if not path.exists():

        raise RuntimeError(
            f"Missing:\n{path}"
        )


# ============================================================
# EVALUATE ONLY 2750 + 3000
# ============================================================

for step in [
    2750,
    3000,
]:

    print()
    print(
        "============================================================"
    )

    print(
        f"EVALUATING BASELINE @ {step} OPTIMIZER UPDATES"
    )

    print(
        "============================================================"
    )


    root = (
        EVAL_ROOT
        / f"opt_{step:04d}"
    )

    metrics_dir = (
        root
        / "metrics"
    )

    oracle_dir = (
        root
        / "oracle"
    )


    subprocess.run(
        [
            sys.executable,
            str(EVALUATOR),

            "--project-root",
            str(PROJECT_ROOT),

            "--checkpoint",
            str(
                snapshots[
                    step
                ]
            ),

            "--output-dir",
            str(metrics_dir),

            "--split",
            "val",

            "--max-scenes",
            "200",

            "--seed",
            "20260917",

            "--num-workers",
            "2",

            "--decode-min-score",
            "0.001",

            "--operating-score",
            "0.05",

            "--topk",
            "100",

            "--nms-iou",
            "0.5",

            "--mask-threshold",
            "0.5",

            "--match-iou",
            "0.5",
        ],
        check=True,
    )


    subprocess.run(
        [
            sys.executable,
            str(ORACLE),

            "--project-root",
            str(PROJECT_ROOT),

            "--checkpoint",
            str(
                snapshots[
                    step
                ]
            ),

            "--evaluator",
            str(EVALUATOR),

            "--output-dir",
            str(oracle_dir),

            "--split",
            "val",

            "--max-scenes",
            "200",

            "--seed",
            "20260917",

            "--num-workers",
            "2",

            "--stride",
            "4",

            "--peak-kernel",
            "3",

            "--topk",
            "100",

            "--mask-threshold",
            "0.5",
        ],
        check=True,
    )


# ============================================================
# READ COMPLETE CURVE
# ============================================================

records = {}


for step in steps:

    root = (
        EVAL_ROOT
        / f"opt_{step:04d}"
    )

    metric_path = (
        root
        / "metrics"
        / "phase5c_metric_summary.json"
    )

    oracle_path = (
        root
        / "oracle"
        / "oracle_component_diagnostic.json"
    )


    if not metric_path.exists():

        raise RuntimeError(
            f"Missing metrics for {step}."
        )


    if not oracle_path.exists():

        raise RuntimeError(
            f"Missing oracle results for {step}."
        )


    metric = json.loads(
        metric_path.read_text(
            encoding="utf-8"
        )
    )

    oracle = json.loads(
        oracle_path.read_text(
            encoding="utf-8"
        )
    )


    ap = metric["mask_ap"]

    operating = (
        metric[
            "operating_point"
        ]
    )

    sweep = (
        metric[
            "validation_threshold_sweep"
        ]
    )

    center = (
        oracle[
            "center_head"
        ]
    )

    geometry = (
        oracle[
            "oracle_gt_cell_geometry"
        ]
    )

    masks = (
        oracle[
            "oracle_gt_cell_mask"
        ]
    )


    records[
        step
    ] = {
        "AP50":
            float(
                ap["mask_ap50"]
            ),

        "AP75":
            float(
                ap["mask_ap75"]
            ),

        "AP50_95":
            float(
                ap["mask_ap50_95"]
            ),

        "BestF1":
            float(
                sweep["best_f1"]
            ),

        "BestThreshold":
            float(
                sweep[
                    "best_threshold_by_f1"
                ]
            ),

        "OperatingRecall":
            float(
                operating["recall"]
            ),

        "CountMAE":
            float(
                operating["count_mae"]
            ),

        "EmptyFP005":
            float(
                operating[
                    "empty_scene_fp_per_image"
                ]
            ),

        "GTCenterScore":
            float(
                center[
                    "gt_cell_score_stats"
                ]["mean"]
            ),

        "PeakRecall4":
            float(
                center[
                    "top100_peak_recall_within_4px"
                ]
            ),

        "PeakRecall8":
            float(
                center[
                    "top100_peak_recall_within_8px"
                ]
            ),

        "PeakRecall16":
            float(
                center[
                    "top100_peak_recall_within_16px"
                ]
            ),

        "BoxIoU":
            float(
                geometry[
                    "box_iou_stats"
                ]["mean"]
            ),

        "WidthRatio":
            float(
                geometry[
                    "pred_to_gt_width_ratio_stats"
                ]["median"]
            ),

        "HeightRatio":
            float(
                geometry[
                    "pred_to_gt_height_ratio_stats"
                ]["median"]
            ),

        "GTBoxMaskIoU":
            float(
                masks[
                    "gt_box_crop_mask_iou_stats"
                ]["mean"]
            ),

        "GTBoxMaskIoU50":
            float(
                masks[
                    "gt_box_crop_mask_iou_ge_0_50"
                ]
            ),
    }


# ============================================================
# PRIMARY TABLE
# ============================================================

print()
print(
    "=============================================================================================="
)

print(
    "PHASE 5C — COMPLETE ORIGINAL-PCC LEARNING CURVE: 250 -> 3000"
)

print(
    "=============================================================================================="
)

print(
    f"{'Updates':>8}"
    f"{'AP50':>10}"
    f"{'AP75':>10}"
    f"{'AP50:95':>11}"
    f"{'BestF1':>10}"
    f"{'PeakR@8':>10}"
    f"{'BoxIoU':>10}"
    f"{'MaskIoU':>10}"
)

print(
    "-" * 79
)


for step in steps:

    r = records[
        step
    ]

    print(
        f"{step:>8d}"
        f"{r['AP50']:>10.4f}"
        f"{r['AP75']:>10.4f}"
        f"{r['AP50_95']:>11.4f}"
        f"{r['BestF1']:>10.4f}"
        f"{r['PeakRecall8']:>10.4f}"
        f"{r['BoxIoU']:>10.4f}"
        f"{r['GTBoxMaskIoU']:>10.4f}"
    )


# ============================================================
# LATE-STAGE TABLE
# ============================================================

print()
print(
    "=============================================================================================="
)

print(
    "LATE-STAGE SCHEDULE REGION"
)

print(
    "=============================================================================================="
)

print(
    f"{'Updates':>8}"
    f"{'AP50:95':>11}"
    f"{'BestF1':>10}"
    f"{'PeakR@8':>10}"
    f"{'PeakR@16':>11}"
    f"{'BoxIoU':>10}"
    f"{'MaskIoU':>10}"
    f"{'Mask>=.5':>11}"
    f"{'FP@.05':>10}"
)

print(
    "-" * 91
)


for step in [
    1750,
    2000,
    2250,
    2500,
    2750,
    3000,
]:

    r = records[
        step
    ]

    print(
        f"{step:>8d}"
        f"{r['AP50_95']:>11.4f}"
        f"{r['BestF1']:>10.4f}"
        f"{r['PeakRecall8']:>10.4f}"
        f"{r['PeakRecall16']:>11.4f}"
        f"{r['BoxIoU']:>10.4f}"
        f"{r['GTBoxMaskIoU']:>10.4f}"
        f"{r['GTBoxMaskIoU50']:>11.4f}"
        f"{r['EmptyFP005']:>10.2f}"
    )


# ============================================================
# 2500 -> 2750 -> 3000 DELTAS
# ============================================================

def delta(
    newer,
    older,
    key,
):

    return (
        records[newer][key]
        -
        records[older][key]
    )


print()
print(
    "2500 -> 2750 deltas:"
)

for key in [
    "AP50",
    "AP75",
    "AP50_95",
    "BestF1",
    "PeakRecall8",
    "BoxIoU",
    "GTBoxMaskIoU",
]:

    print(
        f"  {key:14s}: "
        f"{delta(2750, 2500, key):+.4f}"
    )


print()
print(
    "2750 -> 3000 deltas:"
)

for key in [
    "AP50",
    "AP75",
    "AP50_95",
    "BestF1",
    "PeakRecall8",
    "BoxIoU",
    "GTBoxMaskIoU",
]:

    print(
        f"  {key:14s}: "
        f"{delta(3000, 2750, key):+.4f}"
    )


# ============================================================
# PROVISIONAL 200-SCENE WINNER
#
# Still not final duration freeze.
# Higher-resolution val audit follows.
# ============================================================

best_step = max(
    steps,
    key=lambda step: (
        records[
            step
        ]["AP50_95"],

        records[
            step
        ]["BestF1"],
    ),
)


best_checkpoint = (
    snapshots[
        best_step
    ]
)


print()
print(
    "============================================================"
)

print(
    "200-SCENE PROVISIONAL WINNER"
)

print(
    "============================================================"
)

print(
    "Primary selection criterion: AP50:95"
)

print(
    "Secondary criterion:        BestF1"
)

print()

print(
    "Optimizer updates:",
    best_step,
)

print(
    "AP50:",
    f"{records[best_step]['AP50']:.4f}",
)

print(
    "AP75:",
    f"{records[best_step]['AP75']:.4f}",
)

print(
    "AP50:95:",
    f"{records[best_step]['AP50_95']:.4f}",
)

print(
    "BestF1:",
    f"{records[best_step]['BestF1']:.4f}",
)

print(
    "Checkpoint:"
)

print(
    best_checkpoint
)


# ============================================================
# COUNTERFACTUAL ON 200-SCENE WINNER
# ============================================================

cf_out = (
    EVAL_ROOT
    / "curve3000_best_counterfactual"
)


subprocess.run(
    [
        sys.executable,
        str(COUNTERFACTUAL),

        "--project-root",
        str(PROJECT_ROOT),

        "--checkpoint",
        str(best_checkpoint),

        "--evaluator",
        str(EVALUATOR),

        "--output-dir",
        str(cf_out),

        "--split",
        "val",

        "--max-scenes",
        "200",

        "--seed",
        "20260917",

        "--num-workers",
        "2",

        "--stride",
        "4",

        "--peak-kernel",
        "3",

        "--window-radius-feature",
        "2",
    ],
    check=True,
)


cf_path = (
    cf_out
    / "counterfactual_center_diagnostic.json"
)


cf = json.loads(
    cf_path.read_text(
        encoding="utf-8"
    )
)


print()
print(
    "============================================================"
)

print(
    "PROVISIONAL-WINNER COUNTERFACTUAL"
)

print(
    "============================================================"
)

print(
    "Exact lift:",
    f"{cf['exact_gt_cell']['composite_minus_clean_stats']['mean']:+.4f}",
)

print(
    "Composite > clean:",
    f"{cf['exact_gt_cell']['fraction_composite_gt_clean']:.4f}",
)

print(
    "Local lift:",
    f"{cf['local_window_peak']['composite_minus_clean_stats']['mean']:+.4f}",
)

print(
    "Effect max:",
    f"{cf['scene_level']['composite_effect_max_peak_stats']['mean']:.4f}",
)

print(
    "Clean max:",
    f"{cf['scene_level']['clean_max_peak_stats']['mean']:.4f}",
)


# ============================================================
# SAVE COMPLETE 3000 SUMMARY
# ============================================================

summary_path = (
    EVAL_ROOT
    / "baseline_learning_curve_3000_summary.json"
)


summary = {
    "phase":
        "5C",

    "paper_result":
        False,

    "purpose":
        "final_training_duration_search",

    "schedule_search_upper_bound":
        3000,

    "schedule_search_complete_after_this":
        True,

    "selection_rule":
        {
            "primary":
                "synthetic validation mask AP50:95",

            "secondary":
                "synthetic validation BestF1",
        },

    "records":
        {
            str(step):
                records[step]

            for step in steps
        },

    "provisional_200_scene_winner":
        int(best_step),

    "provisional_checkpoint":
        str(best_checkpoint),

    "counterfactual":
        cf,

    "next_action":
        (
            "Run higher-resolution synthetic-validation "
            "duration-selection audit on late checkpoints, "
            "then freeze final training budget. "
            "Do not extend training beyond 3000."
        ),
}


summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print(
    "Summary:"
)

print(
    summary_path
)

print()
print(
    "========================================"
)

print(
    "PHASE 5C FINAL 3000 CURVE: PASS"
)

print(
    "========================================"
)

print(
    "DO NOT TRAIN BEYOND 3000."
)

print(
    "Next step is duration-selection validation, not more training."
)


EVALUATING BASELINE @ 2750 OPTIMIZER UPDATES
eval 0001/0200 scene=369b96d7adde8c0026a7 gt=1 decoded=99 op_pred=17 elapsed=0.1m
eval 0011/0200 scene=8aab26301d02d13c97c2 gt=3 decoded=100 op_pred=17 elapsed=0.2m
eval 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 decoded=99 op_pred=27 elapsed=0.3m
eval 0031/0200 scene=fe667dc0752460a37390 gt=3 decoded=100 op_pred=26 elapsed=0.4m
eval 0041/0200 scene=11e8bd15bed065be3400 gt=2 decoded=99 op_pred=22 elapsed=0.5m
eval 0051/0200 scene=87b7a8761b585200d3ff gt=1 decoded=100 op_pred=46 elapsed=0.6m
eval 0061/0200 scene=a3cb9b50dca442394cbe gt=2 decoded=100 op_pred=32 elapsed=0.7m
eval 0071/0200 scene=25a9d6b2837276a9dbb7 gt=1 decoded=100 op_pred=51 elapsed=0.8m
eval 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 decoded=100 op_pred=61 elapsed=0.9m
eval 0091/0200 scene=22aeb0f719d57f84b0f7 gt=0 decoded=99 op_pred=72 elapsed=1.0m
eval 0101/0200 scene=d3df4c3febde0d7548bc gt=0 decoded=99 op_pred=53 elapsed=1.1m
eval 0111/0200 scene=858baec99120971a3e47 gt=3

## final 1000-scene duration-selection audit

In [9]:
# ============================================================
# DASO-Net PHASE 5C
# FINAL TRAINING-DURATION SELECTION AUDIT
#
# NO TRAINING.
# NO EXTERNAL TEST DATA.
#
# Candidates:
#   2000
#   2250
#   2500
#   2750
#   3000 optimizer updates
#
# Stage A:
#   1000-scene deterministic synthetic-val metric evaluation
#   on ALL five candidates.
#
# Stage B:
#   1000-scene oracle diagnostic on TOP TWO by:
#       primary   = mask AP50:95
#       secondary = BestF1
#
# Stage C:
#   1000-scene counterfactual diagnostic on the winner.
#
# After this cell:
#   STOP.
#   We review and FREEZE the training budget.
# ============================================================

from __future__ import annotations

import hashlib
import json
import subprocess
import sys
import time
from pathlib import Path

import torch


# ============================================================
# CONSTANTS
# ============================================================

MAX_SCENES = 1000
SEED = 20260917

CANDIDATE_STEPS = [
    2000,
    2250,
    2500,
    2750,
    3000,
]


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "duration_selection_val1000_v1"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# CHECKPOINT MAP
# ============================================================

CHECKPOINTS = {
    2000:
        SNAPSHOT_DIR
        / "epoch_007_opt_002000.pt",

    2250:
        SNAPSHOT_DIR
        / "epoch_008_opt_002250.pt",

    2500:
        SNAPSHOT_DIR
        / "epoch_009_opt_002500.pt",

    2750:
        SNAPSHOT_DIR
        / "epoch_010_opt_002750.pt",

    3000:
        SNAPSHOT_DIR
        / "epoch_011_opt_003000.pt",
}


# ============================================================
# HELPERS
# ============================================================

def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


# ============================================================
# PRE-FLIGHT AUDIT
# ============================================================

print(
    "============================================================"
)

print(
    "PHASE 5C — FINAL DURATION-SELECTION AUDIT"
)

print(
    "============================================================"
)


for path, label in [
    (
        PROJECT_ROOT,
        "Frozen project dataset",
    ),
    (
        EVALUATOR,
        "Phase-5C evaluator",
    ),
    (
        ORACLE,
        "Oracle diagnostic",
    ),
    (
        COUNTERFACTUAL,
        "Counterfactual diagnostic",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


print()
print(
    "Candidate checkpoints:"
)


checkpoint_meta = {}


for step in CANDIDATE_STEPS:

    checkpoint = (
        CHECKPOINTS[
            step
        ]
    )

    require(
        checkpoint,
        f"{step}-update checkpoint",
    )

    payload = torch.load(
        checkpoint,
        map_location="cpu",
        weights_only=False,
    )

    actual_step = int(
        payload.get(
            "optimizer_step",
            -1,
        )
    )

    if actual_step != step:

        raise RuntimeError(
            f"Checkpoint {checkpoint.name} "
            f"contains optimizer_step={actual_step}, "
            f"expected {step}."
        )

    checkpoint_meta[
        step
    ] = {
        "path":
            str(
                checkpoint
            ),

        "sha256":
            sha256_file(
                checkpoint
            ),

        "epoch":
            int(
                payload.get(
                    "epoch",
                    -1,
                )
            ),
    }

    print(
        f"PASS  "
        f"{step:4d} updates  "
        f"epoch={checkpoint_meta[step]['epoch']}  "
        f"sha={checkpoint_meta[step]['sha256'][:18]}..."
    )


# ============================================================
# FREEZE AUDIT SETTINGS
# ============================================================

settings = {
    "phase":
        "5C",

    "purpose":
        "final_training_duration_selection",

    "split":
        "val",

    "max_scenes":
        MAX_SCENES,

    "seed":
        SEED,

    "candidate_optimizer_steps":
        CANDIDATE_STEPS,

    "selection_rule":
        {
            "primary":
                "mask AP50:95",

            "secondary":
                "BestF1",
        },

    "external_data_used":
        False,

    "training_performed":
        False,
}


settings_path = (
    OUTPUT_ROOT
    / "duration_selection_settings.json"
)


if settings_path.exists():

    existing = load_json(
        settings_path
    )

    if existing != settings:

        raise RuntimeError(
            "Existing duration-selection settings do not "
            "match this audit."
        )

else:

    settings_path.write_text(
        json.dumps(
            settings,
            indent=2,
        ),
        encoding="utf-8",
    )


print()
print(
    "Audit settings: PASS"
)

print(
    f"Validation scenes/checkpoint: {MAX_SCENES}"
)

print(
    f"Seed:                        {SEED}"
)

print(
    "External data:               NONE"
)

print(
    "Training:                    NONE"
)


# ============================================================
# STAGE A
#
# 1000-SCENE METRIC EVALUATION ON ALL FIVE CANDIDATES
# ============================================================

print()
print(
    "============================================================"
)

print(
    "STAGE A — 1000-SCENE METRICS"
)

print(
    "============================================================"
)


metric_summaries = {}

stage_a_start = time.time()


for step in CANDIDATE_STEPS:

    checkpoint = (
        CHECKPOINTS[
            step
        ]
    )

    metric_dir = (
        OUTPUT_ROOT
        / f"opt_{step:04d}"
        / "metrics"
    )

    metric_json = (
        metric_dir
        / "phase5c_metric_summary.json"
    )


    if metric_json.exists():

        print()
        print(
            f"[{step}] Existing complete metric JSON found."
        )

        print(
            "Reusing it."
        )

    else:

        print()
        print(
            "------------------------------------------------------------"
        )

        print(
            f"EVALUATING {step} UPDATES "
            f"ON {MAX_SCENES} VALIDATION SCENES"
        )

        print(
            "------------------------------------------------------------"
        )


        subprocess.run(
            [
                sys.executable,
                str(
                    EVALUATOR
                ),

                "--project-root",
                str(
                    PROJECT_ROOT
                ),

                "--checkpoint",
                str(
                    checkpoint
                ),

                "--output-dir",
                str(
                    metric_dir
                ),

                "--split",
                "val",

                "--max-scenes",
                str(
                    MAX_SCENES
                ),

                "--seed",
                str(
                    SEED
                ),

                "--num-workers",
                "2",

                "--decode-min-score",
                "0.001",

                "--operating-score",
                "0.05",

                "--topk",
                "100",

                "--nms-iou",
                "0.5",

                "--mask-threshold",
                "0.5",

                "--match-iou",
                "0.5",
            ],
            check=True,
        )


    require(
        metric_json,
        f"{step} metric summary",
    )


    metric_summaries[
        step
    ] = load_json(
        metric_json
    )


stage_a_elapsed = (
    time.time()
    - stage_a_start
)


# ============================================================
# EXTRACT STAGE-A RESULTS
# ============================================================

records = {}


for step in CANDIDATE_STEPS:

    metric = (
        metric_summaries[
            step
        ]
    )

    ap = (
        metric[
            "mask_ap"
        ]
    )

    sweep = (
        metric[
            "validation_threshold_sweep"
        ]
    )

    op = (
        metric[
            "operating_point"
        ]
    )


    records[
        step
    ] = {
        "AP50":
            float(
                ap[
                    "mask_ap50"
                ]
            ),

        "AP75":
            float(
                ap[
                    "mask_ap75"
                ]
            ),

        "AP50_95":
            float(
                ap[
                    "mask_ap50_95"
                ]
            ),

        "BestF1":
            float(
                sweep[
                    "best_f1"
                ]
            ),

        "BestThreshold":
            float(
                sweep[
                    "best_threshold_by_f1"
                ]
            ),

        "OperatingPrecision005":
            float(
                op[
                    "precision"
                ]
            ),

        "OperatingRecall005":
            float(
                op[
                    "recall"
                ]
            ),

        "CountMAE":
            float(
                op[
                    "count_mae"
                ]
            ),

        "EmptyFP005":
            float(
                op[
                    "empty_scene_fp_per_image"
                ]
            ),
    }


# ============================================================
# PRINT LARGE-VAL METRIC TABLE
# ============================================================

print()
print(
    "=============================================================================================="
)

print(
    "1000-SCENE SYNTHETIC-VALIDATION DURATION COMPARISON"
)

print(
    "=============================================================================================="
)

print(
    f"{'Updates':>8}"
    f"{'AP50':>10}"
    f"{'AP75':>10}"
    f"{'AP50:95':>11}"
    f"{'BestF1':>10}"
    f"{'BestThr':>10}"
    f"{'Recall@.05':>12}"
    f"{'CountMAE':>11}"
    f"{'FP@.05':>10}"
)

print(
    "-" * 92
)


for step in CANDIDATE_STEPS:

    r = (
        records[
            step
        ]
    )

    print(
        f"{step:>8d}"
        f"{r['AP50']:>10.4f}"
        f"{r['AP75']:>10.4f}"
        f"{r['AP50_95']:>11.4f}"
        f"{r['BestF1']:>10.4f}"
        f"{r['BestThreshold']:>10.2f}"
        f"{r['OperatingRecall005']:>12.4f}"
        f"{r['CountMAE']:>11.2f}"
        f"{r['EmptyFP005']:>10.2f}"
    )


# ============================================================
# RANK BY PREDEFINED RULE
# ============================================================

ranking = sorted(
    CANDIDATE_STEPS,
    key=lambda step: (
        records[
            step
        ][
            "AP50_95"
        ],
        records[
            step
        ][
            "BestF1"
        ],
    ),
    reverse=True,
)


winner = (
    ranking[
        0
    ]
)

runner_up = (
    ranking[
        1
    ]
)


print()
print(
    "============================================================"
)

print(
    "1000-SCENE METRIC RANKING"
)

print(
    "============================================================"
)


for rank, step in enumerate(
    ranking,
    start=1,
):

    print(
        f"{rank}. "
        f"{step} updates | "
        f"AP50:95={records[step]['AP50_95']:.4f} | "
        f"BestF1={records[step]['BestF1']:.4f}"
    )


print()
print(
    "Metric winner:",
    winner,
)

print(
    "Runner-up:",
    runner_up,
)


# ============================================================
# STAGE B
#
# 1000-SCENE ORACLE DIAGNOSTIC ON TOP TWO ONLY
# ============================================================

print()
print(
    "============================================================"
)

print(
    "STAGE B — 1000-SCENE ORACLE ON TOP TWO"
)

print(
    "============================================================"
)


oracle_records = {}

stage_b_start = time.time()


for step in [
    winner,
    runner_up,
]:

    checkpoint = (
        CHECKPOINTS[
            step
        ]
    )

    oracle_dir = (
        OUTPUT_ROOT
        / f"opt_{step:04d}"
        / "oracle"
    )

    oracle_json = (
        oracle_dir
        / "oracle_component_diagnostic.json"
    )


    if oracle_json.exists():

        print()
        print(
            f"[{step}] Existing oracle JSON found."
        )

        print(
            "Reusing it."
        )

    else:

        print()
        print(
            "------------------------------------------------------------"
        )

        print(
            f"ORACLE DIAGNOSTIC: {step} UPDATES"
        )

        print(
            "------------------------------------------------------------"
        )


        subprocess.run(
            [
                sys.executable,
                str(
                    ORACLE
                ),

                "--project-root",
                str(
                    PROJECT_ROOT
                ),

                "--checkpoint",
                str(
                    checkpoint
                ),

                "--evaluator",
                str(
                    EVALUATOR
                ),

                "--output-dir",
                str(
                    oracle_dir
                ),

                "--split",
                "val",

                "--max-scenes",
                str(
                    MAX_SCENES
                ),

                "--seed",
                str(
                    SEED
                ),

                "--num-workers",
                "2",

                "--stride",
                "4",

                "--peak-kernel",
                "3",

                "--topk",
                "100",

                "--mask-threshold",
                "0.5",
            ],
            check=True,
        )


    require(
        oracle_json,
        f"{step} oracle JSON",
    )


    oracle = load_json(
        oracle_json
    )


    center = (
        oracle[
            "center_head"
        ]
    )

    geometry = (
        oracle[
            "oracle_gt_cell_geometry"
        ]
    )

    masks = (
        oracle[
            "oracle_gt_cell_mask"
        ]
    )

    empty = (
        oracle[
            "empty_scene_center_behavior"
        ]
    )


    oracle_records[
        step
    ] = {
        "GTCenterScore":
            float(
                center[
                    "gt_cell_score_stats"
                ][
                    "mean"
                ]
            ),

        "PeakRecall4":
            float(
                center[
                    "top100_peak_recall_within_4px"
                ]
            ),

        "PeakRecall8":
            float(
                center[
                    "top100_peak_recall_within_8px"
                ]
            ),

        "PeakRecall16":
            float(
                center[
                    "top100_peak_recall_within_16px"
                ]
            ),

        "BoxIoU":
            float(
                geometry[
                    "box_iou_stats"
                ][
                    "mean"
                ]
            ),

        "WidthRatio":
            float(
                geometry[
                    "pred_to_gt_width_ratio_stats"
                ][
                    "median"
                ]
            ),

        "HeightRatio":
            float(
                geometry[
                    "pred_to_gt_height_ratio_stats"
                ][
                    "median"
                ]
            ),

        "GTBoxMaskIoU":
            float(
                masks[
                    "gt_box_crop_mask_iou_stats"
                ][
                    "mean"
                ]
            ),

        "GTBoxMaskIoU50":
            float(
                masks[
                    "gt_box_crop_mask_iou_ge_0_50"
                ]
            ),

        "PredBoxMaskIoU":
            float(
                masks[
                    "pred_box_crop_mask_iou_stats"
                ][
                    "mean"
                ]
            ),

        "EmptyMaxPeak":
            float(
                empty[
                    "max_peak_score_stats"
                ][
                    "mean"
                ]
            ),
    }


stage_b_elapsed = (
    time.time()
    - stage_b_start
)


print()
print(
    "=============================================================================================="
)

print(
    "TOP-TWO 1000-SCENE ORACLE COMPARISON"
)

print(
    "=============================================================================================="
)

print(
    f"{'Updates':>8}"
    f"{'GTScore':>10}"
    f"{'PeakR@4':>10}"
    f"{'PeakR@8':>10}"
    f"{'PeakR@16':>11}"
    f"{'BoxIoU':>10}"
    f"{'MaskIoU':>10}"
    f"{'Mask>=.5':>11}"
    f"{'EmptyMax':>10}"
)

print(
    "-" * 90
)


for step in [
    winner,
    runner_up,
]:

    r = (
        oracle_records[
            step
        ]
    )

    print(
        f"{step:>8d}"
        f"{r['GTCenterScore']:>10.4f}"
        f"{r['PeakRecall4']:>10.4f}"
        f"{r['PeakRecall8']:>10.4f}"
        f"{r['PeakRecall16']:>11.4f}"
        f"{r['BoxIoU']:>10.4f}"
        f"{r['GTBoxMaskIoU']:>10.4f}"
        f"{r['GTBoxMaskIoU50']:>11.4f}"
        f"{r['EmptyMaxPeak']:>10.4f}"
    )


# ============================================================
# STAGE C
#
# COUNTERFACTUAL SANITY CHECK ON METRIC WINNER
# ============================================================

print()
print(
    "============================================================"
)

print(
    "STAGE C — 1000-SCENE COUNTERFACTUAL SANITY CHECK"
)

print(
    "============================================================"
)


counterfactual_dir = (
    OUTPUT_ROOT
    / f"opt_{winner:04d}"
    / "counterfactual"
)

counterfactual_json = (
    counterfactual_dir
    / "counterfactual_center_diagnostic.json"
)


stage_c_start = time.time()


if counterfactual_json.exists():

    print(
        "Existing counterfactual JSON found."
    )

    print(
        "Reusing it."
    )

else:

    subprocess.run(
        [
            sys.executable,
            str(
                COUNTERFACTUAL
            ),

            "--project-root",
            str(
                PROJECT_ROOT
            ),

            "--checkpoint",
            str(
                CHECKPOINTS[
                    winner
                ]
            ),

            "--evaluator",
            str(
                EVALUATOR
            ),

            "--output-dir",
            str(
                counterfactual_dir
            ),

            "--split",
            "val",

            "--max-scenes",
            str(
                MAX_SCENES
            ),

            "--seed",
            str(
                SEED
            ),

            "--num-workers",
            "2",

            "--stride",
            "4",

            "--peak-kernel",
            "3",

            "--window-radius-feature",
            "2",
        ],
        check=True,
    )


require(
    counterfactual_json,
    "Winner counterfactual JSON",
)


cf = load_json(
    counterfactual_json
)


stage_c_elapsed = (
    time.time()
    - stage_c_start
)


cf_summary = {
    "ExactLift":
        float(
            cf[
                "exact_gt_cell"
            ][
                "composite_minus_clean_stats"
            ][
                "mean"
            ]
        ),

    "ExactCompositeGtClean":
        float(
            cf[
                "exact_gt_cell"
            ][
                "fraction_composite_gt_clean"
            ]
        ),

    "LocalLift":
        float(
            cf[
                "local_window_peak"
            ][
                "composite_minus_clean_stats"
            ][
                "mean"
            ]
        ),

    "LocalCompositeGtClean":
        float(
            cf[
                "local_window_peak"
            ][
                "fraction_composite_gt_clean"
            ]
        ),

    "CleanMax":
        float(
            cf[
                "scene_level"
            ][
                "clean_max_peak_stats"
            ][
                "mean"
            ]
        ),

    "OutsideMax":
        float(
            cf[
                "scene_level"
            ][
                "composite_outside_effect_max_peak_stats"
            ][
                "mean"
            ]
        ),

    "EffectMax":
        float(
            cf[
                "scene_level"
            ][
                "composite_effect_max_peak_stats"
            ][
                "mean"
            ]
        ),
}


print()
print(
    "Winner:",
    winner,
    "optimizer updates"
)

print(
    "Exact GT-cell lift:",
    f"{cf_summary['ExactLift']:+.4f}",
)

print(
    "Composite > clean:",
    f"{cf_summary['ExactCompositeGtClean']:.4f}",
)

print(
    "Local peak lift:",
    f"{cf_summary['LocalLift']:+.4f}",
)

print(
    "Local composite > clean:",
    f"{cf_summary['LocalCompositeGtClean']:.4f}",
)

print(
    "Clean max peak:",
    f"{cf_summary['CleanMax']:.4f}",
)

print(
    "Outside-effect max:",
    f"{cf_summary['OutsideMax']:.4f}",
)

print(
    "Effect-region max:",
    f"{cf_summary['EffectMax']:.4f}",
)


# ============================================================
# DELTA BETWEEN WINNER AND RUNNER-UP
# ============================================================

print()
print(
    "============================================================"
)

print(
    "WINNER vs RUNNER-UP"
)

print(
    "============================================================"
)


for key in [
    "AP50",
    "AP75",
    "AP50_95",
    "BestF1",
    "EmptyFP005",
]:

    difference = (
        records[
            winner
        ][
            key
        ]
        -
        records[
            runner_up
        ][
            key
        ]
    )

    print(
        f"{key:12s}: "
        f"{winner} - {runner_up} = "
        f"{difference:+.4f}"
    )


# ============================================================
# SAVE FINAL DURATION-SELECTION EVIDENCE
# ============================================================

final_summary = {
    "phase":
        "5C",

    "status":
        "duration_selection_evidence_complete",

    "paper_result":
        False,

    "training_performed":
        False,

    "external_data_used":
        False,

    "validation_split":
        "synthetic_val",

    "max_scenes":
        MAX_SCENES,

    "seed":
        SEED,

    "candidate_optimizer_steps":
        CANDIDATE_STEPS,

    "selection_rule":
        {
            "primary":
                "mask AP50:95",

            "secondary":
                "BestF1",
        },

    "checkpoint_metadata":
        {
            str(step):
                checkpoint_meta[
                    step
                ]

            for step in CANDIDATE_STEPS
        },

    "metric_records":
        {
            str(step):
                records[
                    step
                ]

            for step in CANDIDATE_STEPS
        },

    "ranking":
        ranking,

    "metric_winner":
        int(
            winner
        ),

    "runner_up":
        int(
            runner_up
        ),

    "oracle_top_two":
        {
            str(step):
                oracle_records[
                    step
                ]

            for step in [
                winner,
                runner_up,
            ]
        },

    "winner_counterfactual":
        cf_summary,

    "next_action":
        (
            "Review this evidence and freeze final training "
            "budget. Do not train beyond 3000."
        ),
}


final_summary_path = (
    OUTPUT_ROOT
    / "duration_selection_val1000_summary.json"
)


final_summary_path.write_text(
    json.dumps(
        final_summary,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL DURATION-SELECTION AUDIT COMPLETE"
)

print(
    "============================================================"
)

print(
    "Metric winner:     ",
    winner,
)

print(
    "Runner-up:         ",
    runner_up,
)

print(
    "Winner checkpoint:"
)

print(
    CHECKPOINTS[
        winner
    ]
)

print()

print(
    "Stage-A metric time:         "
    f"{stage_a_elapsed / 60.0:.2f} min"
)

print(
    "Stage-B oracle time:         "
    f"{stage_b_elapsed / 60.0:.2f} min"
)

print(
    "Stage-C counterfactual time: "
    f"{stage_c_elapsed / 60.0:.2f} min"
)

print()

print(
    "Summary:"
)

print(
    final_summary_path
)

print()
print(
    "========================================"
)

print(
    "PHASE 5C DURATION SELECTION: PASS"
)

print(
    "========================================"
)

print(
    "NO MORE TRAINING."
)

print(
    "Paste the metric table, top-two oracle table,"
)

print(
    "winner/runner-up, and counterfactual summary."
)

print(
    "Then we freeze the final training budget."
)

PHASE 5C — FINAL DURATION-SELECTION AUDIT
PASS  Frozen project dataset
PASS  Phase-5C evaluator
PASS  Oracle diagnostic
PASS  Counterfactual diagnostic

Candidate checkpoints:
PASS  2000 updates  epoch=7  sha=a215ef3608a930dde3...
PASS  2250 updates  epoch=8  sha=2c4ca43989b1bb0e96...
PASS  2500 updates  epoch=9  sha=a5d555e94f5f69b654...
PASS  2750 updates  epoch=10  sha=04b3e63ad398cfc542...
PASS  3000 updates  epoch=11  sha=31483f44613bd6b4e3...

Audit settings: PASS
Validation scenes/checkpoint: 1000
Seed:                        20260917
External data:               NONE
Training:                    NONE

STAGE A — 1000-SCENE METRICS

------------------------------------------------------------
EVALUATING 2000 UPDATES ON 1000 VALIDATION SCENES
------------------------------------------------------------
eval 0001/1000 scene=53fcddcd471d01372c54 gt=2 decoded=98 op_pred=39 elapsed=0.1m
eval 0011/1000 scene=1321d3c185e33b40ef7d gt=6 decoded=100 op_pred=23 elapsed=0.2m
eval 0021/1000 s

In [10]:
# ============================================================
# DASO-Net — PHASE 5C FINAL TRAINING-RECIPE FREEZE v1
#
# PURPOSE:
#   Permanently freeze the training duration and all currently
#   validated training infrastructure BEFORE final multi-seed
#   training begins.
#
# FINAL TRAINING BUDGET:
#   2750 optimizer updates
#
# IMPORTANT:
#   - This performs NO training.
#   - This performs NO external evaluation.
#   - The 2750 development checkpoint is schedule-selection
#     evidence only; it is NOT a final-paper model.
# ============================================================

from __future__ import annotations

import hashlib
import json
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_baseline_curve1000updates_v1.json"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

SELECTED_DEV_CHECKPOINT = (
    RUN_ROOT
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

CHECKPOINT_3000 = (
    RUN_ROOT
    / "snapshots"
    / "epoch_011_opt_003000.pt"
)

DURATION_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "duration_selection_val1000_v1"
)

DURATION_SUMMARY = (
    DURATION_ROOT
    / "duration_selection_val1000_summary.json"
)

CURVE_3000_SUMMARY = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "baseline_curve1000updates_v1/"
    "baseline_learning_curve_3000_summary.json"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

INSTANCE_LOSS = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "losses"
    / "instance.py"
)

BASE_TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)

CHECKPOINT_LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE_SCRIPT = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL_SCRIPT = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)


FREEZE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

FREEZE_JSON = (
    FREEZE_ROOT
    / "FINAL_TRAINING_RECIPE_FREEZE_v1.json"
)

FREEZE_README = (
    FREEZE_ROOT
    / "README_FREEZE.txt"
)

FREEZE_MANIFEST = (
    FREEZE_ROOT
    / "FREEZE_FILE_MANIFEST.json"
)

FREEZE_ZIP = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1.zip"
)

FREEZE_ZIP_SHA = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1.sha256.txt"
)


# ============================================================
# CONSTANTS
# ============================================================

FINAL_OPTIMIZER_UPDATES = 2750

EXPECTED_SELECTED_EPOCH = 10

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)


# ============================================================
# HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def copy_file(
    src: Path,
    dst: Path,
) -> None:

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        src,
        dst,
    )


# ============================================================
# REQUIRED INPUT AUDIT
# ============================================================

required = [
    (
        PROJECT_ROOT,
        "Frozen project dataset",
    ),
    (
        CONFIG,
        "Development training config",
    ),
    (
        SELECTED_DEV_CHECKPOINT,
        "2750 development checkpoint",
    ),
    (
        CHECKPOINT_3000,
        "3000 boundary checkpoint",
    ),
    (
        DURATION_SUMMARY,
        "1000-scene duration-selection summary",
    ),
    (
        CURVE_3000_SUMMARY,
        "250-to-3000 learning-curve summary",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
    (
        INSTANCE_LOSS,
        "Instance loss",
    ),
    (
        BASE_TRAINER,
        "Base trainer",
    ),
    (
        CHECKPOINT_LAUNCHER,
        "Checkpoint launcher",
    ),
    (
        EVALUATOR,
        "Evaluator",
    ),
    (
        ORACLE_SCRIPT,
        "Oracle diagnostic",
    ),
    (
        COUNTERFACTUAL_SCRIPT,
        "Counterfactual diagnostic",
    ),
]


print(
    "============================================================"
)

print(
    "DASO-Net — FINAL TRAINING-RECIPE FREEZE v1"
)

print(
    "============================================================"
)


for path, label in required:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# VERIFY SELECTED DEVELOPMENT CHECKPOINT
# ============================================================

selected_payload = torch.load(
    SELECTED_DEV_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)


selected_step = int(
    selected_payload.get(
        "optimizer_step",
        -1,
    )
)

selected_epoch = int(
    selected_payload.get(
        "epoch",
        -1,
    )
)


if selected_step != FINAL_OPTIMIZER_UPDATES:

    raise RuntimeError(
        "Selected checkpoint does not contain "
        f"optimizer_step={FINAL_OPTIMIZER_UPDATES}."
    )


if selected_epoch != EXPECTED_SELECTED_EPOCH:

    raise RuntimeError(
        "Selected checkpoint epoch mismatch.\n"
        f"Expected: {EXPECTED_SELECTED_EPOCH}\n"
        f"Actual:   {selected_epoch}"
    )


selected_sha = sha256_file(
    SELECTED_DEV_CHECKPOINT
)


print()
print(
    "Selected development checkpoint: PASS"
)

print(
    "  optimizer step:",
    selected_step,
)

print(
    "  epoch:",
    selected_epoch,
)

print(
    "  SHA256:"
)

print(
    " ",
    selected_sha,
)


# ============================================================
# VERIFY 3000 BOUNDARY CHECKPOINT
# ============================================================

boundary_payload = torch.load(
    CHECKPOINT_3000,
    map_location="cpu",
    weights_only=False,
)


if int(
    boundary_payload.get(
        "optimizer_step",
        -1,
    )
) != 3000:

    raise RuntimeError(
        "3000 boundary checkpoint identity failure."
    )


boundary_sha = sha256_file(
    CHECKPOINT_3000
)


print()
print(
    "3000 schedule-search boundary checkpoint: PASS"
)


# ============================================================
# VERIFY DECODER
# ============================================================

decoder_sha = sha256_file(
    DECODER
)


if decoder_sha != EXPECTED_DECODER_SHA:

    raise RuntimeError(
        "Validated decoder SHA mismatch.\n"
        f"Expected: {EXPECTED_DECODER_SHA}\n"
        f"Actual:   {decoder_sha}"
    )


print(
    "Validated decoder SHA: PASS"
)


# ============================================================
# VERIFY DURATION-SELECTION EVIDENCE
# ============================================================

duration = json.loads(
    DURATION_SUMMARY.read_text(
        encoding="utf-8"
    )
)


if int(
    duration[
        "metric_winner"
    ]
) != 2750:

    raise RuntimeError(
        "Duration-selection winner is not 2750."
    )


if int(
    duration[
        "runner_up"
    ]
) != 2500:

    raise RuntimeError(
        "Expected duration-selection runner-up = 2500."
    )


if bool(
    duration[
        "training_performed"
    ]
):

    raise RuntimeError(
        "Duration audit unexpectedly reports training."
    )


if bool(
    duration[
        "external_data_used"
    ]
):

    raise RuntimeError(
        "Duration audit unexpectedly reports external data usage."
    )


if int(
    duration[
        "max_scenes"
    ]
) != 1000:

    raise RuntimeError(
        "Expected 1000-scene duration-selection audit."
    )


records = (
    duration[
        "metric_records"
    ]
)


winner = (
    records[
        "2750"
    ]
)

runner = (
    records[
        "2500"
    ]
)

boundary = (
    records[
        "3000"
    ]
)


print()
print(
    "1000-scene duration-selection evidence: PASS"
)

print(
    "  winner:    2750"
)

print(
    "  runner-up: 2500"
)

print(
    "  boundary:  3000"
)


# ============================================================
# VERIFY PCC-CENTER REMAINS DISABLED IN CONFIG
# ============================================================

config_obj = json.loads(
    CONFIG.read_text(
        encoding="utf-8"
    )
)


pcc_center = (
    config_obj
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if (
    pcc_center is not None
    and bool(
        pcc_center.get(
            "enabled",
            False,
        )
    )
):

    raise RuntimeError(
        "PCC-center is unexpectedly ENABLED."
    )


print(
    "PCC-center: REJECTED / DISABLED — PASS"
)


# ============================================================
# CORE INFRASTRUCTURE HASHES
# ============================================================

core_files = {
    "development_config":
        CONFIG,

    "decoder":
        DECODER,

    "instance_loss":
        INSTANCE_LOSS,

    "base_trainer":
        BASE_TRAINER,

    "checkpoint_launcher":
        CHECKPOINT_LAUNCHER,

    "evaluator":
        EVALUATOR,

    "oracle_script":
        ORACLE_SCRIPT,

    "counterfactual_script":
        COUNTERFACTUAL_SCRIPT,

    "duration_selection_summary":
        DURATION_SUMMARY,

    "learning_curve_3000_summary":
        CURVE_3000_SUMMARY,

    "selected_dev_checkpoint":
        SELECTED_DEV_CHECKPOINT,

    "schedule_boundary_checkpoint":
        CHECKPOINT_3000,
}


core_hashes = {
    name: {
        "path":
            str(
                path
            ),

        "sha256":
            sha256_file(
                path
            ),

        "bytes":
            int(
                path.stat().st_size
            ),
    }

    for name, path
    in core_files.items()
}


# ============================================================
# CREATE FORMAL FREEZE RECORD
# ============================================================

freeze_record = {
    "freeze_name":
        "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1",

    "freeze_time_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "phase":
        "Phase 5C",

    "status":
        "FINAL_TRAINING_RECIPE_FROZEN",

    "development_only":
        True,

    "paper_test_result":
        False,

    "training_duration": {
        "unit":
            "optimizer_updates",

        "final_budget":
            2750,

        "optimizer_updates_per_training_block":
            250,

        "training_blocks":
            11,

        "development_selected_epoch_zero_based":
            10,

        "schedule_search_upper_bound":
            3000,

        "training_beyond_3000_for_duration_search":
            "PROHIBITED",
    },

    "duration_selection": {
        "validation_source":
            "FruitMP-Syn synthetic validation only",

        "validation_scenes":
            1000,

        "seed":
            20260917,

        "candidate_optimizer_steps":
            [
                2000,
                2250,
                2500,
                2750,
                3000,
            ],

        "primary_criterion":
            "mask AP50:95",

        "secondary_criterion":
            "BestF1",

        "winner":
            2750,

        "runner_up":
            2500,

        "boundary_checkpoint":
            3000,

        "winner_metrics": {
            "mask_ap50":
                winner[
                    "AP50"
                ],

            "mask_ap75":
                winner[
                    "AP75"
                ],

            "mask_ap50_95":
                winner[
                    "AP50_95"
                ],

            "best_f1":
                winner[
                    "BestF1"
                ],

            "best_threshold":
                winner[
                    "BestThreshold"
                ],

            "count_mae_at_0_05":
                winner[
                    "CountMAE"
                ],

            "empty_fp_per_image_at_0_05":
                winner[
                    "EmptyFP005"
                ],
        },

        "runner_up_metrics": {
            "mask_ap50":
                runner[
                    "AP50"
                ],

            "mask_ap75":
                runner[
                    "AP75"
                ],

            "mask_ap50_95":
                runner[
                    "AP50_95"
                ],

            "best_f1":
                runner[
                    "BestF1"
                ],
        },

        "boundary_3000_metrics": {
            "mask_ap50":
                boundary[
                    "AP50"
                ],

            "mask_ap75":
                boundary[
                    "AP75"
                ],

            "mask_ap50_95":
                boundary[
                    "AP50_95"
                ],

            "best_f1":
                boundary[
                    "BestF1"
                ],
        },

        "external_data_used":
            False,
    },

    "winner_oracle_1000": (
        duration[
            "oracle_top_two"
        ][
            "2750"
        ]
    ),

    "winner_counterfactual_1000": (
        duration[
            "winner_counterfactual"
        ]
    ),

    "final_training_recipe": {
        "architecture":
            "DASO-Net Phase5A frozen architecture",

        "parameter_count":
            1160679,

        "native_tile":
            1024,

        "maximum_downsample":
            16,

        "synthetic_batch_per_gpu":
            1,

        "anchor_pair_batch_per_gpu":
            2,

        "world_size":
            2,

        "gradient_accumulation":
            4,

        "effective_synthetic_batch":
            8,

        "synthetic_workers_per_gpu":
            2,

        "anchor_workers_per_gpu":
            1,

        "amp":
            "FP16",

        "amp_initial_scale":
            32,

        "learning_rate":
            3e-4,

        "warmup_optimizer_updates":
            50,

        "post_warmup_schedule":
            "constant",

        "post_warmup_min_lr_ratio":
            1.0,

        "final_optimizer_updates":
            2750,

        "pcc":
            "original feature/embedding PCC retained",

        "pcc_center":
            "rejected and disabled",

        "crpa":
            "retained",

        "decoder":
            "corrected local-peak decoder",

        "decoder_sha256":
            EXPECTED_DECODER_SHA,
    },

    "final_multiseed_policy": {
        "initialization":
            "fresh initialization for every final seed",

        "resume_from_development_checkpoint":
            False,

        "fixed_budget":
            2750,

        "early_stopping":
            False,

        "terminal_checkpoint_policy":
            (
                "Use the checkpoint at exactly 2750 "
                "optimizer updates for each final seed."
            ),

        "external_test_data_for_selection":
            False,

        "rsc_for_selection":
            False,

        "tioman_for_selection":
            False,

        "talanta_for_selection":
            False,
    },

    "scientific_claim_limits": {
        "task":
            "instance segmentation / selective RGB screening",

        "target":
            (
                "visually resolvable suspected "
                "microplastic-like particles / suspected "
                "surface plastic contamination"
            ),

        "chemical_identification":
            False,

        "polymer_classification":
            False,

        "special_hardware_required":
            False,

        "intended_sensor":
            "ordinary smartphone RGB",
    },

    "development_selected_checkpoint": {
        "path":
            str(
                SELECTED_DEV_CHECKPOINT
            ),

        "optimizer_step":
            2750,

        "epoch":
            10,

        "sha256":
            selected_sha,

        "role":
            (
                "development schedule-selection evidence only; "
                "not final-paper model"
            ),
    },

    "schedule_boundary_checkpoint": {
        "path":
            str(
                CHECKPOINT_3000
            ),

        "optimizer_step":
            3000,

        "epoch":
            11,

        "sha256":
            boundary_sha,
    },

    "core_file_hashes":
        core_hashes,

    "next_stage":
        (
            "Fresh independent final multi-seed training "
            "using the frozen 2750-update recipe."
        ),
}


# ============================================================
# CREATE FREEZE DIRECTORY
# ============================================================

if FREEZE_ROOT.exists():

    shutil.rmtree(
        FREEZE_ROOT
    )


FREEZE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


FREEZE_JSON.write_text(
    json.dumps(
        freeze_record,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# COPY EXACT FREEZE EVIDENCE
# ============================================================

copy_map = {
    CONFIG:
        FREEZE_ROOT
        / "evidence"
        / "phase5c_baseline_curve1000updates_v1.json",

    DURATION_SUMMARY:
        FREEZE_ROOT
        / "evidence"
        / "duration_selection_val1000_summary.json",

    CURVE_3000_SUMMARY:
        FREEZE_ROOT
        / "evidence"
        / "baseline_learning_curve_3000_summary.json",

    DECODER:
        FREEZE_ROOT
        / "code"
        / "decoding.py",

    INSTANCE_LOSS:
        FREEZE_ROOT
        / "code"
        / "instance.py",

    BASE_TRAINER:
        FREEZE_ROOT
        / "code"
        / "train_phase5b.py",

    CHECKPOINT_LAUNCHER:
        FREEZE_ROOT
        / "code"
        / "launch_phase5c_checkpointed_baseline.py",

    EVALUATOR:
        FREEZE_ROOT
        / "code"
        / "evaluate_phase5c.py",

    ORACLE_SCRIPT:
        FREEZE_ROOT
        / "code"
        / "diagnose_phase5c_oracle.py",

    COUNTERFACTUAL_SCRIPT:
        FREEZE_ROOT
        / "code"
        / "diagnose_phase5c_counterfactual_center.py",
}


for src, dst in copy_map.items():

    copy_file(
        src,
        dst,
    )


# Keep selected development checkpoint in freeze package because
# it proves the schedule-selection state.
copy_file(
    SELECTED_DEV_CHECKPOINT,
    FREEZE_ROOT
    / "development_checkpoint"
    / SELECTED_DEV_CHECKPOINT.name,
)


# ============================================================
# README
# ============================================================

FREEZE_README.write_text(
    "\n".join(
        [
            "DASO-Net Phase 5C Final Training Recipe FREEZE v1",
            "",
            "FINAL TRAINING BUDGET:",
            "2750 optimizer updates",
            "",
            "WHY 2750:",
            (
                "Selected by predefined synthetic-validation "
                "criterion on 1000 scenes:"
            ),
            "primary = mask AP50:95",
            "secondary = BestF1",
            "",
            (
                f"2750 AP50:95 = "
                f"{winner['AP50_95']:.4f}"
            ),
            (
                f"2750 BestF1 = "
                f"{winner['BestF1']:.4f}"
            ),
            (
                f"2500 AP50:95 = "
                f"{runner['AP50_95']:.4f}"
            ),
            (
                f"2500 BestF1 = "
                f"{runner['BestF1']:.4f}"
            ),
            (
                f"3000 AP50:95 = "
                f"{boundary['AP50_95']:.4f}"
            ),
            "",
            "IMPORTANT:",
            (
                "The enclosed 2750 checkpoint is DEVELOPMENT "
                "EVIDENCE ONLY."
            ),
            (
                "Final paper models must be freshly initialized "
                "independent seeds trained for exactly 2750 "
                "optimizer updates."
            ),
            "",
            "PCC-center remains rejected and disabled.",
            "Original feature/embedding PCC remains enabled.",
            "CRPA remains enabled.",
            "No external test data were used to select duration.",
            "",
        ]
    ),
    encoding="utf-8",
)


# ============================================================
# MANIFEST FREEZE DIRECTORY
# ============================================================

manifest_entries = []


for path in sorted(
    p
    for p in FREEZE_ROOT.rglob("*")
    if p.is_file()
    and p != FREEZE_MANIFEST
):

    manifest_entries.append(
        {
            "path":
                str(
                    path.relative_to(
                        FREEZE_ROOT
                    )
                ),

            "bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "freeze_name":
        "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1",

    "file_count":
        len(
            manifest_entries
        ),

    "files":
        manifest_entries,
}


FREEZE_MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# BUILD ZIP
# ============================================================

if FREEZE_ZIP.exists():

    FREEZE_ZIP.unlink()


with zipfile.ZipFile(
    FREEZE_ZIP,
    "w",
) as zf:

    for path in sorted(
        p
        for p in FREEZE_ROOT.rglob("*")
        if p.is_file()
    ):

        relative = path.relative_to(
            FREEZE_ROOT.parent
        )

        # Checkpoints are already compressed-like binary data.
        compression = (
            zipfile.ZIP_STORED
            if path.suffix.lower()
            in {
                ".pt",
                ".pth",
                ".zip",
            }
            else zipfile.ZIP_DEFLATED
        )

        zf.write(
            path,
            arcname=str(
                relative
            ),
            compress_type=compression,
        )


# ============================================================
# CRC VERIFY
# ============================================================

with zipfile.ZipFile(
    FREEZE_ZIP,
    "r",
) as zf:

    bad = zf.testzip()

    if bad is not None:

        raise RuntimeError(
            f"Freeze ZIP CRC failure: {bad}"
        )


freeze_zip_sha = sha256_file(
    FREEZE_ZIP
)


FREEZE_ZIP_SHA.write_text(
    f"{freeze_zip_sha}  {FREEZE_ZIP.name}\n",
    encoding="utf-8",
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL DASO-Net TRAINING RECIPE"
)

print(
    "============================================================"
)

print(
    "Training budget:         2750 optimizer updates"
)

print(
    "Training blocks:         11"
)

print(
    "Updates/block:           250"
)

print(
    "Final-seed init:         FRESH"
)

print(
    "Early stopping:          NO"
)

print(
    "Final-seed checkpoint:   terminal @ 2750"
)

print(
    "Original PCC:            RETAINED"
)

print(
    "CRPA:                    RETAINED"
)

print(
    "PCC-center:              REJECTED / DISABLED"
)

print(
    "External tuning:         NONE"
)

print()

print(
    "Development winner:"
)

print(
    "  AP50:95:",
    f"{winner['AP50_95']:.4f}",
)

print(
    "  BestF1:",
    f"{winner['BestF1']:.4f}",
)

print()

print(
    "Runner-up 2500:"
)

print(
    "  AP50:95:",
    f"{runner['AP50_95']:.4f}",
)

print(
    "  BestF1:",
    f"{runner['BestF1']:.4f}",
)

print()

print(
    "3000 boundary:"
)

print(
    "  AP50:95:",
    f"{boundary['AP50_95']:.4f}",
)

print(
    "  BestF1:",
    f"{boundary['BestF1']:.4f}",
)

print()

print(
    "Freeze record:"
)

print(
    FREEZE_JSON
)

print()

print(
    "Freeze ZIP:"
)

print(
    FREEZE_ZIP
)

print(
    "SHA256:"
)

print(
    freeze_zip_sha
)

print()

print(
    "============================================================"
)

print(
    "PHASE 5C FINAL TRAINING RECIPE FREEZE: PASS"
)

print(
    "============================================================"
)

print(
    "FINAL BUDGET = 2750 OPTIMIZER UPDATES"
)

print(
    "DO NOT CHANGE TRAINING DURATION."
)

print(
    "DO NOT RESUME THE DEVELOPMENT CHECKPOINT FOR FINAL SEEDS."
)

print(
    "Next stage: fresh independent multi-seed final training."
)

DASO-Net — FINAL TRAINING-RECIPE FREEZE v1
PASS  Frozen project dataset
PASS  Development training config
PASS  2750 development checkpoint
PASS  3000 boundary checkpoint
PASS  1000-scene duration-selection summary
PASS  250-to-3000 learning-curve summary
PASS  Validated decoder
PASS  Instance loss
PASS  Base trainer
PASS  Checkpoint launcher
PASS  Evaluator
PASS  Oracle diagnostic
PASS  Counterfactual diagnostic

Selected development checkpoint: PASS
  optimizer step: 2750
  epoch: 10
  SHA256:
  04b3e63ad398cfc542b6d0cbc4ea9929e57ee4c0f601fc7e833b7dcefd2fa101

3000 schedule-search boundary checkpoint: PASS
Validated decoder SHA: PASS

1000-scene duration-selection evidence: PASS
  winner:    2750
  runner-up: 2500
  boundary:  3000
PCC-center: REJECTED / DISABLED — PASS

FINAL DASO-Net TRAINING RECIPE
Training budget:         2750 optimizer updates
Training blocks:         11
Updates/block:           250
Final-seed init:         FRESH
Early stopping:          NO
Final-seed checkpoint

In [11]:
# ============================================================
# DASO-Net — FINAL MULTI-SEED PREFLIGHT / SEED-MECHANISM AUDIT
#
# PURPOSE:
#   Inspect the frozen trainer/launcher/config before final
#   multi-seed training.
#
# NO TRAINING.
# NO FILE MODIFICATION.
# NO EXTERNAL DATA.
#
# We need to establish:
#   1. how model/data RNG seeds are set
#   2. whether seed is configurable from CLI/config
#   3. how DDP rank-specific randomness is handled
#   4. how DataLoader workers are seeded
#   5. how output/run names are chosen
#   6. how to force FRESH initialization safely
# ============================================================

from __future__ import annotations

import ast
import json
import re
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)

CHECKPOINT_LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_baseline_curve1000updates_v1.json"
)

FREEZE_JSON = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1/"
    "FINAL_TRAINING_RECIPE_FREEZE_v1.json"
)


# ============================================================
# HELPERS
# ============================================================

def require(path: Path, label: str):

    if not path.exists():
        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def context_lines(
    text: str,
    pattern: str,
    radius: int = 3,
):
    lines = text.splitlines()

    hits = []

    rx = re.compile(
        pattern,
        flags=re.IGNORECASE,
    )

    for i, line in enumerate(lines):

        if rx.search(line):

            start = max(
                0,
                i - radius,
            )

            end = min(
                len(lines),
                i + radius + 1,
            )

            hits.append(
                (
                    i + 1,
                    lines[start:end],
                )
            )

    return hits


def print_hits(
    title: str,
    hits,
    max_hits: int = 20,
):

    print()
    print(
        "=" * 72
    )

    print(title)

    print(
        "=" * 72
    )

    if not hits:
        print("NO MATCHES")
        return

    for hit_index, (
        line_no,
        block,
    ) in enumerate(
        hits[:max_hits],
        start=1,
    ):

        print()
        print(
            f"[match {hit_index}] "
            f"around line {line_no}"
        )

        for line in block:
            print(line)

    if len(hits) > max_hits:
        print()
        print(
            f"... {len(hits) - max_hits} "
            f"additional matches omitted"
        )


# ============================================================
# BASIC AUDIT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FINAL MULTI-SEED PREFLIGHT"
)

print(
    "============================================================"
)


for path, label in [
    (
        TRAINER,
        "Base trainer",
    ),
    (
        CHECKPOINT_LAUNCHER,
        "Checkpoint launcher",
    ),
    (
        CONFIG,
        "Development config",
    ),
    (
        FREEZE_JSON,
        "Final recipe freeze record",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


trainer_text = TRAINER.read_text(
    encoding="utf-8"
)

launcher_text = (
    CHECKPOINT_LAUNCHER.read_text(
        encoding="utf-8"
    )
)

config = json.loads(
    CONFIG.read_text(
        encoding="utf-8"
    )
)

freeze = json.loads(
    FREEZE_JSON.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# CONFIRM FROZEN BUDGET
# ============================================================

budget = int(
    freeze[
        "training_duration"
    ][
        "final_budget"
    ]
)


if budget != 2750:

    raise RuntimeError(
        f"Frozen budget unexpectedly = {budget}"
    )


print()
print(
    "Frozen training budget: 2750 optimizer updates — PASS"
)


# ============================================================
# PYTHON SYNTAX
# ============================================================

ast.parse(
    trainer_text
)

ast.parse(
    launcher_text
)


print(
    "Trainer/launcher Python syntax: PASS"
)


# ============================================================
# ARGPARSE OPTION EXTRACTION
# ============================================================

def extract_add_argument_options(
    source: str,
):

    options = []

    pattern = re.compile(
        r'add_argument\s*\(\s*'
        r'([\'"][^\'"]+[\'"])',
        re.MULTILINE,
    )

    for match in pattern.finditer(
        source
    ):

        raw = match.group(1)

        try:
            value = ast.literal_eval(
                raw
            )
        except Exception:
            continue

        options.append(
            value
        )

    return sorted(
        set(
            options
        )
    )


trainer_args = (
    extract_add_argument_options(
        trainer_text
    )
)

launcher_args = (
    extract_add_argument_options(
        launcher_text
    )
)


print()
print(
    "============================================================"
)

print(
    "TRAINER CLI OPTIONS"
)

print(
    "============================================================"
)

for arg in trainer_args:
    print(arg)


print()
print(
    "============================================================"
)

print(
    "CHECKPOINT-LAUNCHER CLI OPTIONS"
)

print(
    "============================================================"
)

for arg in launcher_args:
    print(arg)


# ============================================================
# CONFIG SEARCH FOR SEED / RUN-NAME / OUTPUT
# ============================================================

def recursive_find(
    obj,
    keyword: str,
    prefix: str = "",
):

    found = []

    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            path = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            if keyword.lower() in str(
                key
            ).lower():

                found.append(
                    (
                        path,
                        value,
                    )
                )

            found.extend(
                recursive_find(
                    value,
                    keyword,
                    path,
                )
            )

    elif isinstance(
        obj,
        list,
    ):

        for i, value in enumerate(
            obj
        ):

            path = (
                f"{prefix}[{i}]"
            )

            found.extend(
                recursive_find(
                    value,
                    keyword,
                    path,
                )
            )

    return found


for keyword in [
    "seed",
    "run",
    "name",
    "output",
    "resume",
]:

    matches = recursive_find(
        config,
        keyword,
    )

    print()
    print(
        f"CONFIG KEYS CONTAINING '{keyword}':"
    )

    if not matches:
        print(
            "  NONE"
        )

    else:

        for path, value in matches:
            print(
                f"  {path} = {value!r}"
            )


# ============================================================
# RNG / SEEDING SOURCE AUDIT
# ============================================================

seed_patterns = {
    "seed keyword":
        r"\bseed\b",

    "Python random.seed":
        r"random\.seed",

    "NumPy seed":
        r"(np|numpy)\.random\.seed",

    "torch.manual_seed":
        r"torch\.manual_seed",

    "torch.cuda.manual_seed":
        r"torch\.cuda\.manual_seed",

    "torch.cuda.manual_seed_all":
        r"torch\.cuda\.manual_seed_all",

    "Generator manual_seed":
        r"\.manual_seed\s*\(",

    "DistributedSampler":
        r"DistributedSampler",

    "sampler set_epoch":
        r"\.set_epoch\s*\(",

    "worker_init_fn":
        r"worker_init_fn",

    "DataLoader generator":
        r"generator\s*=",

    "rank usage":
        r"\brank\b",

    "local_rank usage":
        r"local_rank",
}


for title, pattern in seed_patterns.items():

    hits = context_lines(
        trainer_text,
        pattern,
        radius=3,
    )

    print_hits(
        f"TRAINER — {title}",
        hits,
        max_hits=12,
    )


# ============================================================
# OUTPUT / RUN-NAME / RESUME AUDIT
# ============================================================

output_patterns = {
    "output directory":
        r"output",

    "run name":
        r"run[_\- ]?name",

    "experiment name":
        r"experiment",

    "mkdir":
        r"mkdir",

    "resume":
        r"resume",

    "load checkpoint":
        r"load_checkpoint",

    "model construction":
        r"(build_model|DASONet|DASO)",
}


for title, pattern in output_patterns.items():

    hits = context_lines(
        trainer_text,
        pattern,
        radius=4,
    )

    print_hits(
        f"TRAINER — {title}",
        hits,
        max_hits=15,
    )


# ============================================================
# LAUNCHER OVERRIDE AUDIT
# ============================================================

for title, pattern in [
    (
        "seed handling",
        r"\bseed\b",
    ),
    (
        "output/run handling",
        r"(output|run_name|run-name)",
    ),
    (
        "resume handling",
        r"resume",
    ),
    (
        "argv forwarding",
        r"(sys\.argv|parse_args|argv)",
    ),
]:

    hits = context_lines(
        launcher_text,
        pattern,
        radius=4,
    )

    print_hits(
        f"CHECKPOINT LAUNCHER — {title}",
        hits,
        max_hits=15,
    )


# ============================================================
# SIMPLE BOOLEAN CAPABILITY SUMMARY
# ============================================================

trainer_seed_cli = (
    "--seed"
    in trainer_args
)

launcher_seed_cli = (
    "--seed"
    in launcher_args
)

trainer_run_name_cli = any(
    arg in trainer_args
    for arg in [
        "--run-name",
        "--run_name",
        "--name",
    ]
)

launcher_run_name_cli = any(
    arg in launcher_args
    for arg in [
        "--run-name",
        "--run_name",
        "--name",
    ]
)


rng_calls = {
    "python_random_seed":
        "random.seed"
        in trainer_text,

    "numpy_random_seed":
        (
            "np.random.seed"
            in trainer_text
            or
            "numpy.random.seed"
            in trainer_text
        ),

    "torch_manual_seed":
        "torch.manual_seed"
        in trainer_text,

    "torch_cuda_seed":
        (
            "torch.cuda.manual_seed"
            in trainer_text
        ),

    "distributed_sampler":
        "DistributedSampler"
        in trainer_text,

    "sampler_set_epoch":
        ".set_epoch("
        in trainer_text,

    "worker_init_fn":
        "worker_init_fn"
        in trainer_text,

    "dataloader_generator":
        "generator="
        in trainer_text.replace(
            " ",
            "",
        ),
}


summary = {
    "frozen_optimizer_updates":
        2750,

    "trainer_seed_cli":
        trainer_seed_cli,

    "launcher_seed_cli":
        launcher_seed_cli,

    "trainer_run_name_cli":
        trainer_run_name_cli,

    "launcher_run_name_cli":
        launcher_run_name_cli,

    "rng_calls":
        rng_calls,

    "trainer_cli_options":
        trainer_args,

    "launcher_cli_options":
        launcher_args,

    "config_seed_matches":
        [
            {
                "path":
                    path,

                "value":
                    value,
            }

            for path, value
            in recursive_find(
                config,
                "seed",
            )
        ],
}


print()
print(
    "============================================================"
)

print(
    "MULTI-SEED CAPABILITY SUMMARY"
)

print(
    "============================================================"
)

print(
    "Frozen optimizer updates:",
    2750,
)

print(
    "Trainer has --seed:",
    trainer_seed_cli,
)

print(
    "Launcher has --seed:",
    launcher_seed_cli,
)

print(
    "Trainer has run-name CLI:",
    trainer_run_name_cli,
)

print(
    "Launcher has run-name CLI:",
    launcher_run_name_cli,
)

print()

for key, value in rng_calls.items():

    print(
        f"{key:24s}: {value}"
    )


# ============================================================
# SAVE AUDIT REPORT
# ============================================================

audit_path = Path(
    "/kaggle/working/"
    "DASO-Net-FinalMultiSeed-Preflight-v1.json"
)


audit_path.write_text(
    json.dumps(
        summary,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


print()
print(
    "Audit JSON:"
)

print(
    audit_path
)

print()
print(
    "============================================================"
)

print(
    "FINAL MULTI-SEED PREFLIGHT AUDIT: COMPLETE"
)

print(
    "============================================================"
)

print(
    "NO TRAINING WAS STARTED."
)

print(
    "Paste the section beginning:"
)

print(
    "'MULTI-SEED CAPABILITY SUMMARY'"
)

print(
    "plus the TRAINER seed-related source blocks."
)

DASO-Net — FINAL MULTI-SEED PREFLIGHT
PASS  Base trainer
PASS  Checkpoint launcher
PASS  Development config
PASS  Final recipe freeze record

Frozen training budget: 2750 optimizer updates — PASS
Trainer/launcher Python syntax: PASS

TRAINER CLI OPTIONS
--accum
--anchor-batch
--config
--epochs
--log-every
--max-train-steps
--max-val-steps
--no-amp
--num-workers
--output-root
--per-rank-batch
--project-root
--resume
--smoke

CHECKPOINT-LAUNCHER CLI OPTIONS

CONFIG KEYS CONTAINING 'seed':
  seed = 20260916

CONFIG KEYS CONTAINING 'run':
  NONE

CONFIG KEYS CONTAINING 'name':
  experiment_name = 'dasonet_phase5c_baseline_curve1000updates_v1'

CONFIG KEYS CONTAINING 'output':
  storage.output_subdir = 'dasonet_phase5b'

CONFIG KEYS CONTAINING 'resume':
  NONE

TRAINER — seed keyword

[match 1] around line 91
        dist.destroy_process_group()


def set_seed(seed: int, rank: int) -> None:
    value = int(seed) + int(rank) * 100003
    random.seed(value)
    np.random.seed(value % (2**32))

In [12]:
# ============================================================
# DASO-Net — FINAL MULTI-SEED PLAN FREEZE v1
#
# NO TRAINING.
#
# Creates exactly 3 immutable final-training configs.
#
# Final recipe:
#   2750 optimizer updates
#   fresh initialization
#   2x T4
#   native 1024
#   batch/GPU = 1
#   anchor pair batch/GPU = 2
#   accumulation = 4
#   effective synthetic batch = 8
#   LR = 3e-4
#   warmup = 50 optimizer updates
#   constant post-warmup LR
#   FP16 initial scale = 32
#   original PCC retained
#   CRPA retained
#   PCC-center rejected/disabled
#
# Seed selection is deterministically derived from the
# already-frozen training-recipe ZIP SHA256.
#
# STOP after this cell.
# ============================================================

from __future__ import annotations

import copy
import hashlib
import importlib.util
import json
import shutil
import sys
from pathlib import Path

import torch


# ============================================================
# 1. PATHS
# ============================================================

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

SOURCE_CONFIG = Path(
    "/kaggle/working/"
    "phase5c_baseline_curve1000updates_v1.json"
)

TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)

FREEZE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

FREEZE_JSON = (
    FREEZE_ROOT
    / "FINAL_TRAINING_RECIPE_FREEZE_v1.json"
)

FREEZE_ZIP = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1.zip"
)

MULTISEED_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-FinalMultiSeed-v1"
)

CONFIG_DIR = (
    MULTISEED_ROOT
    / "configs"
)

PLAN_JSON = (
    MULTISEED_ROOT
    / "FINAL_MULTISEED_PLAN_v1.json"
)

PLAN_README = (
    MULTISEED_ROOT
    / "README_FINAL_MULTISEED.txt"
)


# ============================================================
# 2. FROZEN IDENTITY
# ============================================================

EXPECTED_FREEZE_ZIP_SHA = (
    "b102deb8ec9de4be3816e4450a4dc7b2"
    "f78f5439634cee02fe4656d1a268bf3a"
)

FINAL_OPTIMIZER_UPDATES = 2750
FINAL_SEED_COUNT = 3


# ============================================================
# 3. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def tensor_state_sha256(
    module: torch.nn.Module,
) -> str:
    """
    Deterministic hash of model state tensor contents.
    Used only to prove initialization repeatability/divergence.
    """

    h = hashlib.sha256()

    state = module.state_dict()

    for key in sorted(
        state.keys()
    ):

        tensor = (
            state[
                key
            ]
            .detach()
            .cpu()
            .contiguous()
        )

        h.update(
            key.encode(
                "utf-8"
            )
        )

        h.update(
            str(
                tensor.dtype
            ).encode(
                "utf-8"
            )
        )

        h.update(
            str(
                tuple(
                    tensor.shape
                )
            ).encode(
                "utf-8"
            )
        )

        h.update(
            tensor.numpy().tobytes()
        )

    return h.hexdigest()


def recursive_diff(
    a,
    b,
    prefix: str = "",
):
    """
    Return leaf paths whose values differ.
    """

    diffs = []

    if isinstance(a, dict) and isinstance(b, dict):

        keys = sorted(
            set(a.keys())
            | set(b.keys())
        )

        for key in keys:

            path = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            if key not in a:

                diffs.append(
                    (
                        path,
                        "<MISSING>",
                        b[key],
                    )
                )

            elif key not in b:

                diffs.append(
                    (
                        path,
                        a[key],
                        "<MISSING>",
                    )
                )

            else:

                diffs.extend(
                    recursive_diff(
                        a[key],
                        b[key],
                        path,
                    )
                )

        return diffs


    if isinstance(a, list) and isinstance(b, list):

        if a != b:

            diffs.append(
                (
                    prefix,
                    a,
                    b,
                )
            )

        return diffs


    if a != b:

        diffs.append(
            (
                prefix,
                a,
                b,
            )
        )

    return diffs


# ============================================================
# 4. REQUIRED FILE AUDIT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FINAL MULTI-SEED PLAN FREEZE v1"
)

print(
    "============================================================"
)


for path, label in [
    (
        CODE_ROOT,
        "Frozen code tree",
    ),
    (
        SOURCE_CONFIG,
        "Frozen source config",
    ),
    (
        TRAINER,
        "Frozen trainer",
    ),
    (
        FREEZE_JSON,
        "Final recipe freeze record",
    ),
    (
        FREEZE_ZIP,
        "Final recipe freeze ZIP",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 5. VERIFY FREEZE ZIP IDENTITY
# ============================================================

actual_freeze_sha = sha256_file(
    FREEZE_ZIP
)


if (
    actual_freeze_sha
    != EXPECTED_FREEZE_ZIP_SHA
):

    raise RuntimeError(
        "Final recipe freeze ZIP SHA mismatch.\n"
        f"Expected: {EXPECTED_FREEZE_ZIP_SHA}\n"
        f"Actual:   {actual_freeze_sha}"
    )


print()
print(
    "Training-recipe freeze SHA: PASS"
)


# ============================================================
# 6. VERIFY FROZEN BUDGET
# ============================================================

freeze = json.loads(
    FREEZE_JSON.read_text(
        encoding="utf-8"
    )
)


frozen_budget = int(
    freeze[
        "training_duration"
    ][
        "final_budget"
    ]
)


if frozen_budget != FINAL_OPTIMIZER_UPDATES:

    raise RuntimeError(
        "Frozen training budget is not 2750."
    )


print(
    "Frozen optimizer budget: 2750 — PASS"
)


# ============================================================
# 7. SOURCE CONFIG
# ============================================================

source_cfg = json.loads(
    SOURCE_CONFIG.read_text(
        encoding="utf-8"
    )
)


# PCC-center must remain disabled.
pcc_center = (
    source_cfg
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if (
    pcc_center is not None
    and bool(
        pcc_center.get(
            "enabled",
            False,
        )
    )
):

    raise RuntimeError(
        "PCC-center unexpectedly enabled."
    )


print(
    "PCC-center rejected/disabled: PASS"
)


# ============================================================
# 8. DERIVE FINAL SEEDS FROM FROZEN SHA
#
# This prevents cherry-picking seed numbers.
# ============================================================

seed_records = []


for index in range(
    1,
    FINAL_SEED_COUNT + 1,
):

    derivation_text = (
        f"DASO-Net-final-seed-{index}|"
        f"{EXPECTED_FREEZE_ZIP_SHA}"
    )

    digest = hashlib.sha256(
        derivation_text.encode(
            "utf-8"
        )
    ).hexdigest()

    raw = int(
        digest[:8],
        16,
    )

    seed = (
        raw
        % (
            2**31 - 1
        )
    )

    if seed == 0:

        seed = index

    seed_records.append(
        {
            "seed_index":
                index,

            "seed":
                seed,

            "derivation_text":
                derivation_text,

            "derivation_sha256":
                digest,
        }
    )


seeds = [
    record[
        "seed"
    ]
    for record in seed_records
]


if len(
    set(
        seeds
    )
) != FINAL_SEED_COUNT:

    raise RuntimeError(
        "Derived final seeds are not unique."
    )


EXPECTED_SEEDS = [
    1782656963,
    901395089,
    252916394,
]


if seeds != EXPECTED_SEEDS:

    raise RuntimeError(
        "Derived seeds do not match expected values.\n"
        f"Expected: {EXPECTED_SEEDS}\n"
        f"Actual:   {seeds}"
    )


print()
print(
    "Deterministically derived final seeds: PASS"
)


for record in seed_records:

    print(
        f"  Seed {record['seed_index']}: "
        f"{record['seed']}"
    )


# ============================================================
# 9. CREATE CLEAN MULTI-SEED PLAN DIRECTORY
# ============================================================

if MULTISEED_ROOT.exists():

    # It is safe to recreate only if no final run output
    # directories have been generated yet.
    existing_final_runs = list(
        Path(
            "/kaggle/working/"
            "dasonet_phase5b"
        ).glob(
            "dasonet_final_seed*"
        )
    )

    if existing_final_runs:

        raise RuntimeError(
            "Final-seed run directories already exist.\n"
            "Refusing to recreate the multi-seed plan."
        )

    shutil.rmtree(
        MULTISEED_ROOT
    )


CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 10. CREATE THREE FINAL CONFIGS
#
# ONLY:
#   seed
#   experiment_name
#
# are permitted to differ from the frozen development config.
# ============================================================

final_configs = []


for record in seed_records:

    index = int(
        record[
            "seed_index"
        ]
    )

    seed = int(
        record[
            "seed"
        ]
    )

    cfg = copy.deepcopy(
        source_cfg
    )

    experiment_name = (
        f"dasonet_final_seed{index:02d}"
        f"_s{seed}"
    )

    cfg[
        "seed"
    ] = seed

    cfg[
        "experiment_name"
    ] = experiment_name


    differences = recursive_diff(
        source_cfg,
        cfg,
    )


    difference_paths = {
        item[
            0
        ]
        for item in differences
    }


    allowed_differences = {
        "seed",
        "experiment_name",
    }


    if (
        difference_paths
        != allowed_differences
    ):

        raise RuntimeError(
            f"Seed {index} config differs in "
            f"unexpected fields:\n"
            f"{differences}"
        )


    final_pcc_center = (
        cfg
        .get(
            "loss",
            {}
        )
        .get(
            "pcc_center"
        )
    )


    if (
        final_pcc_center is not None
        and bool(
            final_pcc_center.get(
                "enabled",
                False,
            )
        )
    ):

        raise RuntimeError(
            f"Seed {index}: PCC-center enabled."
        )


    config_path = (
        CONFIG_DIR
        / (
            f"final_seed{index:02d}"
            f"_s{seed}.json"
        )
    )


    config_path.write_text(
        json.dumps(
            cfg,
            indent=2,
        ),
        encoding="utf-8",
    )


    output_dir = (
        Path(
            "/kaggle/working"
        )
        / cfg[
            "storage"
        ][
            "output_subdir"
        ]
        / experiment_name
    )


    if output_dir.exists():

        raise RuntimeError(
            "Final output directory already exists:\n"
            f"{output_dir}\n"
            "Fresh initialization cannot yet be guaranteed."
        )


    final_configs.append(
        {
            "seed_index":
                index,

            "seed":
                seed,

            "experiment_name":
                experiment_name,

            "config_path":
                str(
                    config_path
                ),

            "config_sha256":
                sha256_file(
                    config_path
                ),

            "output_dir":
                str(
                    output_dir
                ),

            "allowed_config_changes":
                [
                    "seed",
                    "experiment_name",
                ],
        }
    )


print()
print(
    "Final configs created:"
)


for item in final_configs:

    print(
        f"PASS  Seed {item['seed_index']} "
        f"| seed={item['seed']} "
        f"| {Path(item['config_path']).name}"
    )


# ============================================================
# 11. IMPORT FROZEN TRAINER FOR INITIALIZATION TEST
# ============================================================

src_root = (
    CODE_ROOT
    / "src"
)


if str(
    src_root
) not in sys.path:

    sys.path.insert(
        0,
        str(
            src_root
        ),
    )


spec = importlib.util.spec_from_file_location(
    "dasonet_final_seed_audit_trainer",
    TRAINER,
)


if (
    spec is None
    or spec.loader is None
):

    raise RuntimeError(
        "Could not import frozen trainer."
    )


trainer_module = (
    importlib.util.module_from_spec(
        spec
    )
)

spec.loader.exec_module(
    trainer_module
)


# ============================================================
# 12. MODEL-INITIALIZATION DETERMINISM TEST
#
# Same seed/rank -> identical initialization.
# Different seed -> different initialization.
# ============================================================

cfg_seed1 = json.loads(
    Path(
        final_configs[
            0
        ][
            "config_path"
        ]
    ).read_text(
        encoding="utf-8"
    )
)

cfg_seed2 = json.loads(
    Path(
        final_configs[
            1
        ][
            "config_path"
        ]
    ).read_text(
        encoding="utf-8"
    )
)


# Seed 1, first initialization.
trainer_module.set_seed(
    int(
        cfg_seed1[
            "seed"
        ]
    ),
    0,
)

model_a = (
    trainer_module.build_model(
        cfg_seed1
    )
)

hash_a = tensor_state_sha256(
    model_a
)

del model_a


# Seed 1, repeated initialization.
trainer_module.set_seed(
    int(
        cfg_seed1[
            "seed"
        ]
    ),
    0,
)

model_b = (
    trainer_module.build_model(
        cfg_seed1
    )
)

hash_b = tensor_state_sha256(
    model_b
)

del model_b


# Seed 2 initialization.
trainer_module.set_seed(
    int(
        cfg_seed2[
            "seed"
        ]
    ),
    0,
)

model_c = (
    trainer_module.build_model(
        cfg_seed2
    )
)

hash_c = tensor_state_sha256(
    model_c
)

del model_c


if hash_a != hash_b:

    raise RuntimeError(
        "Same-seed model initialization is NOT repeatable."
    )


if hash_a == hash_c:

    raise RuntimeError(
        "Different final seeds produced identical "
        "model initialization."
    )


print()
print(
    "Model initialization reproducibility:"
)

print(
    "PASS  same seed -> identical initialization"
)

print(
    "PASS  different seed -> different initialization"
)

print(
    "Seed-1 init SHA:",
    hash_a,
)

print(
    "Seed-2 init SHA:",
    hash_c,
)


# ============================================================
# 13. FORMAL MULTI-SEED PLAN
# ============================================================

plan = {
    "plan_name":
        "DASO-Net-FinalMultiSeed-v1",

    "status":
        "FROZEN_BEFORE_FINAL_TRAINING",

    "final_seed_count":
        3,

    "seed_selection":
        {
            "method":
                (
                    "SHA256-derived from the frozen "
                    "Phase5C training-recipe ZIP SHA"
                ),

            "freeze_zip_sha256":
                EXPECTED_FREEZE_ZIP_SHA,

            "records":
                seed_records,
        },

    "frozen_training_recipe":
        {
            "optimizer_updates":
                2750,

            "training_blocks":
                11,

            "optimizer_updates_per_block":
                250,

            "epochs_cli":
                11,

            "max_train_steps_per_rank_per_block":
                1000,

            "max_val_steps_per_block":
                100,

            "world_size":
                2,

            "gpu":
                "Tesla T4",

            "native_tile":
                1024,

            "per_rank_synthetic_batch":
                1,

            "per_rank_anchor_pair_batch":
                2,

            "gradient_accumulation":
                4,

            "effective_synthetic_batch":
                8,

            "synthetic_workers_per_rank":
                2,

            "anchor_workers_per_rank":
                1,

            "learning_rate":
                3e-4,

            "warmup_optimizer_updates":
                50,

            "post_warmup_schedule":
                "constant",

            "amp":
                "FP16",

            "amp_initial_scale":
                32,

            "resume":
                "none",

            "initialization":
                "fresh",

            "terminal_checkpoint":
                "optimizer_step_2750",

            "early_stopping":
                False,

            "pcc":
                "original feature/embedding PCC retained",

            "crpa":
                "retained",

            "pcc_center":
                "rejected and disabled",
        },

    "final_seed_configs":
        final_configs,

    "selection_policy":
        {
            "development_checkpoint_reuse":
                False,

            "final_seed_checkpoint_selection":
                (
                    "terminal checkpoint at exactly "
                    "2750 optimizer updates"
                ),

            "best_pt_for_final_reporting":
                False,

            "external_test_data_for_model_selection":
                False,
        },

    "initialization_audit":
        {
            "same_seed_repeat_hash":
                hash_a,

            "same_seed_second_hash":
                hash_b,

            "different_seed_hash":
                hash_c,

            "same_seed_identical":
                True,

            "different_seed_different":
                True,
        },

    "source_files":
        {
            "source_config":
                {
                    "path":
                        str(
                            SOURCE_CONFIG
                        ),

                    "sha256":
                        sha256_file(
                            SOURCE_CONFIG
                        ),
                },

            "trainer":
                {
                    "path":
                        str(
                            TRAINER
                        ),

                    "sha256":
                        sha256_file(
                            TRAINER
                        ),
                },

            "recipe_freeze":
                {
                    "path":
                        str(
                            FREEZE_JSON
                        ),

                    "sha256":
                        sha256_file(
                            FREEZE_JSON
                        ),
                },
        },
}


PLAN_JSON.write_text(
    json.dumps(
        plan,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 14. README
# ============================================================

PLAN_README.write_text(
    "\n".join(
        [
            "DASO-Net Final Multi-Seed Plan v1",
            "",
            "FINAL TRAINING BUDGET:",
            "2750 optimizer updates",
            "",
            "FINAL INDEPENDENT SEEDS:",
            f"Seed 1: {seeds[0]}",
            f"Seed 2: {seeds[1]}",
            f"Seed 3: {seeds[2]}",
            "",
            (
                "Seeds were deterministically derived from "
                "the already-frozen Phase5C recipe ZIP SHA256."
            ),
            "",
            "ONLY seed and experiment_name differ between configs.",
            "",
            "Every final seed:",
            "- starts from fresh random initialization",
            "- uses --resume none",
            "- trains exactly 2750 optimizer updates",
            "- uses terminal checkpoint at step 2750",
            "- does NOT use best.pt for final reporting",
            "- retains original PCC and CRPA",
            "- keeps PCC-center disabled",
            "- does not use external test data for selection",
            "",
            (
                "The development 2750 checkpoint must NOT "
                "be reused as a final seed."
            ),
            "",
        ]
    ),
    encoding="utf-8",
)


# ============================================================
# 15. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL MULTI-SEED PLAN"
)

print(
    "============================================================"
)


for item in final_configs:

    print()
    print(
        f"FINAL SEED {item['seed_index']}"
    )

    print(
        "  RNG seed:       ",
        item[
            "seed"
        ],
    )

    print(
        "  experiment:     ",
        item[
            "experiment_name"
        ],
    )

    print(
        "  config:"
    )

    print(
        "   ",
        item[
            "config_path"
        ],
    )

    print(
        "  output:"
    )

    print(
        "   ",
        item[
            "output_dir"
        ],
    )


print()
print(
    "Frozen budget:       2750 optimizer updates"
)

print(
    "Final seed count:     3"
)

print(
    "Fresh initialization: YES"
)

print(
    "Resume:               NONE"
)

print(
    "Terminal ckpt only:   YES"
)

print(
    "Trainer modified:     NO"
)

print(
    "External tuning:      NO"
)

print()

print(
    "Plan JSON:"
)

print(
    PLAN_JSON
)

print()

print(
    "============================================================"
)

print(
    "FINAL MULTI-SEED PLAN FREEZE: PASS"
)

print(
    "============================================================"
)

print(
    "NO TRAINING WAS STARTED."
)

print(
    "STOP HERE."
)

print(
    "Paste this cell's final output before launching Final Seed 1."
)

DASO-Net — FINAL MULTI-SEED PLAN FREEZE v1
PASS  Frozen code tree
PASS  Frozen source config
PASS  Frozen trainer
PASS  Final recipe freeze record
PASS  Final recipe freeze ZIP

Training-recipe freeze SHA: PASS
Frozen optimizer budget: 2750 — PASS
PCC-center rejected/disabled: PASS

Deterministically derived final seeds: PASS
  Seed 1: 1782656963
  Seed 2: 901395089
  Seed 3: 252916394

Final configs created:
PASS  Seed 1 | seed=1782656963 | final_seed01_s1782656963.json
PASS  Seed 2 | seed=901395089 | final_seed02_s901395089.json
PASS  Seed 3 | seed=252916394 | final_seed03_s252916394.json

Model initialization reproducibility:
PASS  same seed -> identical initialization
PASS  different seed -> different initialization
Seed-1 init SHA: 019b85fc7db44e2ae9665feb01ea59de3a5dbc05daa1024c135803c8fb75f5d1
Seed-2 init SHA: 32bcf83d5d1b3482c3152f8d20e5baa69a5b8c46df56cfa2367e23bf2580c377

FINAL MULTI-SEED PLAN

FINAL SEED 1
  RNG seed:        1782656963
  experiment:      dasonet_final_seed01

In [13]:
# ============================================================
# DASO-Net — FINAL MULTI-SEED TRAINING
# FINAL SEED 1 / 3
#
# Seed:
#   1782656963
#
# Frozen final recipe:
#   fresh initialization
#   2750 optimizer updates
#   11 training blocks
#   1000 microsteps/rank/block
#   accumulation = 4
#   250 optimizer updates/block
#   native 1024
#   2x T4
#   batch/GPU = 1
#   anchor pair batch/GPU = 2
#   synthetic workers/GPU = 2
#   anchor workers/GPU = 1
#   LR = 3e-4
#   warmup = 50 optimizer updates
#   constant post-warmup LR
#   AMP FP16 initial scale = 32
#   original PCC retained
#   CRPA retained
#   PCC-center disabled
#
# IMPORTANT:
#   NO RESUME.
#   NO EARLY STOPPING.
#   DO NOT USE best.pt FOR FINAL REPORTING.
#   TERMINAL checkpoint @ 2750 is the final Seed-1 model.
#
# This cell performs actual final training.
# ============================================================

from __future__ import annotations

import hashlib
import json
import subprocess
import time
from pathlib import Path

import torch


# ============================================================
# 1. CONSTANTS
# ============================================================

FINAL_SEED_INDEX = 1
FINAL_SEED = 1782656963

FINAL_EXPERIMENT = (
    "dasonet_final_seed01_s1782656963"
)

FINAL_OPTIMIZER_UPDATES = 2750
FINAL_EPOCHS = 11

EXPECTED_TERMINAL_EPOCH = 10

EXPECTED_FREEZE_ZIP_SHA = (
    "b102deb8ec9de4be3816e4450a4dc7b2"
    "f78f5439634cee02fe4656d1a268bf3a"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)


# ============================================================
# 2. PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

PLAN_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-FinalMultiSeed-v1"
)

PLAN_JSON = (
    PLAN_ROOT
    / "FINAL_MULTISEED_PLAN_v1.json"
)

CONFIG = (
    PLAN_ROOT
    / "configs"
    / "final_seed01_s1782656963.json"
)

RECIPE_FREEZE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

RECIPE_FREEZE_JSON = (
    RECIPE_FREEZE_ROOT
    / "FINAL_TRAINING_RECIPE_FREEZE_v1.json"
)

RECIPE_FREEZE_ZIP = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1.zip"
)

OUTPUT_ROOT = Path(
    "/kaggle/working"
)

RUN_ROOT = (
    OUTPUT_ROOT
    / "dasonet_phase5b"
    / FINAL_EXPERIMENT
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

TERMINAL_CHECKPOINT = (
    SNAPSHOT_DIR
    / "epoch_010_opt_002750.pt"
)

RUN_STATUS = (
    RUN_ROOT
    / "run_status.json"
)

RESOLVED_CONFIG = (
    RUN_ROOT
    / "resolved_config.json"
)

HISTORY = (
    RUN_ROOT
    / "history.csv"
)


# ============================================================
# 3. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


# ============================================================
# 4. REQUIRED INPUT AUDIT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FINAL SEED 1 / 3 PREFLIGHT"
)

print(
    "============================================================"
)


for path, label in [
    (
        PROJECT_ROOT,
        "Frozen project dataset",
    ),
    (
        CODE_ROOT,
        "Frozen code tree",
    ),
    (
        TRAINER,
        "Frozen trainer",
    ),
    (
        LAUNCHER,
        "Checkpoint/snapshot launcher",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
    (
        PLAN_JSON,
        "Frozen multi-seed plan",
    ),
    (
        CONFIG,
        "Final Seed-1 config",
    ),
    (
        RECIPE_FREEZE_JSON,
        "Final training-recipe freeze",
    ),
    (
        RECIPE_FREEZE_ZIP,
        "Final recipe freeze ZIP",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 5. VERIFY ORIGINAL TRAINING-RECIPE FREEZE
# ============================================================

freeze_zip_sha = sha256_file(
    RECIPE_FREEZE_ZIP
)


if (
    freeze_zip_sha
    != EXPECTED_FREEZE_ZIP_SHA
):

    raise RuntimeError(
        "Training-recipe FREEZE ZIP SHA mismatch.\n"
        f"Expected: {EXPECTED_FREEZE_ZIP_SHA}\n"
        f"Actual:   {freeze_zip_sha}"
    )


print()
print(
    "Training-recipe freeze SHA: PASS"
)


# ============================================================
# 6. VERIFY DECODER
# ============================================================

decoder_sha = sha256_file(
    DECODER
)


if decoder_sha != EXPECTED_DECODER_SHA:

    raise RuntimeError(
        "Decoder SHA mismatch.\n"
        f"Expected: {EXPECTED_DECODER_SHA}\n"
        f"Actual:   {decoder_sha}"
    )


print(
    "Validated decoder SHA: PASS"
)


# ============================================================
# 7. VERIFY MULTI-SEED PLAN
# ============================================================

plan = json.loads(
    PLAN_JSON.read_text(
        encoding="utf-8"
    )
)


if (
    plan.get(
        "status"
    )
    != "FROZEN_BEFORE_FINAL_TRAINING"
):

    raise RuntimeError(
        "Multi-seed plan is not in frozen state."
    )


if int(
    plan[
        "frozen_training_recipe"
    ][
        "optimizer_updates"
    ]
) != FINAL_OPTIMIZER_UPDATES:

    raise RuntimeError(
        "Multi-seed plan training budget mismatch."
    )


if int(
    plan[
        "final_seed_count"
    ]
) != 3:

    raise RuntimeError(
        "Expected exactly 3 final seeds."
    )


planned_seed = None


for item in plan[
    "final_seed_configs"
]:

    if int(
        item[
            "seed_index"
        ]
    ) == FINAL_SEED_INDEX:

        planned_seed = item

        break


if planned_seed is None:

    raise RuntimeError(
        "Final Seed-1 plan entry missing."
    )


if int(
    planned_seed[
        "seed"
    ]
) != FINAL_SEED:

    raise RuntimeError(
        "Final Seed-1 RNG seed mismatch."
    )


if (
    planned_seed[
        "experiment_name"
    ]
    != FINAL_EXPERIMENT
):

    raise RuntimeError(
        "Final Seed-1 experiment name mismatch."
    )


if (
    Path(
        planned_seed[
            "config_path"
        ]
    )
    != CONFIG
):

    raise RuntimeError(
        "Final Seed-1 config path mismatch."
    )


config_sha = sha256_file(
    CONFIG
)


if (
    config_sha
    != planned_seed[
        "config_sha256"
    ]
):

    raise RuntimeError(
        "Final Seed-1 config SHA mismatch.\n"
        f"Expected: {planned_seed['config_sha256']}\n"
        f"Actual:   {config_sha}"
    )


print(
    "Frozen multi-seed plan identity: PASS"
)

print(
    "Final Seed-1 config SHA: PASS"
)


# ============================================================
# 8. VERIFY SEED-1 CONFIG CONTENT
# ============================================================

cfg = json.loads(
    CONFIG.read_text(
        encoding="utf-8"
    )
)


if int(
    cfg[
        "seed"
    ]
) != FINAL_SEED:

    raise RuntimeError(
        "Seed-1 config RNG seed mismatch."
    )


if (
    cfg[
        "experiment_name"
    ]
    != FINAL_EXPERIMENT
):

    raise RuntimeError(
        "Seed-1 experiment_name mismatch."
    )


pcc_center = (
    cfg
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if (
    pcc_center is not None
    and bool(
        pcc_center.get(
            "enabled",
            False,
        )
    )
):

    raise RuntimeError(
        "PCC-center unexpectedly ENABLED."
    )


print(
    "Seed-1 RNG seed: PASS"
)

print(
    "Seed-1 experiment name: PASS"
)

print(
    "PCC-center: DISABLED — PASS"
)


# ============================================================
# 9. VERIFY FRESH INITIALIZATION
#
# Final Seed-1 output directory must NOT exist.
# This prevents accidental resume/reuse.
# ============================================================

if RUN_ROOT.exists():

    raise RuntimeError(
        "FINAL SEED-1 OUTPUT DIRECTORY ALREADY EXISTS:\n"
        f"{RUN_ROOT}\n\n"
        "Refusing to start because fresh initialization "
        "can no longer be guaranteed automatically.\n"
        "Do NOT delete anything manually. Paste this error."
    )


print()
print(
    "Fresh-initialization condition: PASS"
)

print(
    "Seed-1 output directory does not exist."
)


# ============================================================
# 10. VERIFY GPU RUNTIME
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA is unavailable."
    )


gpu_count = torch.cuda.device_count()


if gpu_count != 2:

    raise RuntimeError(
        f"Expected 2 GPUs; found {gpu_count}."
    )


gpu_names = [
    torch.cuda.get_device_name(
        i
    )
    for i in range(
        gpu_count
    )
]


if not all(
    "T4" in name
    for name in gpu_names
):

    raise RuntimeError(
        "Frozen runtime requires 2x Tesla T4.\n"
        f"Detected: {gpu_names}"
    )


print(
    "GPU runtime: 2x Tesla T4 — PASS"
)


# ============================================================
# 11. LAUNCHER INTEGRITY
# ============================================================

launcher_text = LAUNCHER.read_text(
    encoding="utf-8"
)


for marker in [
    "CHECKPOINTED ORIGINAL-PCC BASELINE LAUNCHER v2",
    "torch.amp.GradScaler",
    "snapshotting_save_checkpoint",
]:

    if marker not in launcher_text:

        raise RuntimeError(
            "Required launcher marker missing:\n"
            f"{marker}"
        )


print(
    "AMP32 + snapshot launcher integrity: PASS"
)


# ============================================================
# 12. FINAL PRE-TRAIN REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL SEED 1 TRAINING CONTRACT"
)

print(
    "============================================================"
)

print(
    "Seed index:              1 / 3"
)

print(
    "RNG seed:                1782656963"
)

print(
    "Fresh initialization:    YES"
)

print(
    "Resume checkpoint:       NONE"
)

print(
    "Training blocks:         11"
)

print(
    "Optimizer updates/block: 250"
)

print(
    "Final optimizer updates: 2750"
)

print(
    "Terminal epoch:          10"
)

print(
    "Native tile:             1024"
)

print(
    "World size:              2"
)

print(
    "Synthetic batch/GPU:     1"
)

print(
    "Anchor pair batch/GPU:   2"
)

print(
    "Gradient accumulation:   4"
)

print(
    "Effective batch:         8"
)

print(
    "Synthetic workers/GPU:   2"
)

print(
    "Anchor workers/GPU:      1"
)

print(
    "LR:                      3e-4"
)

print(
    "Warmup updates:          50"
)

print(
    "Post-warmup LR:          constant"
)

print(
    "AMP:                     FP16"
)

print(
    "AMP initial scale:       32"
)

print(
    "Original PCC:            RETAINED"
)

print(
    "CRPA:                    RETAINED"
)

print(
    "PCC-center:              DISABLED"
)

print(
    "Early stopping:          NO"
)

print(
    "Final model policy:      TERMINAL @ 2750"
)

print(
    "best.pt reporting:       PROHIBITED"
)

print(
    "External-data tuning:    NONE"
)

print()


# ============================================================
# 13. TRAIN FINAL SEED 1
#
# Fresh run:
#   --resume none
#
# 11 epochs/blocks * 1000 microsteps/rank
# accumulation 4
# 2 ranks
#
# -> 250 optimizer updates/block
# -> 2750 optimizer updates total
# ============================================================

cmd = [
    "torchrun",

    "--standalone",

    "--nproc_per_node=2",

    str(
        LAUNCHER
    ),

    "--project-root",
    str(
        PROJECT_ROOT
    ),

    "--config",
    str(
        CONFIG
    ),

    "--output-root",
    str(
        OUTPUT_ROOT
    ),

    "--epochs",
    str(
        FINAL_EPOCHS
    ),

    "--max-train-steps",
    "1000",

    "--max-val-steps",
    "100",

    "--accum",
    "4",

    "--per-rank-batch",
    "1",

    "--anchor-batch",
    "2",

    "--num-workers",
    "2",

    "--log-every",
    "100",

    "--resume",
    "none",
]


print(
    "============================================================"
)

print(
    "STARTING FINAL SEED 1"
)

print(
    "============================================================"
)

print(
    " ".join(
        cmd
    )
)

print()


training_start = time.time()


subprocess.run(
    cmd,
    check=True,
)


training_elapsed = (
    time.time()
    - training_start
)


# ============================================================
# 14. REQUIRED POST-TRAIN FILES
# ============================================================

for path, label in [
    (
        RUN_ROOT,
        "Seed-1 run directory",
    ),
    (
        SNAPSHOT_DIR,
        "Seed-1 snapshot directory",
    ),
    (
        TERMINAL_CHECKPOINT,
        "Seed-1 terminal checkpoint",
    ),
    (
        RUN_STATUS,
        "Seed-1 run status",
    ),
    (
        RESOLVED_CONFIG,
        "Seed-1 resolved config",
    ),
    (
        HISTORY,
        "Seed-1 training history",
    ),
]:

    require(
        path,
        label,
    )


# ============================================================
# 15. VERIFY RESOLVED CONFIG
# ============================================================

resolved = json.loads(
    RESOLVED_CONFIG.read_text(
        encoding="utf-8"
    )
)


if int(
    resolved[
        "seed"
    ]
) != FINAL_SEED:

    raise RuntimeError(
        "Resolved Seed-1 RNG seed mismatch."
    )


if (
    resolved[
        "experiment_name"
    ]
    != FINAL_EXPERIMENT
):

    raise RuntimeError(
        "Resolved experiment name mismatch."
    )


if resolved != cfg:

    raise RuntimeError(
        "Resolved config differs from frozen Seed-1 config."
    )


print()
print(
    "Resolved final Seed-1 config: PASS"
)


# ============================================================
# 16. VERIFY ALL 11 CHRONOLOGICAL SNAPSHOTS
# ============================================================

print()
print(
    "============================================================"
)

print(
    "VERIFYING FINAL SEED-1 SNAPSHOTS"
)

print(
    "============================================================"
)


snapshot_records = []


for epoch in range(
    11
):

    optimizer_step = (
        epoch + 1
    ) * 250

    filename = (
        f"epoch_{epoch:03d}"
        f"_opt_{optimizer_step:06d}.pt"
    )

    path = (
        SNAPSHOT_DIR
        / filename
    )

    require(
        path,
        (
            f"Seed-1 checkpoint "
            f"@ {optimizer_step} updates"
        ),
    )


    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )


    actual_epoch = int(
        payload.get(
            "epoch",
            -1,
        )
    )

    actual_step = int(
        payload.get(
            "optimizer_step",
            -1,
        )
    )


    if actual_epoch != epoch:

        raise RuntimeError(
            f"{filename}: "
            f"epoch={actual_epoch}; "
            f"expected={epoch}."
        )


    if actual_step != optimizer_step:

        raise RuntimeError(
            f"{filename}: "
            f"optimizer_step={actual_step}; "
            f"expected={optimizer_step}."
        )


    checkpoint_sha = sha256_file(
        path
    )


    snapshot_records.append(
        {
            "epoch":
                epoch,

            "optimizer_step":
                optimizer_step,

            "path":
                str(
                    path
                ),

            "sha256":
                checkpoint_sha,
        }
    )


    print(
        f"PASS  "
        f"epoch={epoch:02d}  "
        f"optimizer_step={optimizer_step:04d}  "
        f"sha={checkpoint_sha[:18]}..."
    )


# ============================================================
# 17. TERMINAL CHECKPOINT IDENTITY
# ============================================================

terminal_payload = torch.load(
    TERMINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)


terminal_step = int(
    terminal_payload.get(
        "optimizer_step",
        -1,
    )
)

terminal_epoch = int(
    terminal_payload.get(
        "epoch",
        -1,
    )
)


if (
    terminal_step
    != FINAL_OPTIMIZER_UPDATES
):

    raise RuntimeError(
        "FINAL SEED-1 TERMINAL STEP FAILURE.\n"
        f"Expected: {FINAL_OPTIMIZER_UPDATES}\n"
        f"Actual:   {terminal_step}"
    )


if (
    terminal_epoch
    != EXPECTED_TERMINAL_EPOCH
):

    raise RuntimeError(
        "FINAL SEED-1 TERMINAL EPOCH FAILURE.\n"
        f"Expected: {EXPECTED_TERMINAL_EPOCH}\n"
        f"Actual:   {terminal_epoch}"
    )


terminal_sha = sha256_file(
    TERMINAL_CHECKPOINT
)


print()
print(
    "Terminal Seed-1 checkpoint identity: PASS"
)

print(
    "  optimizer step:",
    terminal_step,
)

print(
    "  epoch:",
    terminal_epoch,
)

print(
    "  SHA256:"
)

print(
    " ",
    terminal_sha,
)


# ============================================================
# 18. AMP/RUN-STATUS AUDIT
# ============================================================

status = json.loads(
    RUN_STATUS.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if amp_skips != 0:

    raise RuntimeError(
        "Final Seed-1 AMP skip failure.\n"
        f"AMP skipped optimizer updates: {amp_skips}"
    )


print(
    "AMP skipped optimizer updates: 0 — PASS"
)


# ============================================================
# 19. WRITE SEED-1 FINAL TRAINING MANIFEST
# ============================================================

seed_manifest = {
    "experiment":
        FINAL_EXPERIMENT,

    "role":
        "FINAL_INDEPENDENT_SEED_1_OF_3",

    "seed_index":
        1,

    "rng_seed":
        FINAL_SEED,

    "fresh_initialization":
        True,

    "resume":
        "none",

    "frozen_optimizer_updates":
        FINAL_OPTIMIZER_UPDATES,

    "terminal_checkpoint_policy":
        "exactly_2750_optimizer_updates",

    "best_pt_used_for_final_reporting":
        False,

    "external_data_used_for_selection":
        False,

    "original_pcc":
        "retained",

    "crpa":
        "retained",

    "pcc_center":
        "rejected_and_disabled",

    "wall_time_minutes":
        training_elapsed / 60.0,

    "amp_skipped_optimizer_steps":
        amp_skips,

    "config": {
        "path":
            str(
                CONFIG
            ),

        "sha256":
            config_sha,
    },

    "terminal_checkpoint": {
        "path":
            str(
                TERMINAL_CHECKPOINT
            ),

        "epoch":
            terminal_epoch,

        "optimizer_step":
            terminal_step,

        "sha256":
            terminal_sha,
    },

    "snapshots":
        snapshot_records,

    "multi_seed_plan": {
        "path":
            str(
                PLAN_JSON
            ),

        "sha256":
            sha256_file(
                PLAN_JSON
            ),
    },

    "training_recipe_freeze_zip_sha256":
        EXPECTED_FREEZE_ZIP_SHA,
}


seed_manifest_path = (
    RUN_ROOT
    / "FINAL_SEED01_TRAINING_MANIFEST.json"
)


seed_manifest_path.write_text(
    json.dumps(
        seed_manifest,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 20. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL SEED 1 / 3 TRAINING COMPLETE"
)

print(
    "============================================================"
)

print(
    "RNG seed:               ",
    FINAL_SEED,
)

print(
    "Fresh initialization:    YES"
)

print(
    "Optimizer updates:       ",
    terminal_step,
)

print(
    "Terminal epoch:          ",
    terminal_epoch,
)

print(
    "AMP skipped updates:     ",
    amp_skips,
)

print(
    "Wall time:               "
    f"{training_elapsed / 60.0:.2f} min"
)

print()

print(
    "FINAL SEED-1 MODEL:"
)

print(
    TERMINAL_CHECKPOINT
)

print()

print(
    "SHA256:"
)

print(
    terminal_sha
)

print()

print(
    "Training manifest:"
)

print(
    seed_manifest_path
)

print()

print(
    "IMPORTANT:"
)

print(
    "best.pt is NOT the final Seed-1 model."
)

print(
    "The final Seed-1 model is the terminal "
    "2750-update checkpoint shown above."
)

print()

print(
    "========================================"
)

print(
    "DASO-NET FINAL SEED 1 TRAINING: PASS"
)

print(
    "========================================"
)

print(
    "STOP HERE."
)

print(
    "Do not launch Seed 2 yet."
)

print(
    "Paste the final output so Seed 1 can be "
    "QC-evaluated and recovery-packaged first."
)

DASO-Net — FINAL SEED 1 / 3 PREFLIGHT
PASS  Frozen project dataset
PASS  Frozen code tree
PASS  Frozen trainer
PASS  Checkpoint/snapshot launcher
PASS  Validated decoder
PASS  Frozen multi-seed plan
PASS  Final Seed-1 config
PASS  Final training-recipe freeze
PASS  Final recipe freeze ZIP

Training-recipe freeze SHA: PASS
Validated decoder SHA: PASS
Frozen multi-seed plan identity: PASS
Final Seed-1 config SHA: PASS
Seed-1 RNG seed: PASS
Seed-1 experiment name: PASS
PCC-center: DISABLED — PASS

Fresh-initialization condition: PASS
Seed-1 output directory does not exist.
GPU runtime: 2x Tesla T4 — PASS
AMP32 + snapshot launcher integrity: PASS

FINAL SEED 1 TRAINING CONTRACT
Seed index:              1 / 3
RNG seed:                1782656963
Fresh initialization:    YES
Resume checkpoint:       NONE
Training blocks:         11
Optimizer updates/block: 250
Final optimizer updates: 2750
Terminal epoch:          10
Native tile:             1024
World size:              2
Synthetic batch/GPU

W0914 09:07:28.622000 1126 torch/distributed/run.py:852] 
W0914 09:07:28.622000 1126 torch/distributed/run.py:852] *****************************************
W0914 09:07:28.622000 1126 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0914 09:07:28.622000 1126 torch/distributed/run.py:852] *****************************************
[W914 09:07:29.081355155 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files
DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files


[W914 09:07:31.658259288 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W914 09:07:31.664976624 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_final_seed01_s1782656963
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     1000
Optimizer steps total:      2750
Smoke mode:                 False
epoch=00 step=00000/01000 loss=1367.6277 inst=1367.0336 pcc=0.0012 crpa=0.5929 lr=6.00e-06 vram_alloc=8.84GB vram_res=9.11GB
epoch=00 step=00100/01000 loss=7.3656 inst=6.7455 pcc=0.0059 crpa=0.6142 lr=1.20e-04 vram_alloc=8.97GB vram_res=9.11GB
epoch=00 step=00200/01000 loss=0.4599 inst=0.1134 pcc=0.0000 crpa=0

RuntimeError: Seed-1 terminal checkpoint missing:
/kaggle/working/dasonet_phase5b/dasonet_final_seed01_s1782656963/snapshots/epoch_010_opt_002750.pt

In [15]:
# ============================================================
# DASO-Net — FINAL SEED 1 EXACT-UPDATE COMPLETION v2
#
# FIX IN v2:
#   resolved_config.json is NOT required to equal the frozen
#   source config byte-for-byte, because the trainer resolves
#   legitimate CLI runtime overrides into it.
#
# Instead we verify:
#   - frozen config SHA against multi-seed plan
#   - seed identity
#   - experiment identity
#   - model identity
#   - loss identity
#   - PCC-center disabled
#   - exact trusted 2744 checkpoint
#   - original run reported exactly 6 AMP skips
#
# Then:
#   2744 successful updates -> exactly 2750 successful updates
#
# If AMP skips again during completion, the loop requests only
# the remaining successful updates.
#
# NO scientific change.
# NO external data.
# NO tuning.
# ============================================================

from __future__ import annotations

import hashlib
import json
import subprocess
import time
from pathlib import Path

import torch


# ============================================================
# CONSTANTS
# ============================================================

TARGET_OPTIMIZER_STEP = 2750
ACCUM = 4

FINAL_SEED_INDEX = 1
FINAL_SEED = 1782656963

FINAL_EXPERIMENT = (
    "dasonet_final_seed01_s1782656963"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)

EXPECTED_RECIPE_FREEZE_SHA = (
    "b102deb8ec9de4be3816e4450a4dc7b2"
    "f78f5439634cee02fe4656d1a268bf3a"
)


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

PLAN_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-FinalMultiSeed-v1"
)

PLAN_JSON = (
    PLAN_ROOT
    / "FINAL_MULTISEED_PLAN_v1.json"
)

CONFIG = (
    PLAN_ROOT
    / "configs"
    / "final_seed01_s1782656963.json"
)

RECIPE_FREEZE_ZIP = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1.zip"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_final_seed01_s1782656963"
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

INITIAL_CHECKPOINT = (
    SNAPSHOT_DIR
    / "epoch_010_opt_002744.pt"
)

RUN_STATUS = (
    RUN_ROOT
    / "run_status.json"
)

RESOLVED_CONFIG = (
    RUN_ROOT
    / "resolved_config.json"
)

FINAL_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED01_TRAINING_MANIFEST.json"
)

COMPLETION_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED01_AMP_SKIP_COMPLETION.json"
)


# ============================================================
# HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def trusted_load(
    path: Path,
):

    return torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )


def recursive_diff(
    a,
    b,
    prefix="",
):

    diffs = []

    if isinstance(a, dict) and isinstance(b, dict):

        keys = sorted(
            set(a)
            | set(b)
        )

        for key in keys:

            path = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            if key not in a:

                diffs.append(
                    (
                        path,
                        "<MISSING>",
                        b[key],
                    )
                )

            elif key not in b:

                diffs.append(
                    (
                        path,
                        a[key],
                        "<MISSING>",
                    )
                )

            else:

                diffs.extend(
                    recursive_diff(
                        a[key],
                        b[key],
                        path,
                    )
                )

        return diffs


    if isinstance(a, list) and isinstance(b, list):

        if a != b:

            diffs.append(
                (
                    prefix,
                    a,
                    b,
                )
            )

        return diffs


    if a != b:

        diffs.append(
            (
                prefix,
                a,
                b,
            )
        )

    return diffs


def find_epoch_snapshots(
    epoch: int,
):

    return sorted(
        SNAPSHOT_DIR.glob(
            f"epoch_{epoch:03d}_opt_*.pt"
        )
    )


def newest_snapshot_for_epoch(
    epoch: int,
) -> Path:

    matches = (
        find_epoch_snapshots(
            epoch
        )
    )

    if not matches:

        raise RuntimeError(
            f"No snapshot found for epoch {epoch}."
        )

    # There should normally be exactly one.
    # If multiple exist, choose the one with the greatest
    # optimizer_step after reading checkpoint metadata.

    records = []

    for path in matches:

        payload = trusted_load(
            path
        )

        records.append(
            (
                int(
                    payload.get(
                        "optimizer_step",
                        -1,
                    )
                ),
                path,
            )
        )

    records.sort(
        key=lambda x: x[0]
    )

    return records[-1][1]


# ============================================================
# PRE-FLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FINAL SEED 1 EXACT-UPDATE COMPLETION v2"
)

print(
    "============================================================"
)


for path, label in [
    (
        PROJECT_ROOT,
        "Frozen project dataset",
    ),
    (
        LAUNCHER,
        "Checkpoint launcher",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
    (
        PLAN_JSON,
        "Frozen multi-seed plan",
    ),
    (
        CONFIG,
        "Final Seed-1 config",
    ),
    (
        RECIPE_FREEZE_ZIP,
        "Training-recipe freeze ZIP",
    ),
    (
        RUN_ROOT,
        "Existing Seed-1 run",
    ),
    (
        SNAPSHOT_DIR,
        "Seed-1 snapshots",
    ),
    (
        INITIAL_CHECKPOINT,
        "Seed-1 2744 checkpoint",
    ),
    (
        RUN_STATUS,
        "Seed-1 original run status",
    ),
    (
        RESOLVED_CONFIG,
        "Seed-1 resolved config",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# VERIFY FROZEN INFRASTRUCTURE
# ============================================================

actual_freeze_sha = sha256_file(
    RECIPE_FREEZE_ZIP
)


if (
    actual_freeze_sha
    != EXPECTED_RECIPE_FREEZE_SHA
):

    raise RuntimeError(
        "Training recipe freeze ZIP SHA mismatch.\n"
        f"Expected: {EXPECTED_RECIPE_FREEZE_SHA}\n"
        f"Actual:   {actual_freeze_sha}"
    )


actual_decoder_sha = sha256_file(
    DECODER
)


if (
    actual_decoder_sha
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Validated decoder SHA mismatch.\n"
        f"Expected: {EXPECTED_DECODER_SHA}\n"
        f"Actual:   {actual_decoder_sha}"
    )


print()
print(
    "Training recipe freeze SHA: PASS"
)

print(
    "Decoder SHA: PASS"
)


# ============================================================
# VERIFY MULTI-SEED PLAN + FROZEN CONFIG SHA
# ============================================================

plan = json.loads(
    PLAN_JSON.read_text(
        encoding="utf-8"
    )
)

cfg = json.loads(
    CONFIG.read_text(
        encoding="utf-8"
    )
)

resolved = json.loads(
    RESOLVED_CONFIG.read_text(
        encoding="utf-8"
    )
)


if (
    plan.get(
        "status"
    )
    != "FROZEN_BEFORE_FINAL_TRAINING"
):

    raise RuntimeError(
        "Multi-seed plan is not frozen."
    )


if int(
    plan[
        "frozen_training_recipe"
    ][
        "optimizer_updates"
    ]
) != TARGET_OPTIMIZER_STEP:

    raise RuntimeError(
        "Frozen multi-seed budget is not 2750."
    )


planned_entry = None


for item in plan[
    "final_seed_configs"
]:

    if int(
        item[
            "seed_index"
        ]
    ) == FINAL_SEED_INDEX:

        planned_entry = item
        break


if planned_entry is None:

    raise RuntimeError(
        "Final Seed-1 entry missing from frozen plan."
    )


if int(
    planned_entry[
        "seed"
    ]
) != FINAL_SEED:

    raise RuntimeError(
        "Frozen plan Seed-1 RNG mismatch."
    )


if (
    planned_entry[
        "experiment_name"
    ]
    != FINAL_EXPERIMENT
):

    raise RuntimeError(
        "Frozen plan Seed-1 experiment mismatch."
    )


actual_config_sha = sha256_file(
    CONFIG
)


if (
    actual_config_sha
    != planned_entry[
        "config_sha256"
    ]
):

    raise RuntimeError(
        "Final Seed-1 frozen config SHA mismatch.\n"
        f"Expected: {planned_entry['config_sha256']}\n"
        f"Actual:   {actual_config_sha}"
    )


print()
print(
    "Frozen multi-seed plan identity: PASS"
)

print(
    "Frozen Seed-1 config SHA: PASS"
)


# ============================================================
# VERIFY SCIENTIFIC CONFIG IDENTITY
#
# resolved_config may legitimately differ in runtime training
# fields due CLI overrides.
#
# Scientific model/loss identity MUST remain unchanged.
# ============================================================

if int(
    cfg[
        "seed"
    ]
) != FINAL_SEED:

    raise RuntimeError(
        "Frozen config RNG seed mismatch."
    )


if int(
    resolved[
        "seed"
    ]
) != FINAL_SEED:

    raise RuntimeError(
        "Resolved config RNG seed mismatch."
    )


if (
    cfg[
        "experiment_name"
    ]
    != FINAL_EXPERIMENT
):

    raise RuntimeError(
        "Frozen config experiment mismatch."
    )


if (
    resolved[
        "experiment_name"
    ]
    != FINAL_EXPERIMENT
):

    raise RuntimeError(
        "Resolved config experiment mismatch."
    )


if (
    cfg.get(
        "model"
    )
    != resolved.get(
        "model"
    )
):

    raise RuntimeError(
        "CRITICAL: resolved model configuration differs "
        "from the frozen Seed-1 model configuration."
    )


if (
    cfg.get(
        "loss"
    )
    != resolved.get(
        "loss"
    )
):

    raise RuntimeError(
        "CRITICAL: resolved loss configuration differs "
        "from the frozen Seed-1 loss configuration."
    )


pcc_center = (
    cfg
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if (
    pcc_center is not None
    and bool(
        pcc_center.get(
            "enabled",
            False,
        )
    )
):

    raise RuntimeError(
        "PCC-center unexpectedly enabled."
    )


resolved_pcc_center = (
    resolved
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if (
    resolved_pcc_center is not None
    and bool(
        resolved_pcc_center.get(
            "enabled",
            False,
        )
    )
):

    raise RuntimeError(
        "Resolved PCC-center unexpectedly enabled."
    )


print(
    "Seed identity: PASS"
)

print(
    "Experiment identity: PASS"
)

print(
    "Model configuration identity: PASS"
)

print(
    "Loss configuration identity: PASS"
)

print(
    "PCC-center disabled: PASS"
)


# ============================================================
# REPORT LEGITIMATE CONFIG DIFFERENCES
#
# This is provenance only.
# It does NOT require cfg == resolved.
# ============================================================

config_differences = recursive_diff(
    cfg,
    resolved,
)


print()
print(
    "============================================================"
)

print(
    "FROZEN CONFIG vs RESOLVED RUNTIME CONFIG"
)

print(
    "============================================================"
)


if not config_differences:

    print(
        "No differences."
    )

else:

    print(
        f"Leaf differences found: "
        f"{len(config_differences)}"
    )

    print(
        "These may reflect legitimate CLI runtime overrides."
    )

    print()

    for (
        path,
        frozen_value,
        resolved_value,
    ) in config_differences:

        print(
            f"{path}:"
        )

        print(
            f"  frozen   = {frozen_value!r}"
        )

        print(
            f"  resolved = {resolved_value!r}"
        )


# ============================================================
# VERIFY ORIGINAL 2744 STATE
# ============================================================

initial_payload = trusted_load(
    INITIAL_CHECKPOINT
)


initial_step = int(
    initial_payload.get(
        "optimizer_step",
        -1,
    )
)

initial_epoch = int(
    initial_payload.get(
        "epoch",
        -1,
    )
)


if initial_step != 2744:

    raise RuntimeError(
        "Expected original Seed-1 checkpoint at "
        f"optimizer_step=2744; found {initial_step}."
    )


if initial_epoch != 10:

    raise RuntimeError(
        "Expected original Seed-1 checkpoint epoch=10; "
        f"found {initial_epoch}."
    )


original_shortfall = (
    TARGET_OPTIMIZER_STEP
    - initial_step
)


if original_shortfall != 6:

    raise RuntimeError(
        f"Expected a 6-update shortfall; "
        f"found {original_shortfall}."
    )


initial_sha = sha256_file(
    INITIAL_CHECKPOINT
)


original_status = json.loads(
    RUN_STATUS.read_text(
        encoding="utf-8"
    )
)


reported_original_amp_skips = int(
    original_status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if reported_original_amp_skips != 6:

    raise RuntimeError(
        "Expected original run_status to report "
        f"6 AMP skips; got {reported_original_amp_skips}."
    )


print()
print(
    "Original Seed-1 state: PASS"
)

print(
    "  successful optimizer updates:",
    initial_step,
)

print(
    "  original AMP skipped updates:",
    reported_original_amp_skips,
)

print(
    "  missing successful updates:",
    original_shortfall,
)

print(
    "  2744 checkpoint SHA:"
)

print(
    " ",
    initial_sha,
)


# ============================================================
# GPU AUDIT
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA unavailable."
    )


if torch.cuda.device_count() != 2:

    raise RuntimeError(
        f"Expected exactly 2 GPUs; "
        f"found {torch.cuda.device_count()}."
    )


gpu_names = [
    torch.cuda.get_device_name(
        i
    )
    for i in range(2)
]


if not all(
    "T4" in name
    for name in gpu_names
):

    raise RuntimeError(
        f"Expected 2x Tesla T4; got {gpu_names}"
    )


print()
print(
    "GPU runtime: 2x Tesla T4 — PASS"
)


# ============================================================
# EXACT SUCCESSFUL-UPDATE COMPLETION LOOP
# ============================================================

current_checkpoint = (
    INITIAL_CHECKPOINT
)

current_step = (
    initial_step
)

current_epoch = (
    initial_epoch
)

completion_iterations = []

completion_start = time.time()

MAX_COMPLETION_ITERATIONS = 10


while (
    current_step
    < TARGET_OPTIMIZER_STEP
):

    if (
        len(
            completion_iterations
        )
        >= MAX_COMPLETION_ITERATIONS
    ):

        raise RuntimeError(
            "Exceeded maximum completion iterations.\n"
            "STOP and inspect."
        )


    missing_updates = (
        TARGET_OPTIMIZER_STEP
        - current_step
    )


    # Each successful optimizer update requires exactly
    # ACCUM microsteps in this frozen setup.
    requested_microsteps = (
        missing_updates
        * ACCUM
    )


    next_epoch = (
        current_epoch
        + 1
    )


    epochs_cli = (
        next_epoch
        + 1
    )


    print()
    print(
        "============================================================"
    )

    print(
        f"COMPLETION ITERATION "
        f"{len(completion_iterations) + 1}"
    )

    print(
        "============================================================"
    )

    print(
        "Current successful optimizer step:",
        current_step,
    )

    print(
        "Remaining successful updates:",
        missing_updates,
    )

    print(
        "Requested train microsteps/rank:",
        requested_microsteps,
    )

    print(
        "Gradient accumulation:",
        ACCUM,
    )

    print(
        "Maximum new successful updates:",
        missing_updates,
    )

    print(
        "Resume checkpoint:"
    )

    print(
        current_checkpoint
    )

    print()


    cmd = [
        "torchrun",

        "--standalone",

        "--nproc_per_node=2",

        str(
            LAUNCHER
        ),

        "--project-root",
        str(
            PROJECT_ROOT
        ),

        "--config",
        str(
            CONFIG
        ),

        "--output-root",
        "/kaggle/working",

        "--epochs",
        str(
            epochs_cli
        ),

        "--max-train-steps",
        str(
            requested_microsteps
        ),

        "--max-val-steps",
        "100",

        "--accum",
        str(
            ACCUM
        ),

        "--per-rank-batch",
        "1",

        "--anchor-batch",
        "2",

        "--num-workers",
        "2",

        "--log-every",
        str(
            max(
                1,
                requested_microsteps,
            )
        ),

        "--resume",
        str(
            current_checkpoint
        ),
    ]


    print(
        " ".join(
            cmd
        )
    )

    print()


    iteration_start = time.time()


    subprocess.run(
        cmd,
        check=True,
    )


    iteration_elapsed = (
        time.time()
        - iteration_start
    )


    new_checkpoint = (
        newest_snapshot_for_epoch(
            next_epoch
        )
    )


    new_payload = trusted_load(
        new_checkpoint
    )


    new_step = int(
        new_payload.get(
            "optimizer_step",
            -1,
        )
    )

    new_epoch_actual = int(
        new_payload.get(
            "epoch",
            -1,
        )
    )


    if new_epoch_actual != next_epoch:

        raise RuntimeError(
            "Completion snapshot epoch mismatch."
        )


    if new_step <= current_step:

        raise RuntimeError(
            "Completion iteration made no successful "
            "optimizer progress."
        )


    if new_step > TARGET_OPTIMIZER_STEP:

        raise RuntimeError(
            "CRITICAL: completion overshot frozen target.\n"
            f"Observed optimizer_step={new_step}"
        )


    successful_updates = (
        new_step
        - current_step
    )


    inferred_additional_skips = (
        missing_updates
        - successful_updates
    )


    if inferred_additional_skips < 0:

        raise RuntimeError(
            "Invalid inferred AMP skip count."
        )


    # --------------------------------------------------------
    # Verify constant frozen LR from optimizer state.
    # --------------------------------------------------------

    optimizer_state = (
        new_payload.get(
            "optimizer_state_dict",
            {}
        )
    )

    param_groups = (
        optimizer_state.get(
            "param_groups",
            []
        )
    )


    lr_values = [
        float(
            group[
                "lr"
            ]
        )
        for group in param_groups
        if "lr" in group
    ]


    if not lr_values:

        raise RuntimeError(
            "Could not read LR from optimizer checkpoint."
        )


    for lr in lr_values:

        if abs(
            lr
            - 3e-4
        ) > 1e-10:

            raise RuntimeError(
                "Completion LR changed unexpectedly.\n"
                f"Observed LR={lr}"
            )


    iteration_record = {
        "iteration":
            len(
                completion_iterations
            )
            + 1,

        "start_optimizer_step":
            current_step,

        "requested_missing_updates":
            missing_updates,

        "requested_microsteps":
            requested_microsteps,

        "successful_updates":
            successful_updates,

        "inferred_additional_amp_skips":
            inferred_additional_skips,

        "end_optimizer_step":
            new_step,

        "epoch":
            new_epoch_actual,

        "checkpoint":
            str(
                new_checkpoint
            ),

        "checkpoint_sha256":
            sha256_file(
                new_checkpoint
            ),

        "learning_rates":
            lr_values,

        "wall_time_minutes":
            iteration_elapsed
            / 60.0,
    }


    completion_iterations.append(
        iteration_record
    )


    print()
    print(
        "Completion iteration result:"
    )

    print(
        "  successful updates:",
        successful_updates,
    )

    print(
        "  additional AMP skips:",
        inferred_additional_skips,
    )

    print(
        "  new optimizer step:",
        new_step,
    )

    print(
        "  checkpoint:"
    )

    print(
        " ",
        new_checkpoint
    )

    print(
        "  LR:",
        lr_values,
    )


    current_checkpoint = (
        new_checkpoint
    )

    current_step = (
        new_step
    )

    current_epoch = (
        new_epoch_actual
    )


# ============================================================
# VERIFY EXACT TARGET
# ============================================================

if current_step != TARGET_OPTIMIZER_STEP:

    raise RuntimeError(
        "Exact final target 2750 was not reached."
    )


completion_elapsed = (
    time.time()
    - completion_start
)


final_checkpoint = (
    current_checkpoint
)

final_sha = sha256_file(
    final_checkpoint
)


additional_amp_skips = sum(
    int(
        item[
            "inferred_additional_amp_skips"
        ]
    )
    for item in completion_iterations
)


total_amp_skips_observed = (
    reported_original_amp_skips
    + additional_amp_skips
)


# ============================================================
# AUDIT COMPLETE SNAPSHOT CHAIN
# ============================================================

print()
print(
    "============================================================"
)

print(
    "AUDITING FINAL SEED-1 SNAPSHOT CHAIN"
)

print(
    "============================================================"
)


all_snapshots = sorted(
    SNAPSHOT_DIR.glob(
        "epoch_*_opt_*.pt"
    )
)


snapshot_records = []


for path in all_snapshots:

    payload = trusted_load(
        path
    )

    epoch = int(
        payload.get(
            "epoch",
            -1,
        )
    )

    step = int(
        payload.get(
            "optimizer_step",
            -1,
        )
    )


    if step > TARGET_OPTIMIZER_STEP:

        raise RuntimeError(
            "Found Seed-1 snapshot beyond frozen "
            f"2750-update target:\n{path}"
        )


    snapshot_records.append(
        {
            "epoch":
                epoch,

            "optimizer_step":
                step,

            "path":
                str(
                    path
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


snapshot_records.sort(
    key=lambda x: (
        x[
            "epoch"
        ],
        x[
            "optimizer_step"
        ],
    )
)


previous_epoch = -1
previous_step = -1


for record in snapshot_records:

    epoch = int(
        record[
            "epoch"
        ]
    )

    step = int(
        record[
            "optimizer_step"
        ]
    )


    if epoch <= previous_epoch:

        raise RuntimeError(
            "Seed-1 snapshot epochs are not "
            "strictly increasing."
        )


    if step <= previous_step:

        raise RuntimeError(
            "Seed-1 optimizer steps are not "
            "strictly increasing."
        )


    print(
        f"PASS  "
        f"epoch={epoch:02d}  "
        f"optimizer_step={step:04d}  "
        f"sha={record['sha256'][:18]}..."
    )


    previous_epoch = epoch
    previous_step = step


if (
    snapshot_records[
        -1
    ][
        "optimizer_step"
    ]
    != TARGET_OPTIMIZER_STEP
):

    raise RuntimeError(
        "Final snapshot is not optimizer step 2750."
    )


# ============================================================
# COMPLETION MANIFEST
# ============================================================

completion_manifest = {
    "experiment":
        FINAL_EXPERIMENT,

    "seed_index":
        FINAL_SEED_INDEX,

    "rng_seed":
        FINAL_SEED,

    "scientific_change":
        False,

    "reason":
        (
            "The fresh Final Seed-1 run encountered six "
            "AMP-overflow skipped optimizer updates during "
            "its initial training block. Consequently the "
            "11-block run ended at 2744 successful optimizer "
            "updates. A bounded trusted resume was performed "
            "solely to reach the already-frozen budget of "
            "exactly 2750 successful optimizer updates."
        ),

    "frozen_target_optimizer_updates":
        TARGET_OPTIMIZER_STEP,

    "original_terminal_optimizer_step":
        initial_step,

    "original_terminal_epoch":
        initial_epoch,

    "original_checkpoint":
        str(
            INITIAL_CHECKPOINT
        ),

    "original_checkpoint_sha256":
        initial_sha,

    "original_amp_skips":
        reported_original_amp_skips,

    "completion_iterations":
        completion_iterations,

    "additional_amp_skips_during_completion":
        additional_amp_skips,

    "total_amp_skips_observed":
        total_amp_skips_observed,

    "final_optimizer_step":
        current_step,

    "final_epoch":
        current_epoch,

    "final_checkpoint":
        str(
            final_checkpoint
        ),

    "final_checkpoint_sha256":
        final_sha,

    "learning_rate_during_completion":
        3e-4,

    "architecture_changed":
        False,

    "loss_changed":
        False,

    "pcc_changed":
        False,

    "crpa_changed":
        False,

    "pcc_center":
        "rejected_and_disabled",

    "external_data_used":
        False,

    "config_runtime_differences":
        [
            {
                "path":
                    path,

                "frozen_value":
                    frozen_value,

                "resolved_value":
                    resolved_value,
            }

            for (
                path,
                frozen_value,
                resolved_value,
            )
            in config_differences
        ],

    "completion_wall_time_minutes":
        completion_elapsed
        / 60.0,
}


COMPLETION_MANIFEST.write_text(
    json.dumps(
        completion_manifest,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


# ============================================================
# FINAL SEED-1 TRAINING MANIFEST
# ============================================================

final_manifest = {
    "experiment":
        FINAL_EXPERIMENT,

    "role":
        "FINAL_INDEPENDENT_SEED_1_OF_3",

    "seed_index":
        FINAL_SEED_INDEX,

    "rng_seed":
        FINAL_SEED,

    "fresh_initialization":
        True,

    "initial_training_resume":
        "none",

    "frozen_optimizer_updates":
        TARGET_OPTIMIZER_STEP,

    "successful_optimizer_updates":
        current_step,

    "amp_skip_compensation_required":
        True,

    "original_amp_skips":
        reported_original_amp_skips,

    "additional_completion_amp_skips":
        additional_amp_skips,

    "total_amp_skips_observed":
        total_amp_skips_observed,

    "terminal_checkpoint_policy":
        "exactly_2750_successful_optimizer_updates",

    "best_pt_used_for_final_reporting":
        False,

    "external_data_used_for_selection":
        False,

    "original_pcc":
        "retained",

    "crpa":
        "retained",

    "pcc_center":
        "rejected_and_disabled",

    "config": {
        "path":
            str(
                CONFIG
            ),

        "sha256":
            actual_config_sha,
    },

    "resolved_config": {
        "path":
            str(
                RESOLVED_CONFIG
            ),

        "sha256":
            sha256_file(
                RESOLVED_CONFIG
            ),
    },

    "terminal_checkpoint": {
        "path":
            str(
                final_checkpoint
            ),

        "epoch":
            current_epoch,

        "optimizer_step":
            current_step,

        "sha256":
            final_sha,
    },

    "snapshots":
        snapshot_records,

    "completion_manifest": {
        "path":
            str(
                COMPLETION_MANIFEST
            ),

        "sha256":
            sha256_file(
                COMPLETION_MANIFEST
            ),
    },

    "multi_seed_plan": {
        "path":
            str(
                PLAN_JSON
            ),

        "sha256":
            sha256_file(
                PLAN_JSON
            ),
    },

    "training_recipe_freeze_zip_sha256":
        EXPECTED_RECIPE_FREEZE_SHA,
}


FINAL_MANIFEST.write_text(
    json.dumps(
        final_manifest,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL SEED 1 EXACT-UPDATE COMPLETION"
)

print(
    "============================================================"
)

print(
    "Original successful updates: ",
    initial_step,
)

print(
    "Original AMP skips:          ",
    reported_original_amp_skips,
)

print(
    "Additional AMP skips:        ",
    additional_amp_skips,
)

print(
    "Total AMP skips observed:    ",
    total_amp_skips_observed,
)

print(
    "FINAL successful updates:    ",
    current_step,
)

print(
    "Final checkpoint epoch:      ",
    current_epoch,
)

print(
    "Completion iterations:       ",
    len(
        completion_iterations
    ),
)

print(
    "Completion wall time:        "
    f"{completion_elapsed / 60.0:.2f} min"
)

print()

print(
    "FINAL SEED-1 TERMINAL MODEL:"
)

print(
    final_checkpoint
)

print()

print(
    "SHA256:"
)

print(
    final_sha
)

print()

print(
    "Completion manifest:"
)

print(
    COMPLETION_MANIFEST
)

print()

print(
    "Final training manifest:"
)

print(
    FINAL_MANIFEST
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FINAL SEED 1 EXACT 2750 UPDATES: PASS"
)

print(
    "============================================================"
)

print(
    "STOP HERE."
)

print(
    "Do NOT start Seed 2."
)

print(
    "Next: Seed-1 QC, recovery package, then EOD handoff."
)

DASO-Net — FINAL SEED 1 EXACT-UPDATE COMPLETION v2
PASS  Frozen project dataset
PASS  Checkpoint launcher
PASS  Validated decoder
PASS  Frozen multi-seed plan
PASS  Final Seed-1 config
PASS  Training-recipe freeze ZIP
PASS  Existing Seed-1 run
PASS  Seed-1 snapshots
PASS  Seed-1 2744 checkpoint
PASS  Seed-1 original run status
PASS  Seed-1 resolved config

Training recipe freeze SHA: PASS
Decoder SHA: PASS

Frozen multi-seed plan identity: PASS
Frozen Seed-1 config SHA: PASS
Seed identity: PASS
Experiment identity: PASS
Model configuration identity: PASS
Loss configuration identity: PASS
PCC-center disabled: PASS

FROZEN CONFIG vs RESOLVED RUNTIME CONFIG
Leaf differences found: 2
These may reflect legitimate CLI runtime overrides.

optimization.epochs:
  frozen   = 20
  resolved = 11
runtime:
  frozen   = '<MISSING>'
  resolved = {'distributed': True, 'world_size': 2, 'max_train_steps_per_epoch_per_rank': 1000, 'max_val_steps_per_epoch_per_rank': 100, 'torch_version': '2.10.0+cu128', '

W0914 12:51:29.715000 1307 torch/distributed/run.py:852] 
W0914 12:51:29.715000 1307 torch/distributed/run.py:852] *****************************************
W0914 12:51:29.715000 1307 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0914 12:51:29.715000 1307 torch/distributed/run.py:852] *****************************************
[W914 12:51:29.911983807 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files
DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files


[W914 12:51:32.490676714 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W914 12:51:32.551360901 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


Resumed from: /kaggle/working/dasonet_phase5b/dasonet_final_seed01_s1782656963/snapshots/epoch_010_opt_002744.pt
=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_final_seed01_s1782656963
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     24
Optimizer steps total:      72
Smoke mode:                 False
epoch=11 step=00000/00024 loss=1.7873 inst=1.5998 pcc=0.0000 crpa=0.1875 lr=3.00e-04 vram_alloc=8.85GB vram_res=9.12GB
epoch=11 step=00023/00024 loss=1.4822 inst=1.4152 pcc=0.0001 crpa=0.0669 lr=3.00e-0

In [16]:
# ============================================================
# DASO-Net — FINAL SEED 1 / 3 QC
#
# NO TRAINING.
# NO EXTERNAL DATA.
# NO MODEL SELECTION.
#
# Fixed QC:
#   - verify exact 2750 terminal checkpoint
#   - verify manifests/config/frozen plan
#   - 1000-scene synthetic-val metrics
#   - 1000-scene oracle component diagnostic
#   - 1000-scene counterfactual diagnostic
#   - save formal Seed-1 QC summary
#
# Seed 1 is NOT accepted/rejected based on being better/worse
# than the development seed. This is integrity/sanity QC only.
# ============================================================

from __future__ import annotations

import hashlib
import json
import math
import subprocess
import sys
import time
from pathlib import Path


# ============================================================
# CONSTANTS
# ============================================================

SEED_INDEX = 1
RNG_SEED = 1782656963

EXPERIMENT = (
    "dasonet_final_seed01_s1782656963"
)

TARGET_STEP = 2750

EXPECTED_TERMINAL_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe79"
    "6c9cc301bd6fcc7c596fc290ad3a7e81d"
)

VAL_SCENES = 1000
VAL_SEED = 20260917


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_final_seed01_s1782656963"
)

TERMINAL_CHECKPOINT = (
    RUN_ROOT
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

TRAINING_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED01_TRAINING_MANIFEST.json"
)

COMPLETION_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED01_AMP_SKIP_COMPLETION.json"
)

PLAN_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-FinalMultiSeed-v1"
)

PLAN_JSON = (
    PLAN_ROOT
    / "FINAL_MULTISEED_PLAN_v1.json"
)

CONFIG = (
    PLAN_ROOT
    / "configs"
    / "final_seed01_s1782656963.json"
)

QC_ROOT = Path(
    "/kaggle/working/"
    "dasonet_final_qc/"
    "seed01_s1782656963_val1000_v1"
)

METRIC_DIR = (
    QC_ROOT
    / "metrics"
)

ORACLE_DIR = (
    QC_ROOT
    / "oracle"
)

COUNTERFACTUAL_DIR = (
    QC_ROOT
    / "counterfactual"
)

METRIC_JSON = (
    METRIC_DIR
    / "phase5c_metric_summary.json"
)

ORACLE_JSON = (
    ORACLE_DIR
    / "oracle_component_diagnostic.json"
)

COUNTERFACTUAL_JSON = (
    COUNTERFACTUAL_DIR
    / "counterfactual_center_diagnostic.json"
)

QC_SUMMARY = (
    QC_ROOT
    / "FINAL_SEED01_QC_SUMMARY.json"
)


# ============================================================
# HELPERS
# ============================================================

def require(path: Path, label: str) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(path: Path) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def load_json(path: Path):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def finite_number(value, label):

    value = float(value)

    if not math.isfinite(value):

        raise RuntimeError(
            f"Non-finite QC value: {label}={value}"
        )

    return value


# ============================================================
# INTEGRITY PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FINAL SEED 1 QC"
)

print(
    "============================================================"
)


for path, label in [
    (
        PROJECT_ROOT,
        "Frozen project dataset",
    ),
    (
        EVALUATOR,
        "Evaluator",
    ),
    (
        ORACLE,
        "Oracle diagnostic",
    ),
    (
        COUNTERFACTUAL,
        "Counterfactual diagnostic",
    ),
    (
        TERMINAL_CHECKPOINT,
        "Final Seed-1 terminal checkpoint",
    ),
    (
        TRAINING_MANIFEST,
        "Final Seed-1 training manifest",
    ),
    (
        COMPLETION_MANIFEST,
        "AMP-skip completion manifest",
    ),
    (
        PLAN_JSON,
        "Frozen multi-seed plan",
    ),
    (
        CONFIG,
        "Frozen Seed-1 config",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


terminal_sha = sha256_file(
    TERMINAL_CHECKPOINT
)


if terminal_sha != EXPECTED_TERMINAL_SHA:

    raise RuntimeError(
        "Terminal Seed-1 checkpoint SHA mismatch.\n"
        f"Expected: {EXPECTED_TERMINAL_SHA}\n"
        f"Actual:   {terminal_sha}"
    )


training_manifest = load_json(
    TRAINING_MANIFEST
)

completion_manifest = load_json(
    COMPLETION_MANIFEST
)

plan = load_json(
    PLAN_JSON
)

cfg = load_json(
    CONFIG
)


if int(
    training_manifest[
        "successful_optimizer_updates"
    ]
) != TARGET_STEP:

    raise RuntimeError(
        "Training manifest is not at 2750 updates."
    )


if int(
    completion_manifest[
        "final_optimizer_step"
    ]
) != TARGET_STEP:

    raise RuntimeError(
        "Completion manifest is not at 2750 updates."
    )


if (
    training_manifest[
        "terminal_checkpoint"
    ][
        "sha256"
    ]
    != EXPECTED_TERMINAL_SHA
):

    raise RuntimeError(
        "Training-manifest terminal SHA mismatch."
    )


if int(
    cfg[
        "seed"
    ]
) != RNG_SEED:

    raise RuntimeError(
        "Seed-1 config RNG mismatch."
    )


if (
    cfg[
        "experiment_name"
    ]
    != EXPERIMENT
):

    raise RuntimeError(
        "Seed-1 experiment identity mismatch."
    )


if int(
    plan[
        "frozen_training_recipe"
    ][
        "optimizer_updates"
    ]
) != TARGET_STEP:

    raise RuntimeError(
        "Frozen plan budget mismatch."
    )


print()
print(
    "Terminal checkpoint SHA: PASS"
)

print(
    "Exact successful updates: 2750 — PASS"
)

print(
    "Original AMP skips documented:",
    completion_manifest[
        "original_amp_skips"
    ],
)

print(
    "Additional completion skips:",
    completion_manifest[
        "additional_amp_skips_during_completion"
    ],
)

print(
    "External data used: NONE"
)


# ============================================================
# METRIC EVALUATION — 1000 SYNTHETIC VAL SCENES
# ============================================================

QC_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


qc_start = time.time()


if not METRIC_JSON.exists():

    print()
    print(
        "============================================================"
    )

    print(
        "SEED-1 QC — 1000-SCENE METRIC EVALUATION"
    )

    print(
        "============================================================"
    )


    subprocess.run(
        [
            sys.executable,
            str(
                EVALUATOR
            ),

            "--project-root",
            str(
                PROJECT_ROOT
            ),

            "--checkpoint",
            str(
                TERMINAL_CHECKPOINT
            ),

            "--output-dir",
            str(
                METRIC_DIR
            ),

            "--split",
            "val",

            "--max-scenes",
            str(
                VAL_SCENES
            ),

            "--seed",
            str(
                VAL_SEED
            ),

            "--num-workers",
            "2",

            "--decode-min-score",
            "0.001",

            "--operating-score",
            "0.05",

            "--topk",
            "100",

            "--nms-iou",
            "0.5",

            "--mask-threshold",
            "0.5",

            "--match-iou",
            "0.5",
        ],
        check=True,
    )

else:

    print(
        "Existing Seed-1 metric QC JSON found; reusing."
    )


# ============================================================
# ORACLE — 1000 SCENES
# ============================================================

if not ORACLE_JSON.exists():

    print()
    print(
        "============================================================"
    )

    print(
        "SEED-1 QC — 1000-SCENE ORACLE"
    )

    print(
        "============================================================"
    )


    subprocess.run(
        [
            sys.executable,
            str(
                ORACLE
            ),

            "--project-root",
            str(
                PROJECT_ROOT
            ),

            "--checkpoint",
            str(
                TERMINAL_CHECKPOINT
            ),

            "--evaluator",
            str(
                EVALUATOR
            ),

            "--output-dir",
            str(
                ORACLE_DIR
            ),

            "--split",
            "val",

            "--max-scenes",
            str(
                VAL_SCENES
            ),

            "--seed",
            str(
                VAL_SEED
            ),

            "--num-workers",
            "2",

            "--stride",
            "4",

            "--peak-kernel",
            "3",

            "--topk",
            "100",

            "--mask-threshold",
            "0.5",
        ],
        check=True,
    )

else:

    print(
        "Existing Seed-1 oracle QC JSON found; reusing."
    )


# ============================================================
# COUNTERFACTUAL — 1000 SCENES
# ============================================================

if not COUNTERFACTUAL_JSON.exists():

    print()
    print(
        "============================================================"
    )

    print(
        "SEED-1 QC — 1000-SCENE COUNTERFACTUAL"
    )

    print(
        "============================================================"
    )


    subprocess.run(
        [
            sys.executable,
            str(
                COUNTERFACTUAL
            ),

            "--project-root",
            str(
                PROJECT_ROOT
            ),

            "--checkpoint",
            str(
                TERMINAL_CHECKPOINT
            ),

            "--evaluator",
            str(
                EVALUATOR
            ),

            "--output-dir",
            str(
                COUNTERFACTUAL_DIR
            ),

            "--split",
            "val",

            "--max-scenes",
            str(
                VAL_SCENES
            ),

            "--seed",
            str(
                VAL_SEED
            ),

            "--num-workers",
            "2",

            "--stride",
            "4",

            "--peak-kernel",
            "3",

            "--window-radius-feature",
            "2",
        ],
        check=True,
    )

else:

    print(
        "Existing Seed-1 counterfactual QC JSON found; reusing."
    )


# ============================================================
# LOAD RESULTS
# ============================================================

for path, label in [
    (
        METRIC_JSON,
        "Metric QC JSON",
    ),
    (
        ORACLE_JSON,
        "Oracle QC JSON",
    ),
    (
        COUNTERFACTUAL_JSON,
        "Counterfactual QC JSON",
    ),
]:

    require(
        path,
        label,
    )


metric = load_json(
    METRIC_JSON
)

oracle = load_json(
    ORACLE_JSON
)

cf = load_json(
    COUNTERFACTUAL_JSON
)


ap = metric[
    "mask_ap"
]

op = metric[
    "operating_point"
]

sweep = metric[
    "validation_threshold_sweep"
]

center = oracle[
    "center_head"
]

geometry = oracle[
    "oracle_gt_cell_geometry"
]

masks = oracle[
    "oracle_gt_cell_mask"
]

empty = oracle[
    "empty_scene_center_behavior"
]


results = {
    "AP50":
        finite_number(
            ap[
                "mask_ap50"
            ],
            "AP50",
        ),

    "AP75":
        finite_number(
            ap[
                "mask_ap75"
            ],
            "AP75",
        ),

    "AP50_95":
        finite_number(
            ap[
                "mask_ap50_95"
            ],
            "AP50_95",
        ),

    "BestF1":
        finite_number(
            sweep[
                "best_f1"
            ],
            "BestF1",
        ),

    "BestThreshold":
        finite_number(
            sweep[
                "best_threshold_by_f1"
            ],
            "BestThreshold",
        ),

    "OperatingPrecision005":
        finite_number(
            op[
                "precision"
            ],
            "OperatingPrecision005",
        ),

    "OperatingRecall005":
        finite_number(
            op[
                "recall"
            ],
            "OperatingRecall005",
        ),

    "CountMAE005":
        finite_number(
            op[
                "count_mae"
            ],
            "CountMAE005",
        ),

    "EmptyFP005":
        finite_number(
            op[
                "empty_scene_fp_per_image"
            ],
            "EmptyFP005",
        ),

    "GTCenterScore":
        finite_number(
            center[
                "gt_cell_score_stats"
            ][
                "mean"
            ],
            "GTCenterScore",
        ),

    "PeakRecall4":
        finite_number(
            center[
                "top100_peak_recall_within_4px"
            ],
            "PeakRecall4",
        ),

    "PeakRecall8":
        finite_number(
            center[
                "top100_peak_recall_within_8px"
            ],
            "PeakRecall8",
        ),

    "PeakRecall16":
        finite_number(
            center[
                "top100_peak_recall_within_16px"
            ],
            "PeakRecall16",
        ),

    "BoxIoU":
        finite_number(
            geometry[
                "box_iou_stats"
            ][
                "mean"
            ],
            "BoxIoU",
        ),

    "GTBoxMaskIoU":
        finite_number(
            masks[
                "gt_box_crop_mask_iou_stats"
            ][
                "mean"
            ],
            "GTBoxMaskIoU",
        ),

    "GTBoxMaskIoU50":
        finite_number(
            masks[
                "gt_box_crop_mask_iou_ge_0_50"
            ],
            "GTBoxMaskIoU50",
        ),

    "EmptyMaxPeak":
        finite_number(
            empty[
                "max_peak_score_stats"
            ][
                "mean"
            ],
            "EmptyMaxPeak",
        ),

    "ExactCounterfactualLift":
        finite_number(
            cf[
                "exact_gt_cell"
            ][
                "composite_minus_clean_stats"
            ][
                "mean"
            ],
            "ExactCounterfactualLift",
        ),

    "ExactCompositeGtClean":
        finite_number(
            cf[
                "exact_gt_cell"
            ][
                "fraction_composite_gt_clean"
            ],
            "ExactCompositeGtClean",
        ),

    "LocalCounterfactualLift":
        finite_number(
            cf[
                "local_window_peak"
            ][
                "composite_minus_clean_stats"
            ][
                "mean"
            ],
            "LocalCounterfactualLift",
        ),

    "LocalCompositeGtClean":
        finite_number(
            cf[
                "local_window_peak"
            ][
                "fraction_composite_gt_clean"
            ],
            "LocalCompositeGtClean",
        ),

    "CleanMaxPeak":
        finite_number(
            cf[
                "scene_level"
            ][
                "clean_max_peak_stats"
            ][
                "mean"
            ],
            "CleanMaxPeak",
        ),

    "OutsideEffectMaxPeak":
        finite_number(
            cf[
                "scene_level"
            ][
                "composite_outside_effect_max_peak_stats"
            ][
                "mean"
            ],
            "OutsideEffectMaxPeak",
        ),

    "EffectRegionMaxPeak":
        finite_number(
            cf[
                "scene_level"
            ][
                "composite_effect_max_peak_stats"
            ][
                "mean"
            ],
            "EffectRegionMaxPeak",
        ),
}


# ============================================================
# SANITY FLAGS
#
# These are descriptive QC flags.
# They are NOT seed-selection rules.
# ============================================================

sanity = {
    "all_reported_values_finite":
        True,

    "counterfactual_exact_lift_positive":
        (
            results[
                "ExactCounterfactualLift"
            ]
            > 0.0
        ),

    "counterfactual_local_lift_positive":
        (
            results[
                "LocalCounterfactualLift"
            ]
            > 0.0
        ),

    "effect_response_above_outside_response":
        (
            results[
                "EffectRegionMaxPeak"
            ]
            >
            results[
                "OutsideEffectMaxPeak"
            ]
        ),

    "terminal_checkpoint_exactly_2750":
        True,

    "terminal_checkpoint_sha_verified":
        True,
}


qc_elapsed = (
    time.time()
    - qc_start
)


# ============================================================
# SAVE FORMAL QC SUMMARY
# ============================================================

qc_summary = {
    "qc_name":
        "DASO-Net-FinalSeed01-QC-val1000-v1",

    "seed_index":
        SEED_INDEX,

    "rng_seed":
        RNG_SEED,

    "experiment":
        EXPERIMENT,

    "role":
        "FINAL_SEED_QC_ONLY",

    "paper_test_result":
        False,

    "model_selection":
        False,

    "external_data_used":
        False,

    "validation_source":
        "FruitMP-Syn synthetic validation",

    "validation_scenes":
        VAL_SCENES,

    "validation_seed":
        VAL_SEED,

    "terminal_checkpoint": {
        "path":
            str(
                TERMINAL_CHECKPOINT
            ),

        "optimizer_step":
            TARGET_STEP,

        "sha256":
            terminal_sha,
    },

    "amp_provenance": {
        "original_skipped_optimizer_updates":
            int(
                completion_manifest[
                    "original_amp_skips"
                ]
            ),

        "completion_skipped_optimizer_updates":
            int(
                completion_manifest[
                    "additional_amp_skips_during_completion"
                ]
            ),

        "exact_final_successful_optimizer_updates":
            TARGET_STEP,
    },

    "metrics":
        results,

    "sanity_flags":
        sanity,

    "qc_runtime_minutes":
        qc_elapsed
        / 60.0,

    "interpretation_policy":
        (
            "QC is for integrity and gross failure detection only. "
            "The final seed is not selected or discarded according "
            "to relative validation performance."
        ),
}


QC_SUMMARY.write_text(
    json.dumps(
        qc_summary,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL SEED-1 QC SUMMARY"
)

print(
    "============================================================"
)

print(
    f"AP50:                 {results['AP50']:.4f}"
)

print(
    f"AP75:                 {results['AP75']:.4f}"
)

print(
    f"AP50:95:              {results['AP50_95']:.4f}"
)

print(
    f"BestF1:               {results['BestF1']:.4f}"
)

print(
    f"Best threshold:        {results['BestThreshold']:.2f}"
)

print(
    f"PeakRecall@8:          {results['PeakRecall8']:.4f}"
)

print(
    f"BoxIoU:                {results['BoxIoU']:.4f}"
)

print(
    f"GT-box mask IoU:       {results['GTBoxMaskIoU']:.4f}"
)

print(
    f"Empty FP/image @ .05:  {results['EmptyFP005']:.2f}"
)

print()

print(
    f"Exact CF lift:         "
    f"{results['ExactCounterfactualLift']:+.4f}"
)

print(
    f"Exact comp > clean:    "
    f"{results['ExactCompositeGtClean']:.4f}"
)

print(
    f"Local CF lift:         "
    f"{results['LocalCounterfactualLift']:+.4f}"
)

print(
    f"Local comp > clean:    "
    f"{results['LocalCompositeGtClean']:.4f}"
)

print(
    f"Clean max peak:        "
    f"{results['CleanMaxPeak']:.4f}"
)

print(
    f"Outside-effect max:    "
    f"{results['OutsideEffectMaxPeak']:.4f}"
)

print(
    f"Effect-region max:     "
    f"{results['EffectRegionMaxPeak']:.4f}"
)

print()

print(
    "Sanity flags:"
)

for key, value in sanity.items():

    print(
        f"  {key}: {value}"
    )

print()

print(
    "QC summary:"
)

print(
    QC_SUMMARY
)

print()

print(
    f"QC runtime: {qc_elapsed / 60.0:.2f} min"
)

print()

print(
    "========================================"
)

print(
    "DASO-NET FINAL SEED 1 QC: PASS"
)

print(
    "========================================"
)

print(
    "NO EXTERNAL DATA USED."
)

print(
    "NO MODEL SELECTION PERFORMED."
)

print(
    "Next: recovery + end-of-day handoff package."
)

DASO-Net — FINAL SEED 1 QC
PASS  Frozen project dataset
PASS  Evaluator
PASS  Oracle diagnostic
PASS  Counterfactual diagnostic
PASS  Final Seed-1 terminal checkpoint
PASS  Final Seed-1 training manifest
PASS  AMP-skip completion manifest
PASS  Frozen multi-seed plan
PASS  Frozen Seed-1 config

Terminal checkpoint SHA: PASS
Exact successful updates: 2750 — PASS
Original AMP skips documented: 6
Additional completion skips: 0
External data used: NONE

SEED-1 QC — 1000-SCENE METRIC EVALUATION
eval 0001/1000 scene=53fcddcd471d01372c54 gt=2 decoded=99 op_pred=15 elapsed=0.1m
eval 0011/1000 scene=1321d3c185e33b40ef7d gt=6 decoded=100 op_pred=45 elapsed=0.2m
eval 0021/1000 scene=4841ebcebecc85c1b2d5 gt=3 decoded=100 op_pred=36 elapsed=0.3m
eval 0031/1000 scene=a44182b4459025ec8df8 gt=1 decoded=100 op_pred=40 elapsed=0.4m
eval 0041/1000 scene=8308f9314aadda093efb gt=0 decoded=100 op_pred=14 elapsed=0.5m
eval 0051/1000 scene=3b14d3675c7b5a921bdc gt=4 decoded=99 op_pred=12 elapsed=0.6m
eval 0061

In [17]:
# ============================================================
# DASO-Net — AFTER FINAL SEED 1
# RECOVERY PACKAGE + END-OF-DAY HANDOFF
#
# Date: 2026-09-14
#
# Requires:
#   Final Seed 1 exact 2750 updates
#   Final Seed 1 QC complete
#
# Produces:
#
# 1. Compact Seed-1 recovery ZIP
# 2. Full tomorrow-restore EOD handoff ZIP
# 3. SHA256 sidecars
# 4. EOD state JSON
# 5. Restore script for next Kaggle notebook
#
# NO TRAINING.
# ============================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path


# ============================================================
# CONSTANTS
# ============================================================

DATE_TAG = "20260914"

SEED1 = 1782656963
SEED2 = 901395089
SEED3 = 252916394

EXPECTED_SEED1_TERMINAL_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe79"
    "6c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_RECIPE_ZIP_SHA = (
    "b102deb8ec9de4be3816e4450a4dc7b2"
    "f78f5439634cee02fe4656d1a268bf3a"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)


# ============================================================
# SOURCE PATHS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

MULTISEED_ROOT = (
    WORKING
    / "DASO-Net-FinalMultiSeed-v1"
)

MULTISEED_PLAN = (
    MULTISEED_ROOT
    / "FINAL_MULTISEED_PLAN_v1.json"
)

SEED1_CONFIG = (
    MULTISEED_ROOT
    / "configs"
    / "final_seed01_s1782656963.json"
)

SEED2_CONFIG = (
    MULTISEED_ROOT
    / "configs"
    / "final_seed02_s901395089.json"
)

SEED3_CONFIG = (
    MULTISEED_ROOT
    / "configs"
    / "final_seed03_s252916394.json"
)

RECIPE_FREEZE_DIR = (
    WORKING
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

RECIPE_FREEZE_ZIP = (
    WORKING
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1.zip"
)

SOURCE_CONFIG = (
    WORKING
    / "phase5c_baseline_curve1000updates_v1.json"
)

PREFLIGHT_JSON = (
    WORKING
    / "DASO-Net-FinalMultiSeed-Preflight-v1.json"
)

SEED1_RUN = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
)

SEED1_TERMINAL = (
    SEED1_RUN
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

SEED1_TRAINING_MANIFEST = (
    SEED1_RUN
    / "FINAL_SEED01_TRAINING_MANIFEST.json"
)

SEED1_COMPLETION_MANIFEST = (
    SEED1_RUN
    / "FINAL_SEED01_AMP_SKIP_COMPLETION.json"
)

SEED1_QC = (
    WORKING
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
)

SEED1_QC_SUMMARY = (
    SEED1_QC
    / "FINAL_SEED01_QC_SUMMARY.json"
)

DURATION_SELECTION_ROOT = (
    WORKING
    / "dasonet_phase5c"
    / "duration_selection_val1000_v1"
)

DURATION_SELECTION_SUMMARY = (
    DURATION_SELECTION_ROOT
    / "duration_selection_val1000_summary.json"
)

CURVE_SUMMARY = (
    WORKING
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
    / "baseline_learning_curve_3000_summary.json"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)


# ============================================================
# OUTPUT PATHS
# ============================================================

RECOVERY_NAME = (
    "DASO-Net-FinalSeed01-RECOVERY-20260914"
)

RECOVERY_STAGE = (
    WORKING
    / RECOVERY_NAME
)

RECOVERY_ZIP = (
    WORKING
    / f"{RECOVERY_NAME}.zip"
)

RECOVERY_SHA_FILE = (
    WORKING
    / f"{RECOVERY_NAME}.sha256.txt"
)


EOD_NAME = (
    "DASO-Net-EOD-20260914-AFTER-SEED1"
)

EOD_STAGE = (
    WORKING
    / EOD_NAME
)

EOD_ZIP = (
    WORKING
    / f"{EOD_NAME}.zip"
)

EOD_SHA_FILE = (
    WORKING
    / f"{EOD_NAME}.sha256.txt"
)

EOD_STATE_OUT = (
    WORKING
    / f"{EOD_NAME}-STATE.json"
)


# ============================================================
# HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def copy_file(
    src: Path,
    dst: Path,
) -> None:

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        src,
        dst,
    )


def copy_tree(
    src: Path,
    dst: Path,
) -> None:

    if dst.exists():

        shutil.rmtree(
            dst
        )

    shutil.copytree(
        src,
        dst,
        ignore=shutil.ignore_patterns(
            "__pycache__",
            "*.pyc",
            ".git",
            ".pytest_cache",
        ),
    )


def create_manifest(
    root: Path,
    manifest_name: str,
):

    entries = []

    manifest_path = (
        root
        / manifest_name
    )


    for path in sorted(
        p
        for p in root.rglob("*")
        if p.is_file()
        and p != manifest_path
    ):

        entries.append(
            {
                "path":
                    str(
                        path.relative_to(
                            root
                        )
                    ),

                "bytes":
                    int(
                        path.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


    manifest = {
        "root_name":
            root.name,

        "file_count":
            len(
                entries
            ),

        "total_bytes":
            sum(
                item[
                    "bytes"
                ]
                for item in entries
            ),

        "files":
            entries,
    }


    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2,
        ),
        encoding="utf-8",
    )


    return (
        manifest_path,
        manifest,
    )


def make_zip(
    staging_root: Path,
    zip_path: Path,
) -> None:

    if zip_path.exists():

        zip_path.unlink()


    with zipfile.ZipFile(
        zip_path,
        "w",
    ) as zf:

        for path in sorted(
            p
            for p in staging_root.rglob("*")
            if p.is_file()
        ):

            relative = (
                path.relative_to(
                    staging_root.parent
                )
            )

            if path.suffix.lower() in {
                ".pt",
                ".pth",
                ".zip",
            }:

                compression = (
                    zipfile.ZIP_STORED
                )

            else:

                compression = (
                    zipfile.ZIP_DEFLATED
                )


            zf.write(
                path,
                arcname=str(
                    relative
                ),
                compress_type=compression,
            )


    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as zf:

        bad = zf.testzip()

        if bad is not None:

            raise RuntimeError(
                f"ZIP CRC failure: {bad}"
            )


def verify_staged_files(
    root: Path,
    manifest: dict,
) -> None:

    for item in manifest[
        "files"
    ]:

        path = (
            root
            / item[
                "path"
            ]
        )

        require(
            path,
            item[
                "path"
            ],
        )


        actual = sha256_file(
            path
        )


        if (
            actual
            != item[
                "sha256"
            ]
        ):

            raise RuntimeError(
                "Staged manifest SHA mismatch:\n"
                f"{path}"
            )


# ============================================================
# REQUIRED INPUT AUDIT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — SEED-1 RECOVERY + EOD HANDOFF"
)

print(
    "============================================================"
)


required = [
    (
        CODE_ROOT,
        "Validated code tree",
    ),
    (
        MULTISEED_ROOT,
        "Frozen multi-seed plan directory",
    ),
    (
        MULTISEED_PLAN,
        "Multi-seed plan JSON",
    ),
    (
        SEED1_CONFIG,
        "Seed-1 config",
    ),
    (
        SEED2_CONFIG,
        "Seed-2 config",
    ),
    (
        SEED3_CONFIG,
        "Seed-3 config",
    ),
    (
        RECIPE_FREEZE_DIR,
        "Training-recipe freeze directory",
    ),
    (
        RECIPE_FREEZE_ZIP,
        "Training-recipe freeze ZIP",
    ),
    (
        SOURCE_CONFIG,
        "Source training config",
    ),
    (
        SEED1_RUN,
        "Final Seed-1 run",
    ),
    (
        SEED1_TERMINAL,
        "Seed-1 terminal model",
    ),
    (
        SEED1_TRAINING_MANIFEST,
        "Seed-1 training manifest",
    ),
    (
        SEED1_COMPLETION_MANIFEST,
        "Seed-1 AMP completion manifest",
    ),
    (
        SEED1_QC,
        "Seed-1 QC directory",
    ),
    (
        SEED1_QC_SUMMARY,
        "Seed-1 QC summary",
    ),
    (
        DURATION_SELECTION_SUMMARY,
        "Duration-selection summary",
    ),
    (
        CURVE_SUMMARY,
        "Learning-curve summary",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
]


for path, label in required:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# CRITICAL HASH AUDIT
# ============================================================

seed1_terminal_sha = sha256_file(
    SEED1_TERMINAL
)


if (
    seed1_terminal_sha
    != EXPECTED_SEED1_TERMINAL_SHA
):

    raise RuntimeError(
        "Seed-1 terminal checkpoint SHA failure."
    )


recipe_zip_sha = sha256_file(
    RECIPE_FREEZE_ZIP
)


if (
    recipe_zip_sha
    != EXPECTED_RECIPE_ZIP_SHA
):

    raise RuntimeError(
        "Training-recipe ZIP SHA failure."
    )


decoder_sha = sha256_file(
    DECODER
)


if (
    decoder_sha
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Decoder SHA failure."
    )


training_manifest = json.loads(
    SEED1_TRAINING_MANIFEST.read_text(
        encoding="utf-8"
    )
)

qc_summary = json.loads(
    SEED1_QC_SUMMARY.read_text(
        encoding="utf-8"
    )
)

seed2_cfg = json.loads(
    SEED2_CONFIG.read_text(
        encoding="utf-8"
    )
)


if int(
    training_manifest[
        "successful_optimizer_updates"
    ]
) != 2750:

    raise RuntimeError(
        "Seed-1 training manifest is not at 2750."
    )


if int(
    seed2_cfg[
        "seed"
    ]
) != SEED2:

    raise RuntimeError(
        "Seed-2 frozen config has wrong RNG seed."
    )


print()
print(
    "Seed-1 terminal checkpoint SHA: PASS"
)

print(
    "Training-recipe ZIP SHA: PASS"
)

print(
    "Validated decoder SHA: PASS"
)

print(
    "Seed-1 exact successful updates: 2750 — PASS"
)

print(
    "Seed-1 QC summary: PASS"
)

print(
    "Seed-2 frozen config ready: PASS"
)


# ============================================================
# CLEAN OLD STAGING
# ============================================================

for path in [
    RECOVERY_STAGE,
    EOD_STAGE,
]:

    if path.exists():

        shutil.rmtree(
            path
        )


# ============================================================
# BUILD COMPACT SEED-1 RECOVERY STAGING
# ============================================================

RECOVERY_STAGE.mkdir(
    parents=True,
    exist_ok=True,
)


copy_file(
    SEED1_TERMINAL,
    RECOVERY_STAGE
    / "model"
    / SEED1_TERMINAL.name,
)

copy_file(
    SEED1_TRAINING_MANIFEST,
    RECOVERY_STAGE
    / "manifests"
    / SEED1_TRAINING_MANIFEST.name,
)

copy_file(
    SEED1_COMPLETION_MANIFEST,
    RECOVERY_STAGE
    / "manifests"
    / SEED1_COMPLETION_MANIFEST.name,
)

for name in [
    "history.csv",
    "run_status.json",
    "resolved_config.json",
]:

    source = (
        SEED1_RUN
        / name
    )

    if source.exists():

        copy_file(
            source,
            RECOVERY_STAGE
            / "run_metadata"
            / name,
        )


copy_file(
    SEED1_CONFIG,
    RECOVERY_STAGE
    / "configs"
    / SEED1_CONFIG.name,
)

copy_tree(
    SEED1_QC,
    RECOVERY_STAGE
    / "qc",
)

copy_file(
    MULTISEED_PLAN,
    RECOVERY_STAGE
    / "plan"
    / MULTISEED_PLAN.name,
)

copy_file(
    SEED2_CONFIG,
    RECOVERY_STAGE
    / "plan"
    / "configs"
    / SEED2_CONFIG.name,
)

copy_file(
    SEED3_CONFIG,
    RECOVERY_STAGE
    / "plan"
    / "configs"
    / SEED3_CONFIG.name,
)

copy_file(
    RECIPE_FREEZE_DIR
    / "FINAL_TRAINING_RECIPE_FREEZE_v1.json",
    RECOVERY_STAGE
    / "freeze"
    / "FINAL_TRAINING_RECIPE_FREEZE_v1.json",
)

copy_file(
    RECIPE_FREEZE_DIR
    / "README_FREEZE.txt",
    RECOVERY_STAGE
    / "freeze"
    / "README_FREEZE.txt",
)


recovery_state = {
    "package":
        RECOVERY_NAME,

    "role":
        "FINAL_SEED1_RECOVERY",

    "seed_index":
        1,

    "rng_seed":
        SEED1,

    "successful_optimizer_updates":
        2750,

    "terminal_checkpoint":
        SEED1_TERMINAL.name,

    "terminal_sha256":
        seed1_terminal_sha,

    "original_amp_skips":
        training_manifest[
            "original_amp_skips"
        ],

    "additional_completion_amp_skips":
        training_manifest[
            "additional_completion_amp_skips"
        ],

    "qc_summary_sha256":
        sha256_file(
            SEED1_QC_SUMMARY
        ),

    "next_final_seed":
        {
            "seed_index":
                2,

            "rng_seed":
                SEED2,
        },
}


(
    RECOVERY_STAGE
    / "RECOVERY_STATE.json"
).write_text(
    json.dumps(
        recovery_state,
        indent=2,
    ),
    encoding="utf-8",
)


recovery_manifest_path, recovery_manifest = (
    create_manifest(
        RECOVERY_STAGE,
        "RECOVERY_MANIFEST.json",
    )
)


verify_staged_files(
    RECOVERY_STAGE,
    recovery_manifest,
)


make_zip(
    RECOVERY_STAGE,
    RECOVERY_ZIP,
)


recovery_zip_sha = sha256_file(
    RECOVERY_ZIP
)


RECOVERY_SHA_FILE.write_text(
    f"{recovery_zip_sha}  "
    f"{RECOVERY_ZIP.name}\n",
    encoding="utf-8",
)


# ============================================================
# BUILD FULL EOD HANDOFF STAGING
# ============================================================

EOD_STAGE.mkdir(
    parents=True,
    exist_ok=True,
)


workspace = (
    EOD_STAGE
    / "workspace"
)


# Core code
copy_tree(
    CODE_ROOT,
    workspace
    / "DASO-Net-Phase5B-KaggleTraining",
)


# Frozen final-seed plan + configs
copy_tree(
    MULTISEED_ROOT,
    workspace
    / "DASO-Net-FinalMultiSeed-v1",
)


# Frozen recipe directory
copy_tree(
    RECIPE_FREEZE_DIR,
    workspace
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1",
)


# Original recipe ZIP is required because its SHA is part
# of the frozen seed derivation / identity.
copy_file(
    RECIPE_FREEZE_ZIP,
    workspace
    / RECIPE_FREEZE_ZIP.name,
)


# Source development config
copy_file(
    SOURCE_CONFIG,
    workspace
    / SOURCE_CONFIG.name,
)


# Preflight report if present
if PREFLIGHT_JSON.exists():

    copy_file(
        PREFLIGHT_JSON,
        workspace
        / PREFLIGHT_JSON.name,
    )


# Full Seed-1 run with all chronological checkpoints
copy_tree(
    SEED1_RUN,
    workspace
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963",
)


# Seed-1 QC
copy_tree(
    SEED1_QC,
    workspace
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1",
)


# Preserve complete final duration-selection evidence.
copy_tree(
    DURATION_SELECTION_ROOT,
    workspace
    / "dasonet_phase5c"
    / "duration_selection_val1000_v1",
)


# Preserve final 250->3000 curve summary.
copy_file(
    CURVE_SUMMARY,
    workspace
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
    / "baseline_learning_curve_3000_summary.json",
)


# ============================================================
# EOD STATE
# ============================================================

eod_state = {
    "handoff":
        EOD_NAME,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "project_phase":
        "Final multi-seed training",

    "completed_final_seeds":
        [
            {
                "seed_index":
                    1,

                "rng_seed":
                    SEED1,

                "optimizer_updates":
                    2750,

                "terminal_checkpoint":
                    (
                        "/kaggle/working/"
                        "dasonet_phase5b/"
                        "dasonet_final_seed01_s1782656963/"
                        "snapshots/"
                        "epoch_011_opt_002750.pt"
                    ),

                "terminal_sha256":
                    seed1_terminal_sha,

                "qc_complete":
                    True,

                "original_amp_skips":
                    training_manifest[
                        "original_amp_skips"
                    ],

                "completion_amp_skips":
                    training_manifest[
                        "additional_completion_amp_skips"
                    ],
            }
        ],

    "next_action":
        "Start Final Seed 2 from fresh initialization.",

    "next_seed":
        {
            "seed_index":
                2,

            "rng_seed":
                SEED2,

            "experiment":
                (
                    "dasonet_final_seed02_"
                    "s901395089"
                ),

            "config":
                (
                    "/kaggle/working/"
                    "DASO-Net-FinalMultiSeed-v1/"
                    "configs/"
                    "final_seed02_s901395089.json"
                ),

            "resume":
                "none",

            "target_successful_optimizer_updates":
                2750,
        },

    "remaining_seed":
        {
            "seed_index":
                3,

            "rng_seed":
                SEED3,
        },

    "frozen_recipe":
        {
            "optimizer_updates":
                2750,

            "recipe_zip_sha256":
                recipe_zip_sha,

            "decoder_sha256":
                decoder_sha,

            "pcc":
                "original retained",

            "crpa":
                "retained",

            "pcc_center":
                "rejected_and_disabled",

            "external_tuning":
                False,
        },

    "critical_hashes":
        {
            "seed1_terminal":
                seed1_terminal_sha,

            "seed1_training_manifest":
                sha256_file(
                    SEED1_TRAINING_MANIFEST
                ),

            "seed1_completion_manifest":
                sha256_file(
                    SEED1_COMPLETION_MANIFEST
                ),

            "seed1_qc_summary":
                sha256_file(
                    SEED1_QC_SUMMARY
                ),

            "multiseed_plan":
                sha256_file(
                    MULTISEED_PLAN
                ),

            "seed2_config":
                sha256_file(
                    SEED2_CONFIG
                ),

            "seed3_config":
                sha256_file(
                    SEED3_CONFIG
                ),

            "recipe_freeze_zip":
                recipe_zip_sha,

            "decoder":
                decoder_sha,
        },
}


state_inside = (
    EOD_STAGE
    / "EOD_STATE.json"
)


state_inside.write_text(
    json.dumps(
        eod_state,
        indent=2,
    ),
    encoding="utf-8",
)


EOD_STATE_OUT.write_text(
    json.dumps(
        eod_state,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# RESTORE SCRIPT FOR TOMORROW
# ============================================================

restore_script = f'''\
from __future__ import annotations

import hashlib
import json
import shutil
from pathlib import Path


EXPECTED_SEED1_SHA = "{EXPECTED_SEED1_TERMINAL_SHA}"
EXPECTED_RECIPE_SHA = "{EXPECTED_RECIPE_ZIP_SHA}"
EXPECTED_DECODER_SHA = "{EXPECTED_DECODER_SHA}"


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def copy_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


def copy_tree(src: Path, dst: Path):
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)


PACKAGE_ROOT = Path(__file__).resolve().parent
SOURCE = PACKAGE_ROOT / "workspace"
DEST = Path("/kaggle/working")


print("=" * 60)
print("DASO-Net AFTER-SEED1 EOD RESTORE")
print("=" * 60)


tree_copies = [
    (
        SOURCE / "DASO-Net-Phase5B-KaggleTraining",
        DEST / "DASO-Net-Phase5B-KaggleTraining",
    ),
    (
        SOURCE / "DASO-Net-FinalMultiSeed-v1",
        DEST / "DASO-Net-FinalMultiSeed-v1",
    ),
    (
        SOURCE / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1",
        DEST / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1",
    ),
    (
        SOURCE / "dasonet_phase5b" / "dasonet_final_seed01_s1782656963",
        DEST / "dasonet_phase5b" / "dasonet_final_seed01_s1782656963",
    ),
    (
        SOURCE / "dasonet_final_qc" / "seed01_s1782656963_val1000_v1",
        DEST / "dasonet_final_qc" / "seed01_s1782656963_val1000_v1",
    ),
    (
        SOURCE / "dasonet_phase5c" / "duration_selection_val1000_v1",
        DEST / "dasonet_phase5c" / "duration_selection_val1000_v1",
    ),
]


for src, dst in tree_copies:
    if not src.exists():
        raise RuntimeError(f"Missing restore source: {{src}}")
    copy_tree(src, dst)
    print(f"PASS copied: {{dst}}")


file_copies = [
    (
        SOURCE / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1.zip",
        DEST / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1.zip",
    ),
    (
        SOURCE / "phase5c_baseline_curve1000updates_v1.json",
        DEST / "phase5c_baseline_curve1000updates_v1.json",
    ),
]


preflight = SOURCE / "DASO-Net-FinalMultiSeed-Preflight-v1.json"
if preflight.exists():
    file_copies.append(
        (
            preflight,
            DEST / "DASO-Net-FinalMultiSeed-Preflight-v1.json",
        )
    )


curve_summary = (
    SOURCE
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
    / "baseline_learning_curve_3000_summary.json"
)

file_copies.append(
    (
        curve_summary,
        DEST
        / "dasonet_phase5c"
        / "baseline_curve1000updates_v1"
        / "baseline_learning_curve_3000_summary.json",
    )
)


for src, dst in file_copies:
    if not src.exists():
        raise RuntimeError(f"Missing restore source: {{src}}")
    copy_file(src, dst)
    print(f"PASS copied: {{dst}}")


seed1_terminal = (
    DEST
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

recipe_zip = (
    DEST
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1.zip"
)

decoder = (
    DEST
    / "DASO-Net-Phase5B-KaggleTraining"
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

seed2_config = (
    DEST
    / "DASO-Net-FinalMultiSeed-v1"
    / "configs"
    / "final_seed02_s901395089.json"
)


if sha256_file(seed1_terminal) != EXPECTED_SEED1_SHA:
    raise RuntimeError("Seed-1 terminal SHA restore failure.")

if sha256_file(recipe_zip) != EXPECTED_RECIPE_SHA:
    raise RuntimeError("Recipe ZIP SHA restore failure.")

if sha256_file(decoder) != EXPECTED_DECODER_SHA:
    raise RuntimeError("Decoder SHA restore failure.")

if not seed2_config.exists():
    raise RuntimeError("Seed-2 config missing after restore.")

seed2 = json.loads(seed2_config.read_text(encoding="utf-8"))

if int(seed2["seed"]) != 901395089:
    raise RuntimeError("Seed-2 RNG mismatch after restore.")

seed2_output = (
    DEST
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
)

if seed2_output.exists():
    raise RuntimeError(
        "Seed-2 output already exists. "
        "Fresh initialization condition violated."
    )


print()
print("Seed-1 terminal SHA: PASS")
print("Recipe freeze SHA:  PASS")
print("Decoder SHA:        PASS")
print("Seed-2 config:       PASS")
print("Seed-2 output absent: PASS")
print()
print("=" * 60)
print("DASO-NET AFTER-SEED1 EOD RESTORE: PASS")
print("=" * 60)
print("Next action: Final Seed 2, fresh initialization.")
'''


restore_path = (
    EOD_STAGE
    / "RESTORE_AFTER_SEED1.py"
)


restore_path.write_text(
    restore_script,
    encoding="utf-8",
)


# ============================================================
# EOD MANIFEST
# ============================================================

eod_manifest_path, eod_manifest = (
    create_manifest(
        EOD_STAGE,
        "EOD_MANIFEST.json",
    )
)


verify_staged_files(
    EOD_STAGE,
    eod_manifest,
)


# ============================================================
# BUILD + VERIFY EOD ZIP
# ============================================================

make_zip(
    EOD_STAGE,
    EOD_ZIP,
)


eod_zip_sha = sha256_file(
    EOD_ZIP
)


EOD_SHA_FILE.write_text(
    f"{eod_zip_sha}  "
    f"{EOD_ZIP.name}\n",
    encoding="utf-8",
)


# ============================================================
# FINAL PACKAGE AUDIT
# ============================================================

with zipfile.ZipFile(
    RECOVERY_ZIP,
    "r",
) as zf:

    if zf.testzip() is not None:

        raise RuntimeError(
            "Recovery ZIP CRC verification failed."
        )


with zipfile.ZipFile(
    EOD_ZIP,
    "r",
) as zf:

    if zf.testzip() is not None:

        raise RuntimeError(
            "EOD ZIP CRC verification failed."
        )


# ============================================================
# FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "SEED-1 RECOVERY PACKAGE"
)

print(
    "============================================================"
)

print(
    RECOVERY_ZIP
)

print(
    f"Size: "
    f"{RECOVERY_ZIP.stat().st_size / (1024**2):.2f} MiB"
)

print(
    "SHA256:"
)

print(
    recovery_zip_sha
)

print(
    "CRC verification: PASS"
)

print(
    "Staged SHA verification: PASS"
)


print()
print(
    "============================================================"
)

print(
    "AFTER-SEED1 END-OF-DAY HANDOFF"
)

print(
    "============================================================"
)

print(
    EOD_ZIP
)

print(
    f"Size: "
    f"{EOD_ZIP.stat().st_size / (1024**2):.2f} MiB"
)

print(
    "SHA256:"
)

print(
    eod_zip_sha
)

print(
    "CRC verification: PASS"
)

print(
    "Staged SHA verification: PASS"
)

print()

print(
    "EOD state:"
)

print(
    EOD_STATE_OUT
)

print()

print(
    "Tomorrow restore script inside package:"
)

print(
    restore_path
)

print()

print(
    "Seed-1 terminal:"
)

print(
    SEED1_TERMINAL
)

print(
    "Seed-1 terminal SHA:"
)

print(
    seed1_terminal_sha
)

print()

print(
    "Tomorrow next seed:"
)

print(
    "  Seed index: 2"
)

print(
    "  RNG seed:   901395089"
)

print(
    "  Resume:     NONE"
)

print(
    "  Target:     2750 successful optimizer updates"
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET AFTER-SEED1 EOD HANDOFF: PASS"
)

print(
    "============================================================"
)

print(
    "DOWNLOAD BOTH ZIP FILES + BOTH SHA256 FILES."
)

print(
    "Then it is safe to stop Kaggle for today."
)

DASO-Net — SEED-1 RECOVERY + EOD HANDOFF
PASS  Validated code tree
PASS  Frozen multi-seed plan directory
PASS  Multi-seed plan JSON
PASS  Seed-1 config
PASS  Seed-2 config
PASS  Seed-3 config
PASS  Training-recipe freeze directory
PASS  Training-recipe freeze ZIP
PASS  Source training config
PASS  Final Seed-1 run
PASS  Seed-1 terminal model
PASS  Seed-1 training manifest
PASS  Seed-1 AMP completion manifest
PASS  Seed-1 QC directory
PASS  Seed-1 QC summary
PASS  Duration-selection summary
PASS  Learning-curve summary
PASS  Validated decoder

Seed-1 terminal checkpoint SHA: PASS
Training-recipe ZIP SHA: PASS
Validated decoder SHA: PASS
Seed-1 exact successful updates: 2750 — PASS
Seed-1 QC summary: PASS
Seed-2 frozen config ready: PASS

SEED-1 RECOVERY PACKAGE
/kaggle/working/DASO-Net-FinalSeed01-RECOVERY-20260914.zip
Size: 13.85 MiB
SHA256:
636463137cc967baca7c47731046e6241b601583ad11e7787d250bea043f8bc0
CRC verification: PASS
Staged SHA verification: PASS

AFTER-SEED1 END-OF-DAY HAN